# Autonomous and Verifiable Data Mining Using Agentic AI: study pipeline

This notebook implements the protocol in `main-plan.md` end to end: benchmark construction (D1-D6, T1-T6, C0-C6, N1/N2),
the seven systems (F, S0, M0, MQ, MP, SQP, MQP), the independent evaluator, the pilot (E0), the protocol freeze, the formal
experiments (E1, E4, E5, optional E7), replay checks (E6) and the prespecified analysis.

The platform is detected automatically (`PLATFORM = "auto"`): the UA Little Rock CRC JupyterHub server, Kaggle, or local mock mode.

## How to run it on the CRC server (primary)
Hardware: 1 x NVIDIA RTX PRO 6000 Blackwell (96 GB), 64 CPU cores. Follows `main-plan-v2.md`: backbone A
`Qwen/Qwen3.8-27B`, backbone B `google/gemma-4-31B-it`, served one at a time in **BF16** with reasoning mode off. Use the **Python 3 (ipykernel)** kernel; missing packages are installed with `pip install --user`.
1. **Session 1** runs the tests, evaluator fixtures, model preflight and the pilot (E0), then writes
   `adm_crc_run/adm_state/reports/pilot.md` and a freeze proposal. Read the report.
2. **Session 2**: set `FREEZE_CONFIRMED = True` (optionally `FREEZE_REPETITIONS`) and run again. The protocol is frozen and
   the formal experiments run to completion; there is no session time limit.
3. If a session is interrupted (server restart, idle culling), run again: completed runs are kept and skipped.

Everything the notebook creates lives in `adm_crc_run/` next to it and persists: state in `adm_state`, model weights in
`adm_models` and the vLLM environment in `vllm_env` (both reused across sessions).

## How to run it on Kaggle
1. Create a notebook, upload this file, then in **Settings** choose **Accelerator: GPU T4 x2** and **Internet: On**.
2. Click **Save Version -> Save & Run All (Commit)**. Every session runs all cells; the `auto` stage picks the next step.
3. **Session 1** runs the tests, evaluator fixtures, model preflight and the pilot (E0), then writes `adm_state/reports/pilot.md`
   and a freeze proposal. Read the report.
4. **Session 2**: open the notebook, choose **Add Input -> Your Work -> (this notebook's output from the previous version)**,
   set `FREEZE_CONFIRMED = True` in the configuration cell (optionally `FREEZE_REPETITIONS`), and Save & Run All again.
   The protocol is frozen and E1 starts.
5. **Later sessions**: each time, attach the newest output version as input and Save & Run All. Work resumes from the saved
   state; each session stops dispatching new runs at about 10.5 hours and saves everything. The session summary at the end
   says what happens next. The study is complete when `adm_state/reports/final_report.md` exists.

Everything that must persist lives in `/kaggle/working/adm_state` (records, traces, submissions, manifests, reports).
Generated task data and model weights are rebuilt each session under `/tmp` and verified against recorded content digests.

## Running locally
Without Kaggle or the CRC server the notebook runs in **local mock mode**: no GPU, a scripted MockLLM replaces the language model, and a reduced
rehearsal design replaces E1/E4/E5. This tests every component; its numbers are not research results.


## 1. Configuration

In [ ]:
# ===================== CONFIGURATION (edit here) =====================
import time as _time
NOTEBOOK_START = _time.time()

STAGE = "auto"                 # "auto" | "pilot" | "supplement" | "calibrate" | "calibrate2" | "freeze" | "formal" | "report"
FREEZE_CONFIRMED = True        # set True after reviewing reports/pilot.md
FREEZE_REPETITIONS = [0, 1, 2]      # e.g. [0, 1, 2, 3]; None = proposal (power-based, at least 3)
FREEZE_LIMIT_OVERRIDES = {}    # e.g. {"aggregate_tokens": 120000}; overrides are recorded in the freeze manifest
ENABLE_E7 = False              # optional budget-sensitivity experiment; must be chosen before E1 starts
CONCURRENCY = None             # concurrent agent runs (None: 8 on CRC/Kaggle, 6 locally); locked at freeze
RUN_TESTS = True               # run the unit/isolation test suite each session
PLATFORM = "auto"              # "auto" | "crc" | "kaggle" | "local"
SESSION_STOP_NEW_HOURS = None  # stop dispatching new runs after this many hours (None: 10.5 on Kaggle/local, no limit on CRC)
SESSION_HARD_STOP_HOURS = None # kill in-flight runs after this many hours (None: 11.25 on Kaggle/local, no limit on CRC)
SESSION_DEADLINE = "2026-11-12 08:00"      # e.g. "2026-11-11 08:50" (server local time): overrides the two limits above; no new runs
                               # start in the last 35 minutes and the session stops at the deadline, even after a restart
FORCE_MOCK = False             # CRC/Kaggle: use the MockLLM instead of vLLM (plumbing test)
CRC_ROOT = "adm_crc_run"       # CRC only: run root (state + cache), relative to the working directory
LOCAL_ROOT = "adm_local_run"   # local mode only
LOCAL_REHEARSAL = {"families": ["D5", "D6"], "templates": ["T1", "T3"], "conditions": ["C0", "C3", "C6"]}
LOCAL_REPETITIONS = [0, 1]


## 2. Environment setup

In [ ]:
import os, sys, json, shutil
from pathlib import Path
if PLATFORM == "auto":
    if os.environ.get("ADM_PLATFORM"):
        PLATFORM = os.environ["ADM_PLATFORM"]
    elif Path("/kaggle/working").exists() and os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        PLATFORM = "kaggle"
    elif Path("/srv/exaiops-shared").exists() and "/exaiops/" in os.environ.get("JUPYTERHUB_API_URL", ""):
        PLATFORM = "crc"
    else:
        PLATFORM = "local"
assert PLATFORM in ("crc", "kaggle", "local"), PLATFORM
ON_KAGGLE = PLATFORM == "kaggle"
_unlimited = PLATFORM == "crc"
if SESSION_STOP_NEW_HOURS is None:
    SESSION_STOP_NEW_HOURS = 10_000.0 if _unlimited else 10.5
if SESSION_HARD_STOP_HOURS is None:
    SESSION_HARD_STOP_HOURS = 10_000.0 if _unlimited else 11.25
if SESSION_DEADLINE:
    import datetime as _dt
    _left = (_dt.datetime.fromisoformat(SESSION_DEADLINE).timestamp() - NOTEBOOK_START) / 3600
    if _left <= 0.6:
        raise RuntimeError(f"SESSION_DEADLINE {SESSION_DEADLINE} is less than 36 minutes away; nothing would run")
    SESSION_HARD_STOP_HOURS = _left
    SESSION_STOP_NEW_HOURS = _left - 35 / 60
    print(f"deadline {SESSION_DEADLINE}: new runs stop after {SESSION_STOP_NEW_HOURS:.2f} h, hard stop after {_left:.2f} h")
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
os.chdir(WORK)
CODE = Path("adm_code")
if CODE.exists():
    shutil.rmtree(CODE)            # the package is rewritten from this notebook every session
for d in ["adm/prompts", "adm/schemas", "tests"]:
    (CODE / d).mkdir(parents=True, exist_ok=True)
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"
import importlib, importlib.metadata as _md, site, subprocess, sysconfig
PACKAGES = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "duckdb": "duckdb",
            "pyarrow": "pyarrow", "matplotlib": "matplotlib", "xlrd": "xlrd", "openpyxl": "openpyxl",
            "jsonschema": "jsonschema", "pytest": "pytest"}
_user = not os.access(sysconfig.get_paths()["purelib"], os.W_OK)  # read-only kernel env (CRC): install with --user
if _user:
    _bin = os.path.join(site.getuserbase(), "bin")                  # ~/.local/bin: pip-installed scripts (pytest, uv)
    if _bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = _bin + os.pathsep + os.environ.get("PATH", "")
missing = []
for mod, pkg in PACKAGES.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
# CRC: bring every package to the latest release for this kernel's Python until the protocol is frozen; after the
# freeze the installed versions are kept so the formal runs use the same stack as the pilot. Elsewhere install only
# what is missing (Kaggle's preinstalled stack is left alone).
_frozen = (WORK / CRC_ROOT / "adm_state" / "manifests" / "freeze.json").exists()
_upgrade = PLATFORM == "crc" and not _frozen
install = list(PACKAGES.values()) if _upgrade else missing
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", "--disable-pip-version-check",
                    *(["--upgrade"] if _upgrade else []), *(["--user"] if _user else []), *install], check=True)
    if _user:
        site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()
    for mod in PACKAGES:
        importlib.import_module(mod)                                 # fail here, not later, if anything is still missing
PACKAGE_VERSIONS = {pkg: _md.version(pkg) for pkg in PACKAGES.values()}
print("packages:", PACKAGE_VERSIONS)
print("platform:", PLATFORM, "| python:", sys.version.split()[0], "| working directory:", WORK, "| installed:", missing or "nothing")


## 3. Study package
Each cell writes one module of the `adm` package. Do not edit between frozen sessions: the code hash is part of the protocol freeze.

### Core utilities: seeds, hashing, paths, session clock

In [ ]:
%%writefile adm_code/adm/__init__.py
"""Autonomous and Verifiable Data Mining (ADM) study package.

Generated by the Kaggle notebook ``adm_kaggle_pipeline.ipynb``. Do not edit the
written copy; edit the notebook cells instead (the code hash is part of the
protocol freeze).
"""

PACKAGE_VERSION = "1.0.0"


In [ ]:
%%writefile adm_code/adm/core.py
"""Core utilities: seeds, hashing, paths, settings, JSON I/O and the session clock."""
from __future__ import annotations

import decimal
import gzip
import hashlib
import io
import json
import math
import os
import shutil
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

PROTOCOL_VERSION = "adm-kaggle-1.0"
MASTER_SEED = 20261030
NAMESPACES = ("development", "evaluation", "corruption", "split", "estimator", "agent")

FAMILIES = ["D1", "D2", "D3", "D4", "D5", "D6"]
DEV_FAMILIES = ["D5", "D6"]
TEMPLATES = ["T1", "T2", "T3", "T4", "T5", "T6"]
CONDITIONS = ["C0", "C1", "C2", "C3", "C4", "C5", "C6"]
NEGATIVE_CONDITIONS = ["N1", "N2"]
TIER2_CONDITIONS = ["X1a", "X1b", "X1c", "X2a", "X2b", "X2c", "X3a", "X3b", "X3c"]  # first calibration (E0X)
# main-plan-v2.1 Section 7.6 (amended): X1 is dropped from the design and replaced by X4, a provenance-type defect
TIER2_DESIGN = ["X2a", "X2b", "X2c", "X3a", "X3b", "X3c", "X4a", "X4b", "X4c"]  # second calibration (E0X2)
SYSTEMS = ["F", "S0", "M0", "MQ", "MP", "SQP", "MQP"]
AGENTIC_SYSTEMS = ["S0", "M0", "MQ", "MP", "SQP", "MQP"]
REPLICATION_TASKS = ["D1-T5", "D2-T2", "D3-T4", "D4-T6", "D5-T3", "D6-T1"]
REPLICATION_SYSTEMS = ["M0", "SQP", "MQP"]

# Architecture table (Section 11): code -> (planning, Q gates, P gates)
SYSTEM_SPEC = {
    "F": {"planner": "fixed", "q": True, "p": True},
    "S0": {"planner": "single", "q": False, "p": False},
    "M0": {"planner": "multi", "q": False, "p": False},
    "MQ": {"planner": "multi", "q": True, "p": False},
    "MP": {"planner": "multi", "q": False, "p": True},
    "SQP": {"planner": "single", "q": True, "p": True},
    "MQP": {"planner": "multi", "q": True, "p": True},
}

ABSTENTION_REASONS = [
    "REQUIRED_SOURCE_MISSING",
    "AMBIGUOUS_AUTHORITY",
    "CONTRACT_UNSATISFIABLE",
    "BUDGET_INSUFFICIENT",
]

TERMINAL_STATES = ["SUBMITTED", "ABSTAINED", "BUDGET_EXCEEDED", "EXECUTION_ERROR", "INFRASTRUCTURE_ERROR"]

PILOT_LIMITS = {
    "aggregate_tokens": 300_000,
    "model_calls": 48,
    "tool_calls": 40,
    "repair_cycles": 2,
    "pipeline_evaluations": 6,
    "wall_seconds": 1200,
}
PROTOCOL_LIMITS = {
    "aggregate_tokens": 30_000,
    "model_calls": 48,
    "tool_calls": 40,
    "repair_cycles": 2,
    "pipeline_evaluations": 6,
    "wall_seconds": 600,
}
GENERATION = {
    "temperature": 0.2,
    "top_p": 0.9,
    "top_k": 40,
    "max_context_tokens": 16384,
    "max_output_tokens_per_call": 1024,
    # main-plan-v2.md Section 9.2: reasoning ("thinking") mode off for every backbone; actions are schema-constrained
    # JSON within a 1,024-token output limit. Sent as chat_template_kwargs; templates without the switch ignore it.
    "enable_thinking": False,
}


# --------------------------------------------------------------------------- seeds
def seed_string(namespace: str, family: str = "", task: str = "", repetition: Any = "",
                component: str = "", master_seed: int = MASTER_SEED,
                protocol_version: str = PROTOCOL_VERSION) -> str:
    """The exact string hashed for seed derivation (Section 5.1)."""
    if namespace not in NAMESPACES:
        raise ValueError(f"unknown seed namespace {namespace!r}")
    parts = [protocol_version, str(master_seed), namespace, str(family), str(task), str(repetition), str(component)]
    return "|".join(parts)


def derive_seed(namespace: str, family: str = "", task: str = "", repetition: Any = "",
                component: str = "", master_seed: int = MASTER_SEED,
                protocol_version: str = PROTOCOL_VERSION) -> int:
    """First eight bytes of SHA-256 over the UTF-8 seed string, read as unsigned big-endian."""
    s = seed_string(namespace, family, task, repetition, component, master_seed, protocol_version)
    return int.from_bytes(hashlib.sha256(s.encode("utf-8")).digest()[:8], "big", signed=False)


def make_rng(namespace: str, family: str = "", task: str = "", repetition: Any = "",
             component: str = "") -> np.random.Generator:
    return np.random.Generator(np.random.PCG64(derive_seed(namespace, family, task, repetition, component)))


def small_seed(seed: int) -> int:
    """Fold a 64-bit seed into the 31-bit range accepted by scikit-learn and vLLM."""
    return int(seed % (2**31 - 1))


# --------------------------------------------------------------------------- hashing
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def sha256_text(s: str) -> str:
    return sha256_bytes(s.encode("utf-8"))


def sha256_file(path: str | Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def sha256_tree(root: str | Path, patterns: Iterable[str] = ("**/*",), exclude_dirs: Iterable[str] = ("__pycache__",)) -> str:
    root = Path(root)
    entries = []
    ex = set(exclude_dirs)
    for pat in patterns:
        for p in sorted(root.glob(pat)):
            if p.is_file() and not (set(p.relative_to(root).parts) & ex):
                entries.append((p.relative_to(root).as_posix(), sha256_file(p)))
    entries = sorted(set(entries))
    return sha256_text(json.dumps(entries))


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=_json_default)


def json_hash(obj: Any) -> str:
    return sha256_text(canonical_json(obj))


def df_hash(df: pd.DataFrame) -> str:
    """Content hash of a table: independent of file encoding, row order and column order, but
    sensitive to every value and to row multiplicity."""
    d = df[sorted(df.columns)]
    buf = io.StringIO()
    d.to_csv(buf, index=False, float_format="%.17g", na_rep="<NA>", lineterminator="\n")
    lines = buf.getvalue().split("\n")
    header, rows = lines[0], sorted(l for l in lines[1:] if l != "")
    return sha256_text(header + "\n" + "\n".join(rows))


# --------------------------------------------------------------------------- JSON I/O
def _json_default(o: Any):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        v = float(o)
        return None if not math.isfinite(v) else v
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, decimal.Decimal):  # DuckDB returns DECIMAL columns as decimal.Decimal
        v = float(o)
        return None if not math.isfinite(v) else v
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, set):
        return sorted(o)
    if isinstance(o, (pd.Timestamp,)):
        return o.isoformat()
    raise TypeError(f"not JSON serializable: {type(o)}")


def clean_json(obj: Any) -> Any:
    """Replace non-finite floats with None recursively (JSON has no NaN)."""
    if isinstance(obj, dict):
        return {str(k): clean_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean_json(v) for v in obj]
    if isinstance(obj, (float, np.floating)):
        v = float(obj)
        return v if math.isfinite(v) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def dumps(obj: Any, indent: int | None = None) -> str:
    return json.dumps(clean_json(obj), indent=indent, default=_json_default, sort_keys=indent is not None)


def write_json(path: str | Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(dumps(obj, indent=indent))
    os.replace(tmp, path)


def write_json_once(path: str | Path, obj: Any) -> bool:
    """Write an immutable record. Returns False (and does not overwrite) if it exists."""
    path = Path(path)
    if path.exists():
        return False
    write_json(path, obj)
    return True


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text())


def append_jsonl_gz(path: str | Path, records: Iterable[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "at", encoding="utf-8") as f:
        for r in records:
            f.write(dumps(r) + "\n")


def read_jsonl_gz(path: str | Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def append_jsonl(path: str | Path, record: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(dumps(record) + "\n")


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


# --------------------------------------------------------------------------- paths
@dataclass
class Paths:
    """All directories used by the study. ``state`` is persisted between Kaggle sessions;
    ``cache`` is session-local (rebuilt deterministically and hash-verified)."""

    state: Path
    cache: Path
    models: Path

    def __post_init__(self):
        self.state = Path(self.state)
        self.cache = Path(self.cache)
        self.models = Path(self.models)

    # persisted
    @property
    def configs(self) -> Path: return self.state / "configs"
    @property
    def manifests(self) -> Path: return self.state / "manifests"
    @property
    def raw(self) -> Path: return self.state / "data" / "raw"
    @property
    def canonical(self) -> Path: return self.state / "data" / "canonical"
    @property
    def outputs(self) -> Path: return self.state / "outputs"
    @property
    def runs(self) -> Path: return self.outputs / "runs"
    @property
    def attempts(self) -> Path: return self.outputs / "attempts"
    @property
    def traces(self) -> Path: return self.outputs / "traces"
    @property
    def submissions(self) -> Path: return self.outputs / "submissions"
    @property
    def tables(self) -> Path: return self.outputs / "tables"
    @property
    def figures(self) -> Path: return self.outputs / "figures"
    @property
    def reports(self) -> Path: return self.state / "reports"
    # session-local
    @property
    def generated(self) -> Path: return self.cache / "data" / "generated"
    @property
    def trusted(self) -> Path: return self.cache / "data" / "trusted"
    @property
    def private(self) -> Path: return self.cache / "private_evaluation"
    @property
    def work(self) -> Path: return self.cache / "work"

    def ensure(self) -> "Paths":
        for p in [self.configs, self.manifests, self.raw, self.canonical, self.runs, self.attempts,
                  self.traces, self.submissions, self.tables, self.figures, self.reports,
                  self.generated, self.trusted, self.private, self.work, self.models]:
            p.mkdir(parents=True, exist_ok=True)
        return self

    def to_dict(self) -> dict:
        return {"state": str(self.state), "cache": str(self.cache), "models": str(self.models)}

    @classmethod
    def from_dict(cls, d: dict) -> "Paths":
        return cls(Path(d["state"]), Path(d["cache"]), Path(d["models"]))


# --------------------------------------------------------------------------- session clock
@dataclass
class SessionClock:
    """Guards the 12-hour Kaggle session. New runs stop being dispatched at ``stop_new_h``;
    in-flight runs are killed at ``hard_stop_h`` and marked as infrastructure failures."""

    start: float = field(default_factory=time.time)
    stop_new_h: float = 10.5
    hard_stop_h: float = 11.25
    already_elapsed_s: float = 0.0
    now_fn: Any = time.time

    def elapsed(self) -> float:
        return self.now_fn() - self.start + self.already_elapsed_s

    def may_start_new(self, expected_duration_s: float = 0.0) -> bool:
        return self.elapsed() + expected_duration_s < self.stop_new_h * 3600

    def must_stop(self) -> bool:
        return self.elapsed() >= self.hard_stop_h * 3600

    def remaining_new_s(self) -> float:
        return self.stop_new_h * 3600 - self.elapsed()

    def to_dict(self) -> dict:
        return {"start": self.start, "stop_new_h": self.stop_new_h, "hard_stop_h": self.hard_stop_h,
                "already_elapsed_s": self.already_elapsed_s}


# --------------------------------------------------------------------------- misc
def now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def dir_size_bytes(path: str | Path) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total


def rmtree(path: str | Path) -> None:
    shutil.rmtree(path, ignore_errors=True)


def task_id(family: str, template: str) -> str:
    return f"{family}-{template}"


def base_task_index(family: str, template: str) -> int:
    return FAMILIES.index(family) * len(TEMPLATES) + TEMPLATES.index(template)


def instance_key(suite: str, family: str, template: str, condition: str, rep: int) -> str:
    return f"{suite}/{family}-{template}/{condition}/r{rep}"


def instance_dirs(paths: Paths, suite: str, family: str, template: str, condition: str, rep: int) -> dict:
    rel = Path(suite) / f"{family}-{template}" / condition / f"r{rep}"
    return {
        "visible": paths.generated / rel,
        "trusted": paths.trusted / rel,
        "private": paths.private / rel,
    }


def safe_name(s: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in s)


### Datasets: acquisition, canonical import (D1-D4), synthetic generators (D5-D6), splits

In [ ]:
%%writefile adm_code/adm/datasets.py
"""Dataset acquisition, canonical import (D1-D4), synthetic generators (D5-D6), and splits."""
from __future__ import annotations

import io
import time
import urllib.request
import zipfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd

from .core import (Paths, make_rng, now_iso, read_json, sha256_file, write_json, df_hash, small_seed,
                   derive_seed)

# --------------------------------------------------------------------------- sources
UCI_SOURCES = {
    "D1": {"uci_id": 601, "name": "AI4I 2020 Predictive Maintenance Dataset",
           "url": "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip",
           "page": "https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset",
           "doi": "10.24432/C5HS5C", "citation": "Matzka, S. (2020). AI4I 2020 Predictive Maintenance Dataset. UCI Machine Learning Repository."},
    "D2": {"uci_id": 198, "name": "Steel Plates Faults",
           "url": "https://archive.ics.uci.edu/static/public/198/steel+plates+faults.zip",
           "page": "https://archive.ics.uci.edu/dataset/198/steel+plates+faults",
           "doi": "10.24432/C5J88N", "citation": "Buscema, M., Terzi, S., & Tastle, W. (2010). Steel Plates Faults. UCI Machine Learning Repository."},
    "D3": {"uci_id": 165, "name": "Concrete Compressive Strength",
           "url": "https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip",
           "page": "https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength",
           "doi": "10.24432/C5PK67", "citation": "Yeh, I.-C. (1998). Concrete Compressive Strength. UCI Machine Learning Repository."},
    "D4": {"uci_id": 242, "name": "Energy Efficiency",
           "url": "https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip",
           "page": "https://archive.ics.uci.edu/dataset/242/energy+efficiency",
           "doi": "10.24432/C51307", "citation": "Tsanas, A. & Xifara, A. (2012). Energy Efficiency. UCI Machine Learning Repository."},
}
LICENSE = "CC BY 4.0"

STEEL_LABELS = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]


@dataclass
class FamilySpec:
    family: str
    name: str
    task_type: str                 # binary | multiclass | regression
    target: str
    features: list                 # ordered permitted predictors
    categorical: list
    excluded: list                 # columns present in source records but not permitted as predictors
    group_col: str | None          # internal grouping column (never shown as such)
    split_rule: str
    selection_metric: str
    classes: list = field(default_factory=list)
    synthetic: bool = False
    notes: str = ""

    @property
    def numeric(self) -> list:
        return [f for f in self.features if f not in self.categorical]

    def to_dict(self) -> dict:
        return asdict(self)


def _snake(s: str) -> str:
    out = []
    for ch in s.strip():
        out.append(ch.lower() if ch.isalnum() else "_")
    t = "".join(out)
    while "__" in t:
        t = t.replace("__", "_")
    return t.strip("_")


# --------------------------------------------------------------------------- acquisition
def _download(url: str, dest: Path, retries: int = 4) -> None:
    last = None
    for k in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "adm-research/1.0"})
            with urllib.request.urlopen(req, timeout=120) as r:
                data = r.read()
            dest.write_bytes(data)
            return
        except Exception as e:  # pragma: no cover - network
            last = e
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"download failed for {url}: {last}")


def acquire(paths: Paths, local_source_dir: str | Path | None = None) -> dict:
    """Download (or reuse) original archives, never modifying them; record checksums."""
    paths.raw.mkdir(parents=True, exist_ok=True)
    manifest_path = paths.manifests / "datasets.json"
    previous = read_json(manifest_path) if manifest_path.exists() else {}
    manifest = {}
    for fam, src in UCI_SOURCES.items():
        dest = paths.raw / f"uci_{src['uci_id']}.zip"
        if not dest.exists():
            if local_source_dir and (Path(local_source_dir) / f"{src['uci_id']}.zip").exists():
                dest.write_bytes((Path(local_source_dir) / f"{src['uci_id']}.zip").read_bytes())
            else:
                _download(src["url"], dest)
        digest = sha256_file(dest)
        prev = previous.get(fam, {})
        if prev.get("sha256") and prev["sha256"] != digest:
            raise RuntimeError(f"{fam}: raw archive checksum changed ({prev['sha256']} -> {digest}). "
                               "The frozen protocol requires the original archive.")
        manifest[fam] = {
            **src, "license": LICENSE, "archive": dest.name, "sha256": digest,
            "retrieved": prev.get("retrieved", now_iso()), "bytes": dest.stat().st_size,
        }
    write_json(manifest_path, manifest)
    return manifest


def _zip_member(path: Path, name: str) -> bytes:
    with zipfile.ZipFile(path) as z:
        return z.read(name)


# --------------------------------------------------------------------------- canonical import
def import_d1(raw: Path) -> tuple[pd.DataFrame, FamilySpec, dict]:
    df = pd.read_csv(io.BytesIO(_zip_member(raw / "uci_601.zip", "ai4i2020.csv")))
    mapping = {"UDI": "uid", "Product ID": "product_id", "Type": "type",
               "Air temperature [K]": "air_temperature_k", "Process temperature [K]": "process_temperature_k",
               "Rotational speed [rpm]": "rotational_speed_rpm", "Torque [Nm]": "torque_nm",
               "Tool wear [min]": "tool_wear_min", "Machine failure": "machine_failure",
               "TWF": "twf", "HDF": "hdf", "PWF": "pwf", "OSF": "osf", "RNF": "rnf"}
    assert list(df.columns) == list(mapping), f"D1 columns differ from documentation: {list(df.columns)}"
    assert len(df) == 10000, f"D1 expected 10000 rows, got {len(df)}"
    df = df.rename(columns=mapping)
    df["type"] = df["type"].astype(str)
    for c in ["air_temperature_k", "process_temperature_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min"]:
        df[c] = df[c].astype(float)
    df["machine_failure"] = df["machine_failure"].astype(int)
    df.insert(0, "observation_id", [f"D1-{i:06d}" for i in range(len(df))])
    df["_group"] = np.arange(len(df))
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D1", "AI4I 2020", "binary", "machine_failure",
                      ["type", "air_temperature_k", "process_temperature_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min"],
                      ["type"], ["uid", "product_id", "twf", "hdf", "pwf", "osf", "rnf"], None, "temporal",
                      "average_precision", [0, 1],
                      notes="The identifier column is UDI in the archive (the protocol text says UID); mapped to uid and excluded.")
    return df, spec, mapping


def import_d2(raw: Path) -> tuple[pd.DataFrame, FamilySpec, dict]:
    names = _zip_member(raw / "uci_198.zip", "Faults27x7_var").decode().split()
    data = pd.read_csv(io.BytesIO(_zip_member(raw / "uci_198.zip", "Faults.NNA")), sep=r"\s+", header=None)
    assert len(names) == 34 and data.shape == (1941, 34), f"D2 shape {data.shape}, {len(names)} names"
    data.columns = names
    labels = names[27:]
    assert labels == STEEL_LABELS, f"D2 label names differ: {labels}"
    active = data[labels].sum(axis=1)
    assert (active == 1).all(), "D2: every row must have exactly one active target indicator"
    mapping = {n: _snake(n) for n in names}
    df = data.rename(columns=mapping)
    lab_cols = [mapping[l] for l in labels]
    df["fault_class"] = [labels[i] for i in np.argmax(df[lab_cols].to_numpy(), axis=1)]
    feats = [mapping[n] for n in names[:27]]
    for c in feats:
        df[c] = df[c].astype(float)
    df.insert(0, "observation_id", [f"D2-{i:06d}" for i in range(len(df))])
    # group identical predictor rows
    key = df[feats].round(10).astype(str).agg("|".join, axis=1)
    df["_group"] = pd.factorize(key)[0]
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D2", "Steel Plates Faults", "multiclass", "fault_class", feats, [], lab_cols, "_group",
                      "stratified_group", "macro_f1", list(STEEL_LABELS))
    mapping["__label_mapping__"] = {l: l for l in labels}
    return df, spec, mapping


def import_d3(raw: Path) -> tuple[pd.DataFrame, FamilySpec, dict]:
    data = pd.read_excel(io.BytesIO(_zip_member(raw / "uci_165.zip", "Concrete_Data.xls")))
    assert data.shape == (1030, 9), f"D3 shape {data.shape}"
    canon = ["cement_kg_m3", "blast_furnace_slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticizer_kg_m3",
             "coarse_aggregate_kg_m3", "fine_aggregate_kg_m3", "age_day", "compressive_strength_mpa"]
    expected_tokens = ["Cement", "Blast Furnace Slag", "Fly Ash", "Water", "Superplasticizer", "Coarse Aggregate",
                       "Fine Aggregate", "Age", "compressive strength"]
    for col, tok in zip(data.columns, expected_tokens):
        assert tok.lower() in str(col).lower(), f"D3 column mapping mismatch: {col!r} vs {tok!r}"
    mapping = {str(c): n for c, n in zip(data.columns, canon)}
    df = data.copy()
    df.columns = canon
    df = df.astype(float)
    df.insert(0, "observation_id", [f"D3-{i:06d}" for i in range(len(df))])
    comp = df[canon[:7]].round(6).astype(str).agg("|".join, axis=1)
    df["_group"] = pd.factorize(comp)[0]
    df["mix_id"] = [f"MIX-{g:04d}" for g in df["_group"]]
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D3", "Concrete Compressive Strength", "regression", "compressive_strength_mpa", canon[:8], [],
                      ["mix_id"], "_group", "group", "rmse")
    return df, spec, mapping


def import_d4(raw: Path) -> tuple[pd.DataFrame, FamilySpec, dict]:
    data = pd.read_excel(io.BytesIO(_zip_member(raw / "uci_242.zip", "ENB2012_data.xlsx")))
    data = data.dropna(how="all").dropna(axis=1, how="all")
    assert list(data.columns)[:10] == ["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8", "Y1", "Y2"], list(data.columns)
    assert len(data) == 768, f"D4 expected 768 rows, got {len(data)}"
    names = ["relative_compactness", "surface_area", "wall_area", "roof_area", "overall_height", "orientation",
             "glazing_area", "glazing_area_distribution", "heating_load", "cooling_load"]
    mapping = dict(zip(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8", "Y1", "Y2"], names))
    df = data[list(mapping)].rename(columns=mapping).astype(float)
    df.insert(0, "observation_id", [f"D4-{i:06d}" for i in range(len(df))])
    geo = df[names[:5]].round(6).astype(str).agg("|".join, axis=1)
    df["_group"] = pd.factorize(geo)[0]
    assert df["_group"].nunique() == 12, f"D4 expected 12 base geometries (X1-X5), got {df['_group'].nunique()}"
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D4", "Energy Efficiency", "regression", "heating_load", names[:8], [], ["cooling_load"], "_group",
                      "geometry_groups", "rmse")
    return df, spec, mapping


# --------------------------------------------------------------------------- synthetic generators
def generate_d5(n_machines: int = 500, per_machine: int = 20, namespace: str = "evaluation", prefix: str = "D5"):
    """SYN-MACHINE-v1 (Section 5.2)."""
    rng = make_rng(namespace, "D5", "generator", "", "SYN-MACHINE-v1")
    b = rng.standard_normal((n_machines, 6))
    cat = rng.choice(np.array(["A", "B", "C"]), size=n_machines)
    eps = rng.standard_normal((n_machines, per_machine, 6))
    z = 0.4 * b[:, None, :] + np.sqrt(0.84) * eps
    eta = (-2.5 + 0.9 * z[..., 0] + 0.7 * z[..., 1] + 0.5 * z[..., 0] * z[..., 2]
           + 0.6 * (cat == "C")[:, None].astype(float))
    p = 1.0 / (1.0 + np.exp(-eta))
    y = (rng.random((n_machines, per_machine)) < p).astype(int)
    mids = [f"{prefix}M-{m:04d}" for m in range(n_machines)]
    machines = pd.DataFrame({"machine_id": mids, "machine_category": cat})
    rows = []
    for m in range(n_machines):
        for i in range(per_machine):
            rows.append((m, i))
    mi = np.array(rows)
    obs = pd.DataFrame({
        "observation_id": [f"{prefix}-{k:06d}" for k in range(len(mi))],
        "machine_id": [mids[m] for m in mi[:, 0]],
        **{f"sensor_{j + 1}": z[mi[:, 0], mi[:, 1], j] for j in range(6)},
        "failure": y[mi[:, 0], mi[:, 1]],
    })
    diag = pd.DataFrame({"observation_id": obs["observation_id"], "p_true": p[mi[:, 0], mi[:, 1]]})
    for j in range(6):
        diag[f"latent_b{j + 1}"] = b[mi[:, 0], j]
    df = obs.merge(machines, on="machine_id", how="left", validate="many_to_one")
    df["_group"] = mi[:, 0]
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D5", "SYN-MACHINE-v1", "binary", "failure",
                      [f"sensor_{j}" for j in range(1, 7)] + ["machine_category"], ["machine_category"],
                      ["machine_id"], "_group", "stratified_group", "average_precision", [0, 1], synthetic=True)
    return df, spec, {"machines": machines, "observations": obs, "diagnostics": diag}


def generate_d6(n_batches: int = 300, per_batch: int = 30, namespace: str = "evaluation", prefix: str = "D6"):
    """SYN-PROCESS-v1 (Section 5.3)."""
    rng = make_rng(namespace, "D6", "generator", "", "SYN-PROCESS-v1")
    u = rng.standard_normal(n_batches)
    x = rng.uniform(-1, 1, (n_batches, per_batch, 8)) + 0.2 * u[:, None, None]
    eps = rng.standard_normal((n_batches, per_batch))
    y = (50 + 10 * x[..., 0] - 7 * x[..., 1] + 5 * x[..., 2] * x[..., 3] + 4 * np.sin(np.pi * x[..., 4])
         + 2 * u[:, None] + eps)
    bids = [f"{prefix}B-{b:04d}" for b in range(n_batches)]
    batches = pd.DataFrame({"batch_id": bids, "line": [f"L{(b % 3) + 1}" for b in range(n_batches)]})
    bi = np.array([(b, i) for b in range(n_batches) for i in range(per_batch)])
    obs = pd.DataFrame({
        "observation_id": [f"{prefix}-{k:06d}" for k in range(len(bi))],
        "batch_id": [bids[b] for b in bi[:, 0]],
        **{f"x{j + 1}": x[bi[:, 0], bi[:, 1], j] for j in range(8)},
        "process_output": y[bi[:, 0], bi[:, 1]],
    })
    diag = pd.DataFrame({"observation_id": obs["observation_id"], "latent_u": u[bi[:, 0]]})
    df = obs.copy()
    df["_group"] = bi[:, 0]
    df["_order"] = np.arange(len(df))
    spec = FamilySpec("D6", "SYN-PROCESS-v1", "regression", "process_output", [f"x{j}" for j in range(1, 9)], [],
                      ["batch_id"], "_group", "group", "rmse", synthetic=True)
    return df, spec, {"batches": batches, "observations": obs, "diagnostics": diag}


DEV_SIZES = {"D5": (100, 20), "D6": (60, 30)}


def build_canonical(paths: Paths, include_public: bool = True) -> dict:
    """Write canonical tables + specs for the evaluation suite and the development suite."""
    out = {}
    imp = {"D1": import_d1, "D2": import_d2, "D3": import_d3, "D4": import_d4}
    mappings = {}
    for fam in (["D1", "D2", "D3", "D4"] if include_public else []):
        df, spec, mapping = imp[fam](paths.raw)
        mappings[fam] = mapping
        out[("evaluation", fam)] = (df, spec)
    for suite, ns in [("evaluation", "evaluation"), ("development", "development")]:
        n5 = (500, 20) if suite == "evaluation" else DEV_SIZES["D5"]
        n6 = (300, 30) if suite == "evaluation" else DEV_SIZES["D6"]
        p5 = "D5" if suite == "evaluation" else "D5X"
        p6 = "D6" if suite == "evaluation" else "D6X"
        df5, s5, parts5 = generate_d5(*n5, namespace=ns, prefix=p5)
        df6, s6, parts6 = generate_d6(*n6, namespace=ns, prefix=p6)
        out[(suite, "D5")] = (df5, s5)
        out[(suite, "D6")] = (df6, s6)
        d = paths.canonical / suite
        d.mkdir(parents=True, exist_ok=True)
        parts5["machines"].to_parquet(d / "D5_machines.parquet", index=False)
        parts5["diagnostics"].to_parquet(d / "D5_diagnostics.parquet", index=False)
        parts6["batches"].to_parquet(d / "D6_batches.parquet", index=False)
        parts6["diagnostics"].to_parquet(d / "D6_diagnostics.parquet", index=False)
    summary = {}
    for (suite, fam), (df, spec) in out.items():
        d = paths.canonical / suite
        d.mkdir(parents=True, exist_ok=True)
        df.to_parquet(d / f"{fam}.parquet", index=False)
        write_json(d / f"{fam}_spec.json", spec.to_dict())
        info = {"rows": len(df), "groups": int(df["_group"].nunique()), "hash": df_hash(df),
                "features": len(spec.features), "task_type": spec.task_type}
        if spec.task_type != "regression":
            vc = df[spec.target].value_counts().sort_index()
            info["class_counts"] = {str(k): int(v) for k, v in vc.items()}
        summary[f"{suite}/{fam}"] = info
    write_json(paths.canonical / "column_mappings.json", {k: {kk: vv for kk, vv in v.items()} for k, v in mappings.items()})
    # Development and evaluation must not share observation IDs
    for fam in ["D5", "D6"]:
        ev = set(out[("evaluation", fam)][0]["observation_id"])
        dv = set(out[("development", fam)][0]["observation_id"])
        assert not (ev & dv), f"{fam}: development and evaluation share observation IDs"
    write_json(paths.manifests / "canonical.json", summary)
    return summary


def load_canonical(paths: Paths, suite: str, family: str) -> tuple[pd.DataFrame, FamilySpec]:
    d = paths.canonical / suite
    df = pd.read_parquet(d / f"{family}.parquet")
    spec = FamilySpec(**read_json(d / f"{family}_spec.json"))
    return df, spec


# --------------------------------------------------------------------------- splits
def _assign_groups(groups: np.ndarray, fractions, rng) -> dict:
    """Assign whole groups to partitions approximating the target fractions of rows."""
    uniq, counts = np.unique(groups, return_counts=True)
    order = rng.permutation(len(uniq))
    total = counts.sum()
    targets = np.cumsum(np.array(fractions)) * total
    assign = {}
    acc = 0
    names = ["train", "validation", "test"]
    for idx in order:
        acc_mid = acc + counts[idx] / 2.0
        part = int(np.searchsorted(targets, acc_mid, side="right"))
        part = min(part, 2)
        assign[uniq[idx]] = names[part]
        acc += counts[idx]
    return assign


def _stratified_group_assign(y: np.ndarray, groups: np.ndarray, rng, fractions=(0.6, 0.2, 0.2)) -> np.ndarray:
    """Stratified group-aware 60/20/20: greedy assignment of shuffled groups to the partition with
    the largest deficit for the group's dominant class."""
    names = np.array(["train", "validation", "test"])
    classes = np.unique(y)
    gdf = pd.DataFrame({"g": groups, "y": y})
    gcounts = gdf.groupby(["g", "y"]).size().unstack(fill_value=0).reindex(columns=classes, fill_value=0)
    totals = gcounts.sum(axis=0).to_numpy().astype(float)
    target = np.outer(np.array(fractions), totals)  # partitions x classes
    have = np.zeros_like(target)
    order = rng.permutation(len(gcounts))
    # largest groups first improves balance
    sizes = gcounts.sum(axis=1).to_numpy()
    order = order[np.argsort(-sizes[order], kind="stable")]
    assign = {}
    for i in order:
        c = gcounts.iloc[i].to_numpy().astype(float)
        deficit = ((target - have) * (c[None, :] > 0)).sum(axis=1) / np.maximum(c.sum(), 1)
        p = int(np.argmax(deficit))
        have[p] += c
        assign[gcounts.index[i]] = names[p]
    return np.array([assign[g] for g in groups])


def make_splits(df: pd.DataFrame, spec: FamilySpec, suite: str, repetition: int) -> pd.DataFrame:
    """Return a frame (observation_id, partition, fold) for one repetition (Section 8.1)."""
    fam = spec.family
    rng = make_rng("split", fam, suite, repetition, "outer")
    n = len(df)
    ids = df["observation_id"].to_numpy()
    groups = df["_group"].to_numpy()
    if spec.split_rule == "temporal":
        # fixed across repetitions: original order 60/20/20, forward-chaining folds
        order = df["_order"].to_numpy()
        part = np.where(order < int(0.6 * n), "train", np.where(order < int(0.8 * n), "validation", "test"))
        fold = np.full(n, -1)
        tr = order < int(0.6 * n)
        tr_order = order[tr]
        fold[tr] = np.minimum((tr_order * 4) // int(0.6 * n), 3)  # chunks 0..3
    elif spec.split_rule == "geometry_groups":
        uniq = np.unique(groups)
        assert len(uniq) == 12
        perm = rng.permutation(uniq)
        gpart = {g: ("train" if i < 7 else "validation" if i < 9 else "test") for i, g in enumerate(perm)}
        part = np.array([gpart[g] for g in groups])
        fold = np.full(n, -1)
        trg = perm[:7]
        frng = make_rng("split", fam, suite, repetition, "folds")
        trg = frng.permutation(trg)
        gfold = {g: i % 3 for i, g in enumerate(trg)}
        for i in range(n):
            if part[i] == "train":
                fold[i] = gfold[groups[i]]
    else:
        y = df[spec.target].to_numpy()
        if spec.split_rule == "stratified_group":
            part = _stratified_group_assign(y, groups, rng)
        else:
            assign = _assign_groups(groups, (0.6, 0.2, 0.2), rng)
            part = np.array([assign[g] for g in groups])
        fold = np.full(n, -1)
        tr = part == "train"
        frng = make_rng("split", fam, suite, repetition, "folds")
        if spec.task_type != "regression":
            from sklearn.model_selection import StratifiedGroupKFold
            sgk = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=small_seed(int(frng.integers(0, 2**31 - 1))))
            tr_idx = np.where(tr)[0]
            for k, (_, te) in enumerate(sgk.split(np.zeros(len(tr_idx)), y[tr_idx], groups[tr_idx])):
                fold[tr_idx[te]] = k
        else:
            tg = np.unique(groups[tr])
            tg = frng.permutation(tg)
            gfold = {g: i % 3 for i, g in enumerate(tg)}
            for i in np.where(tr)[0]:
                fold[i] = gfold[groups[i]]
    return pd.DataFrame({"observation_id": ids, "partition": part, "fold": fold.astype(int), "group": groups})


def cv_splits(split_df: pd.DataFrame, rule: str) -> list:
    """List of (train_ids, valid_ids) for the three inner folds of the training partition."""
    tr = split_df[split_df["partition"] == "train"]
    out = []
    if rule == "temporal":
        for k in range(3):
            out.append((tr.loc[tr["fold"] <= k, "observation_id"].tolist(), tr.loc[tr["fold"] == k + 1, "observation_id"].tolist()))
    else:
        for k in range(3):
            out.append((tr.loc[tr["fold"] != k, "observation_id"].tolist(), tr.loc[tr["fold"] == k, "observation_id"].tolist()))
    return out


def check_split(df: pd.DataFrame, spec: FamilySpec, split_df: pd.DataFrame) -> dict:
    """Feasibility assertions (Section 8.1). Raises on failure."""
    assert len(split_df) == len(df) and set(split_df["observation_id"]) == set(df["observation_id"])
    g = split_df.groupby("group")["partition"].nunique()
    if spec.split_rule != "temporal":
        assert (g == 1).all(), f"{spec.family}: a group spans more than one partition"
    counts = split_df["partition"].value_counts().to_dict()
    for p in ["train", "validation", "test"]:
        assert counts.get(p, 0) > 0, f"{spec.family}: empty partition {p}"
    info = {"sizes": {k: int(v) for k, v in counts.items()}}
    if spec.task_type != "regression":
        m = df[["observation_id", spec.target]].merge(split_df, on="observation_id")
        for p in ["train", "validation", "test"]:
            present = set(m.loc[m["partition"] == p, spec.target])
            assert present == set(spec.classes) or set(map(str, present)) == set(map(str, spec.classes)), \
                f"{spec.family}: partition {p} lacks classes {set(spec.classes) - present}"
        for k, (a, b) in enumerate(cv_splits(split_df, spec.split_rule)):
            ya = set(m.set_index("observation_id").loc[a, spec.target])
            yb = set(m.set_index("observation_id").loc[b, spec.target])
            assert len(ya) == len(spec.classes), f"{spec.family}: fold {k} training lacks a class"
            assert len(yb) >= 2, f"{spec.family}: fold {k} validation has fewer than two classes"
        info["positives"] = {p: int(((m["partition"] == p) & (m[spec.target] == 1)).sum()) for p in ["train", "validation", "test"]} \
            if spec.task_type == "binary" else None
    if spec.split_rule == "geometry_groups":
        ng = split_df.groupby("partition")["group"].nunique().to_dict()
        assert ng == {"train": 7, "validation": 2, "test": 3}, f"D4 geometry assignment {ng}"
        info["groups"] = ng
    return info


def build_splits(paths: Paths, suite: str, families: list, repetitions: list) -> dict:
    summary = {}
    for fam in families:
        df, spec = load_canonical(paths, suite, fam)
        for r in repetitions:
            s = make_splits(df, spec, suite, r)
            info = check_split(df, spec, s)
            d = paths.canonical / suite / "splits"
            d.mkdir(parents=True, exist_ok=True)
            s.to_parquet(d / f"{fam}_r{r}.parquet", index=False)
            info["hash"] = df_hash(s)
            summary[f"{suite}/{fam}/r{r}"] = info
    existing = read_json(paths.manifests / "splits.json") if (paths.manifests / "splits.json").exists() else {}
    existing.update(summary)
    write_json(paths.manifests / "splits.json", existing)
    return summary


def load_split(paths: Paths, suite: str, family: str, rep: int) -> pd.DataFrame:
    return pd.read_parquet(paths.canonical / suite / "splits" / f"{family}_r{rep}.parquet")


### Shared preprocessing, candidate grids and metrics (Section 10)

In [ ]:
%%writefile adm_code/adm/modeling.py
"""Shared preprocessing, the fixed candidate grids (Section 10), fitting and metrics."""
from __future__ import annotations

import math
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from .core import derive_seed, small_seed

CLASSIFICATION = {
    "CL1": ("logistic", {"C": 0.1, "solver": "lbfgs", "max_iter": 2000}),
    "CL2": ("logistic", {"C": 1.0, "solver": "lbfgs", "max_iter": 2000}),
    "CL3": ("rf", {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"}),
    "CL4": ("rf", {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 5, "max_features": "sqrt"}),
    "CL5": ("hgb", {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 15, "early_stopping": False}),
    "CL6": ("hgb", {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 31, "early_stopping": False}),
}
REGRESSION = {
    "RG1": ("ridge", {"alpha": 1.0}),
    "RG2": ("ridge", {"alpha": 10.0}),
    "RG3": ("rf", {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1, "max_features": 1.0}),
    "RG4": ("rf", {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 5, "max_features": 1.0}),
    "RG5": ("hgb", {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 15, "early_stopping": False}),
    "RG6": ("hgb", {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 31, "early_stopping": False}),
}
MAX_ONEHOT_CATEGORIES = 64  # safeguard against identifier columns used as categorical predictors


def candidate_ids(task_type: str) -> list:
    return list(REGRESSION) if task_type == "regression" else list(CLASSIFICATION)


def is_linear(cid: str) -> bool:
    return cid in ("CL1", "CL2", "RG1", "RG2")


class CategoricalImputer(BaseEstimator, TransformerMixin):
    """Training-fitted most-frequent imputation; an entirely missing training column uses 'MISSING'."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X).astype(object)
        self.fill_ = []
        for c in X.columns:
            s = X[c].dropna()
            if len(s) == 0:
                self.fill_.append("MISSING")
            else:
                vc = s.astype(str).value_counts()
                top = vc[vc == vc.max()].index.sort_values()[0]
                self.fill_.append(top)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).astype(object).copy()
        for i, c in enumerate(X.columns):
            X[c] = X[c].where(X[c].notna(), self.fill_[i]).astype(str)
        return X.to_numpy()


def make_estimator(cid: str, seed: int):
    kind, params = (CLASSIFICATION.get(cid) or REGRESSION.get(cid))
    rs = small_seed(seed)
    if cid.startswith("CL"):
        if kind == "logistic":
            return LogisticRegression(**params)
        if kind == "rf":
            return RandomForestClassifier(**params, n_jobs=1, random_state=rs)
        return HistGradientBoostingClassifier(**params, random_state=rs)
    if kind == "ridge":
        return Ridge(**params)
    if kind == "rf":
        return RandomForestRegressor(**params, n_jobs=1, random_state=rs)
    return HistGradientBoostingRegressor(**params, random_state=rs)


def make_pipeline(cid: str, numeric: list, categorical: list, seed: int) -> Pipeline:
    num_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if is_linear(cid):
        num_steps.append(("scale", StandardScaler()))
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline(num_steps), list(numeric)))
    if categorical:
        transformers.append(("cat", Pipeline([("impute", CategoricalImputer()),
                                              ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                                       max_categories=MAX_ONEHOT_CATEGORIES,
                                                                       sparse_output=False))]), list(categorical)))
    pre = ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("model", make_estimator(cid, seed))])


def effective_params(cid: str) -> dict:
    est = make_estimator(cid, 0)
    return {k: (v if isinstance(v, (int, float, str, bool, type(None))) else repr(v)) for k, v in est.get_params().items()}


class FeatureTypeError(ValueError):
    pass


def split_columns(X: pd.DataFrame, contract: dict) -> tuple[list, list]:
    """Assign columns of a materialised feature table to the numeric or categorical branch."""
    num, cat = [], []
    contract_num = set(contract.get("numeric_features", []))
    contract_cat = set(contract.get("categorical_features", []))
    for c in X.columns:
        if c == "observation_id":
            continue
        if c in contract_cat:
            cat.append(c)
        elif pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c]):
            num.append(c)
        elif c in contract_num:
            bad = X[c].dropna().astype(str)
            ex = bad[~bad.str.match(r"^-?[0-9.eE+-]+$")].head(2).tolist()
            raise FeatureTypeError(f"numeric feature '{c}' has non-numeric values (examples: {ex}); "
                                   "convert it with parse_numeric before modeling")
        else:
            cat.append(c)
    return num, cat


def attach_labels(X: pd.DataFrame, labels: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, dict]:
    m = X.merge(labels[["observation_id", "target"]], on="observation_id", how="inner")
    info = {"feature_rows": int(len(X)), "labelled_rows": int(len(m)), "unmatched_rows": int(len(X) - len(m))}
    return m.drop(columns=["target"]), m["target"].to_numpy(), info


# --------------------------------------------------------------------------- metrics
def _metric(task_type: str, name: str, y, pred) -> tuple[float | None, str | None]:
    try:
        if task_type == "regression":
            if name == "rmse":
                return float(math.sqrt(mean_squared_error(y, pred))), None
            if name == "mae":
                return float(mean_absolute_error(y, pred)), None
        if task_type == "binary":
            p = pred
            if name == "average_precision":
                if len(set(y)) < 2:
                    return None, "only one class present"
                return float(average_precision_score(y, p)), None
            if name == "roc_auc":
                if len(set(y)) < 2:
                    return None, "only one class present"
                return float(roc_auc_score(y, p)), None
            if name == "macro_f1":
                return float(f1_score(y, (p >= 0.5).astype(int), average="macro", labels=[0, 1], zero_division=0)), None
        if task_type == "multiclass":
            labels, proba = pred
            yhat = np.asarray(labels)[np.argmax(proba, axis=1)]
            if name == "macro_f1":
                return float(f1_score(y, yhat, average="macro", labels=list(labels), zero_division=0)), None
            if name == "balanced_accuracy":
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    return float(balanced_accuracy_score(y, yhat)), None
    except Exception as e:  # pragma: no cover
        return None, f"{type(e).__name__}: {e}"
    return None, f"unknown metric {name}"


def predict_scores(model: Pipeline, X: pd.DataFrame, task_type: str):
    if task_type == "regression":
        return model.predict(X)
    proba = model.predict_proba(X)
    if task_type == "binary":
        classes = list(model.classes_)
        return proba[:, classes.index(1)] if 1 in classes else np.zeros(len(X))
    return (list(model.classes_), proba)


def selection_score(task_type: str, metric: str, y, pred):
    return _metric(task_type, metric, y, pred)


def full_metrics(task_type: str, y, pred) -> dict:
    out, reasons = {}, {}
    names = {"binary": ["average_precision", "macro_f1", "roc_auc"], "multiclass": ["macro_f1", "balanced_accuracy"],
             "regression": ["rmse", "mae"]}[task_type]
    for n in names:
        v, r = _metric(task_type, n, y, pred)
        out[n] = v
        if r:
            reasons[n] = r
    if task_type == "binary":
        out["prevalence"] = float(np.mean(np.asarray(y) == 1)) if len(y) else None
    if task_type == "multiclass":
        labels, proba = pred
        yhat = np.asarray(labels)[np.argmax(proba, axis=1)]
        out["confusion_matrix"] = {"labels": list(map(str, labels)),
                                   "matrix": confusion_matrix(y, yhat, labels=labels).tolist()}
    if reasons:
        out["undefined_reasons"] = reasons
    return out


def higher_is_better(metric: str) -> bool:
    return metric not in ("rmse", "mae")


# --------------------------------------------------------------------------- fit / evaluate
def fit_evaluate(cid: str, train_X: pd.DataFrame, val_X: pd.DataFrame, labels: pd.DataFrame, cv_folds: list,
                 contract: dict, seed: int) -> dict:
    """Fit one permitted candidate: three inner CV folds on training plus a validation score."""
    t0 = time.time()
    task_type = contract["task_type"]
    metric = contract["selection_metric"]
    if cid not in candidate_ids(task_type):
        raise ValueError(f"candidate {cid} is not permitted; choose one of {candidate_ids(task_type)}")
    tr_lab = labels[labels["partition"] == "train"]
    va_lab = labels[labels["partition"] == "validation"]
    Xtr, ytr, info_tr = attach_labels(train_X, tr_lab)
    Xva, yva, info_va = attach_labels(val_X, va_lab)
    if len(Xtr) == 0:
        raise ValueError("no training rows matched canonical labelled observation_ids")
    num, cat = split_columns(Xtr, contract)
    split_columns(Xva, contract)
    fits = 0
    folds = []
    ids = Xtr["observation_id"].to_numpy()
    for k, (a, b) in enumerate(cv_folds):
        ma, mb = np.isin(ids, a), np.isin(ids, b)
        if ma.sum() == 0 or mb.sum() == 0:
            folds.append({"fold": k, "score": None, "reason": "empty fold after label matching"})
            continue
        pipe = make_pipeline(cid, num, cat, seed)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe.fit(Xtr.loc[ma, num + cat], ytr[ma])
        fits += 1
        s, r = selection_score(task_type, metric, ytr[mb], predict_scores(pipe, Xtr.loc[mb, num + cat], task_type))
        folds.append({"fold": k, "score": s, "reason": r})
    pipe = make_pipeline(cid, num, cat, seed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipe.fit(Xtr[num + cat], ytr)
    fits += 1
    if len(Xva):
        vs, vr = selection_score(task_type, metric, yva, predict_scores(pipe, Xva[num + cat], task_type))
    else:
        vs, vr = None, "no validation rows matched labels"
    cv_scores = [f["score"] for f in folds if f["score"] is not None]
    return {
        "candidate": cid, "metric": metric,
        "cv_mean": float(np.mean(cv_scores)) if cv_scores else None, "cv_folds": folds,
        "validation": vs, "validation_reason": vr,
        "rows": {"train": info_tr, "validation": info_va,
                 "validation_coverage": (info_va["labelled_rows"] / max(1, len(va_lab)))},
        "features": {"numeric": num, "categorical": cat},
        "fits": fits, "seconds": round(time.time() - t0, 3),
    }


def refit(cid: str, train_X: pd.DataFrame, val_X: pd.DataFrame, labels: pd.DataFrame, contract: dict, seed: int):
    """Trusted final refit on training plus validation (the test partition is never used)."""
    X = pd.concat([train_X, val_X], ignore_index=True)
    lab = labels[labels["partition"].isin(["train", "validation"])]
    Xl, y, info = attach_labels(X, lab)
    num, cat = split_columns(Xl, contract)
    pipe = make_pipeline(cid, num, cat, seed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipe.fit(Xl[num + cat], y)
    return pipe, {"numeric": num, "categorical": cat, "rows": info, "fit_partitions": ["train", "validation"]}


def select_best(results: list, metric: str) -> str | None:
    """Best validation score; ties within 1e-12 broken by lexicographic candidate ID."""
    ok = [r for r in results if r.get("validation") is not None]
    if not ok:
        return None
    hib = higher_is_better(metric)
    best = max(r["validation"] for r in ok) if hib else min(r["validation"] for r in ok)
    tied = sorted(r["candidate"] for r in ok if abs(r["validation"] - best) <= 1e-12)
    return tied[0]


def dummy_model(task_type: str):
    if task_type == "regression":
        return DummyRegressor(strategy="mean")
    return DummyClassifier(strategy="prior")


### Task templates, catalogs, contracts, defects, negative controls, hidden references (Sections 6-7)

In [ ]:
%%writefile adm_code/adm/tasks.py
"""Task templates T1-T6, catalogs, contracts, controlled defects C1-C6, negative controls N1/N2,
hidden corruption ledgers and independently computed reference features (Sections 6-7)."""
from __future__ import annotations

import gzip
import hashlib
import json
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from .core import (Paths, TEMPLATES, base_task_index, df_hash, instance_dirs, make_rng, read_json,
                   write_json, sha256_file, json_hash, dumps)
from .datasets import FamilySpec, load_canonical, load_split

N_SHARDS = 20
AS_OF = "2025-06-30"
SNAPSHOTS = {"v1": "2024-12-31", "v2": "2025-06-30", "v3": "2025-09-30"}
MODIFIED = {"v1": "2024-12-31T00:00:00Z", "v2": "2025-06-30T00:00:00Z", "v3": "2025-09-30T00:00:00Z",
            "refresh": "2025-10-15T00:00:00Z"}
READINGS_PER_SERIES = 5
CONDITION_SPEC = {
    "C0": ("clean", 0.0), "C1": ("cell", 0.05), "C2": ("cell", 0.20), "C3": ("key", 0.05), "C4": ("key", 0.20),
    "C5": ("semantic", 0.05), "C6": ("semantic", 0.20), "N1": ("negative", 0.0), "N2": ("negative", 0.0),
}
SEMANTIC_SHARDS = {"C5": 1, "C6": 4}
# Tier 2 (main-plan-v2.1 Section 7.6): recoverable defects outside the declared data-format specification, each at three
# graded levels for calibration on development tasks. X1 near-duplicates (differ only in the packaging column
# _ingest_batch), X2 key-format variants not covered by the alias table, X3 undeclared shard representations.
TIER2 = {"X1a": ("near_dup", 0.05), "X1b": ("near_dup", 0.10), "X1c": ("near_dup", 0.20),
         "X2a": ("key_variant", 0.05), "X2b": ("key_variant", 0.10), "X2c": ("key_variant", 0.20),
         "X3a": ("undeclared_rep", 1), "X3b": ("undeclared_rep", 2), "X3c": ("undeclared_rep", 4),
         # X4 content drift (provenance type): the primary authorised file's content for k shards no longer matches the
         # authority hash, while the catalog file entry still declares the authorised hash; a replica with the authorised
         # content exists. Detectable only by verifying content hashes (provenance check); repaired by
         # load policy 'authoritative_verified'.
         "X4a": ("content_drift", 1), "X4b": ("content_drift", 2), "X4c": ("content_drift", 4)}
CONDITION_SPEC.update(TIER2)
from .core import TIER2_CONDITIONS as _T2, TIER2_DESIGN as _T2D  # noqa: E402
assert set(_T2) | set(_T2D) == set(TIER2)
TIER2_KINDS = {"near_dup", "key_variant", "undeclared_rep", "content_drift"}
REPLICA_MODIFIED = "2025-06-29T12:00:00Z"  # earlier than the primary: modification-time heuristics never pick it
GENERATOR_VERSION = 2  # cached instances built by an older generator are rebuilt (content digests are still verified)
PACKAGING_COLUMNS = ("_shard", "_ingest_batch")
PARTS = ["train", "validation", "test"]
VISIBLE_PARTS = ["train", "validation"]
TEMPLATE_DESCRIPTIONS = {
    "T1": "Single-source modeling: locate the observation-level feature source, respect predictor exclusions, prepare data and fit a model.",
    "T2": "Two-table reconstruction: permitted predictors are split across two vertical tables joined one-to-one by observation_id.",
    "T3": "Three-table integration: observations, a second feature block and a profile metadata relation joined through declared keys.",
    "T4": "Unit/representation harmonization: numeric features are stored in shard-level representations that must be converted to canonical units.",
    "T5": "Version-aware analysis: several snapshots exist; select the version required by the authoritative as-of catalog.",
    "T6": "Aggregate-feature construction: two predictors are supplied only as child readings that must be summarised before modeling.",
}


# --------------------------------------------------------------------------- numeric grammar
def encode_value(x: float) -> str:
    return f"value:{repr(float(x))}"


def plain_literal(x) -> str | None:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    return repr(float(x))


# --------------------------------------------------------------------------- representations
_REAL_UNITS = {
    "air_temperature_k": [("celsius", 1.0, -273.15), ("fahrenheit", 1.8, -459.67)],
    "process_temperature_k": [("celsius", 1.0, -273.15), ("fahrenheit", 1.8, -459.67)],
    "rotational_speed_rpm": [("rad_per_s", 2 * math.pi / 60.0, 0.0), ("rev_per_s", 1 / 60.0, 0.0)],
    "torque_nm": [("kilonewton_metre", 0.001, 0.0), ("pound_force_foot", 0.737562149277, 0.0)],
    "tool_wear_min": [("hours", 1 / 60.0, 0.0), ("seconds", 60.0, 0.0)],
    "age_day": [("weeks", 1 / 7.0, 0.0), ("hours", 24.0, 0.0)],
}
_CANON_UNITS = {"air_temperature_k": "kelvin", "process_temperature_k": "kelvin", "rotational_speed_rpm": "rpm",
                "torque_nm": "newton_metre", "tool_wear_min": "minutes", "age_day": "days"}


def representation_options(spec: FamilySpec) -> dict:
    out = {}
    for f in spec.numeric:
        if f in _REAL_UNITS:
            opts = _REAL_UNITS[f]
            canon = _CANON_UNITS[f]
        elif f.endswith("_kg_m3"):
            opts = [("pound_per_cubic_yard", 1.685555, 0.0), ("gram_per_cubic_centimetre", 0.001, 0.0)]
            canon = "kilogram_per_cubic_metre"
        else:
            opts = [("synthetic_scaled_x100", 100.0, 0.0), ("synthetic_scaled_0.01_offset_1000", 0.01, 1000.0)]
            canon = "canonical (dimensionless or as published)"
        out[f] = {"canonical_unit": canon,
                  "options": {"R0": {"a": 1.0, "b": 0.0, "unit": canon}}
                  | {f"R{i + 1}": {"a": a, "b": b, "unit": u} for i, (u, a, b) in enumerate(opts)}}
    return out


# --------------------------------------------------------------------------- shards and helpers
def shard_map(split_df: pd.DataFrame, family: str, suite: str, rep: int) -> pd.Series:
    rng = make_rng("split", family, suite, rep, "shards")
    out = {}
    for p in PARTS:
        ids = np.sort(split_df.loc[split_df["partition"] == p, "observation_id"].to_numpy())
        perm = rng.permutation(len(ids))
        for rank, i in enumerate(perm):
            out[ids[i]] = f"{p}-{rank % N_SHARDS:02d}"
    return pd.Series(out, name="_shard")


def _alias(oid: str) -> str:
    fam, num = oid.split("-")
    return f"{fam.lower()}_{int(num)}"


def _profile_key(values: tuple) -> str:
    return "P" + hashlib.sha1(repr(values).encode()).hexdigest()[:12]


def _ranges(df: pd.DataFrame, cols: list) -> dict:
    out = {}
    for c in cols:
        v = df[c].astype(float)
        lo, hi = float(v.min()), float(v.max())
        span = max(hi - lo, 1e-9)
        out[c] = [lo - 0.25 * span, hi + 0.25 * span]
    return out


# --------------------------------------------------------------------------- relational templates
def _template_layout(spec: FamilySpec, template: str) -> dict:
    """Which canonical columns go to which logical table (pure function of the template)."""
    feats = list(spec.features)
    ex = list(spec.excluded)
    if template in ("T1", "T4", "T5"):
        return {"observations": {"cols": ex + feats, "grain": "observation"}}
    if template == "T2":
        k = math.ceil(len(feats) / 2)
        return {"block_a": {"cols": ex + feats[:k], "grain": "observation"},
                "block_b": {"cols": feats[k:], "grain": "observation"}}
    if template == "T3":
        blocks = [list(b) for b in np.array_split(np.array(feats, dtype=object), 3)]
        return {"observations": {"cols": ex + blocks[0], "grain": "observation", "fk": "profile_key"},
                "measurements_b": {"cols": blocks[1], "grain": "observation"},
                "profiles": {"cols": blocks[2], "grain": "profile"}}
    if template == "T6":
        sel = spec.numeric[:2]
        rest = [f for f in feats if f not in sel]
        return {"observations": {"cols": ex + rest, "grain": "observation"},
                "readings": {"cols": ["series", "reading_index", "value"], "grain": "reading", "series": sel}}
    raise ValueError(template)


def required_features(spec: FamilySpec, template: str) -> list:
    if template != "T6":
        return list(spec.features)
    sel = spec.numeric[:2]
    out = []
    for f in spec.features:
        if f in sel:
            out += [f"{f}_mean", f"{f}_std", f"{f}_count"]
        else:
            out.append(f)
    return out


def _readings(df: pd.DataFrame, spec: FamilySpec, suite: str, sel: list) -> pd.DataFrame:
    """Five child readings per selected predictor, derived with seeded feature-only noise (no labels)."""
    ns = "evaluation" if suite == "evaluation" else "development"
    rows = []
    for f in sel:
        rng = make_rng(ns, spec.family, "T6", "", f"readings:{f}")
        v = df[f].to_numpy(dtype=float)
        sd = float(np.nanstd(v)) or 1.0
        noise = rng.normal(0.0, 0.05 * sd, size=(len(v), READINGS_PER_SERIES))
        vals = v[:, None] + noise
        for j in range(READINGS_PER_SERIES):
            rows.append(pd.DataFrame({"observation_id": df["observation_id"].to_numpy(), "series": f,
                                      "reading_index": j, "value": vals[:, j]}))
    r = pd.concat(rows, ignore_index=True)
    return r.sort_values(["observation_id", "series", "reading_index"], kind="mergesort").reset_index(drop=True)


def _snapshot(df: pd.DataFrame, spec: FamilySpec, version: str, suite: str, cols: list) -> pd.DataFrame:
    """Earlier/later snapshot: numeric features perturbed independently of labels."""
    if version == "v2":
        return df.copy()
    ns = "evaluation" if suite == "evaluation" else "development"
    rng = make_rng(ns, spec.family, "snapshot", "", version)
    out = df.copy()
    for c in cols:
        if c in spec.numeric:
            v = out[c].to_numpy(dtype=float)
            sd = float(np.nanstd(v)) or 1.0
            out[c] = v + rng.normal(0.0, 0.1 * sd, size=len(v))
    return out


def reference_features(df: pd.DataFrame, spec: FamilySpec, template: str, suite: str) -> pd.DataFrame:
    """Independent reference computation of the correct canonical feature table (all partitions, clean)."""
    req = required_features(spec, template)
    if template != "T6":
        return df[["observation_id"] + req].copy()
    sel = spec.numeric[:2]
    r = _readings(df, spec, suite, sel)
    out = df[["observation_id"] + [f for f in spec.features if f not in sel]].copy()
    for f in sel:
        g = r[r["series"] == f].groupby("observation_id")["value"]
        agg = pd.DataFrame({f"{f}_mean": g.mean(), f"{f}_std": g.std(ddof=0), f"{f}_count": g.count().astype(float)})
        out = out.merge(agg, left_on="observation_id", right_index=True, how="left")
    return out[["observation_id"] + req]


# --------------------------------------------------------------------------- instance builder
class InstanceBuilder:
    def __init__(self, paths: Paths, suite: str, family: str, template: str, condition: str, rep: int):
        self.paths, self.suite, self.family, self.template = paths, suite, family, template
        self.condition, self.rep = condition, rep
        self.df, self.spec = load_canonical(paths, suite, family)
        self.split = load_split(paths, suite, family, rep)
        self.part_of = dict(zip(self.split["observation_id"], self.split["partition"]))
        self.shards = shard_map(self.split, family, suite, rep)
        self.layout = _template_layout(self.spec, template)
        self.reps = representation_options(self.spec)
        self.ledger: list = []
        self.task = f"{family}-{template}"
        self.versions = ["v2"]
        if template == "T5":
            self.versions = ["v1", "v2", "v3"]
        self.primary = "observations" if "observations" in self.layout else "block_a"
        if condition in ("N1", "N2") and template != "T5":
            self.versions_for = {self.primary: ["v1", "v2", "v3"]}
        else:
            self.versions_for = {}
        self.tier2 = CONDITION_SPEC[condition][0] in TIER2_KINDS

    # ----- clean relational tables (canonical units, all partitions)
    def clean_tables(self) -> dict:
        d = self.df.copy()
        d["_shard"] = d["observation_id"].map(self.shards)
        d["_partition"] = d["observation_id"].map(self.part_of)
        tables = {}
        for name, info in self.layout.items():
            if info["grain"] == "observation":
                cols = ["observation_id"] + info["cols"]
                t = d[cols + ["_shard", "_partition"]].copy()
                if info.get("fk") == "profile_key":
                    pcols = self.layout["profiles"]["cols"]
                    t["profile_key"] = [_profile_key(tuple(r)) for r in d[pcols].itertuples(index=False, name=None)]
                    t = t[["observation_id", "profile_key"] + info["cols"] + ["_shard", "_partition"]]
                tables[name] = t
            elif info["grain"] == "profile":
                pcols = info["cols"]
                t = d[pcols + ["_partition"]].copy()
                t.insert(0, "profile_key", [_profile_key(tuple(r)) for r in d[pcols].itertuples(index=False, name=None)])
                t = t.drop_duplicates(["profile_key", "_partition"]).sort_values(["_partition", "profile_key"], kind="mergesort")
                t["_shard"] = t["_partition"] + "-all"
                tables[name] = t[["profile_key"] + pcols + ["_shard", "_partition"]].reset_index(drop=True)
            elif info["grain"] == "reading":
                r = _readings(d, self.spec, self.suite, info["series"])
                r["_shard"] = r["observation_id"].map(self.shards)
                r["_partition"] = r["observation_id"].map(self.part_of)
                tables[name] = r
        if self.template == "T1":
            # decoy: per-shard summary (wrong grain)
            agg = d.groupby(["_partition", "_shard"])[self.spec.numeric].mean().reset_index()
            agg.insert(0, "shard_summary_id", agg["_shard"])
            tables["observations_rollup"] = agg[["shard_summary_id"] + self.spec.numeric + ["_shard", "_partition"]]
        al = pd.DataFrame({"alias": [_alias(o) for o in d["observation_id"]], "observation_id": d["observation_id"],
                           "_partition": d["_partition"]})
        al["_shard"] = al["_partition"] + "-all"
        tables["key_aliases"] = al.sort_values(["_partition", "observation_id"], kind="mergesort").reset_index(drop=True)
        return {k: t.reset_index(drop=True) for k, t in tables.items()}

    def table_meta(self, name: str) -> dict:
        if name == "key_aliases":
            return {"grain": "alias", "key": "alias", "description": "Complete catalog mapping every permitted key alias to its canonical observation_id."}
        if name == "observations_rollup":
            return {"grain": "shard_summary", "key": "shard_summary_id",
                    "description": "Per-shard mean summary of numeric measurements (one row per packaging shard; NOT observation-level)."}
        info = self.layout[name]
        desc = {"observation": "Observation-level records keyed by observation_id.",
                "profile": "Profile metadata relation keyed by profile_key (many observations may share one profile).",
                "reading": "Child readings: several rows per observation and series (one-to-many)."}[info["grain"]]
        key = {"observation": "observation_id", "profile": "profile_key", "reading": "observation_id"}[info["grain"]]
        return {"grain": info["grain"], "key": key, "description": desc}

    # ----- defects
    def _obs_tables(self, tables: dict) -> list:
        return [n for n in self.layout if self.layout[n]["grain"] in ("observation", "reading")]

    def _cell_tables(self) -> list:
        return [n for n in self.layout if self.layout[n]["grain"] == "observation"]

    def apply_representations(self, tables: dict) -> dict:
        """Per-shard stored representations: all shards for T4, selected shards for C5/C6 unit subtype."""
        shard_reps: dict = {}
        if self.template == "T4":
            rng = make_rng("evaluation" if self.suite == "evaluation" else "development", self.family, self.task, self.rep, "T4-representations")
            t = tables["observations"]
            for shard in sorted(t["_shard"].unique()):
                for f in self.spec.numeric:
                    r = ["R0", "R1", "R2"][int(rng.integers(0, 3))]
                    if r != "R0":
                        shard_reps.setdefault("observations", {}).setdefault(shard, {})[f] = r
        return shard_reps

    @staticmethod
    def _apply_affine(t: pd.DataFrame, shard_reps_t: dict, reps: dict) -> pd.DataFrame:
        t = t.copy()
        for shard, fmap in shard_reps_t.items():
            m = t["_shard"] == shard
            for f, r in fmap.items():
                o = reps[f]["options"][r]
                t.loc[m, f] = o["a"] * t.loc[m, f].astype(float) + o["b"]
        return t

    def corrupt_cells(self, tables: dict, rate: float, part: str) -> dict:
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:cell:{part}")
        cells = []
        for n in self._cell_tables():
            t = tables[n]
            cols = [c for c in self.layout[n]["cols"] if c in self.spec.numeric]
            idx = t.index[t["_partition"] == part]
            ids = t.loc[idx, "observation_id"].to_numpy()
            for c in cols:
                valid = t.loc[idx, c].notna().to_numpy()
                for i, oid, ok in zip(idx, ids, valid):
                    if ok:
                        cells.append((oid, n, c, i))
        cells.sort(key=lambda x: (x[0], x[1], x[2]))
        N = len(cells)
        k = int(math.floor(rate * N + 0.5))
        pick = rng.permutation(N)[:k]
        n_missing = (k + 1) // 2
        string_cols = set()
        actions = []
        for j, pi in enumerate(pick):
            oid, n, c, i = cells[pi]
            actions.append((oid, n, c, i, "missing" if j < n_missing else "encoded"))
            if j >= n_missing:
                string_cols.add((n, c))
        # convert affected columns to string (plain literals) for this partition's files
        for (n, c) in string_cols:
            t = tables[n]
            if t[c].dtype != object:
                t[c] = t[c].astype(object)
                t[c] = [plain_literal(v) if isinstance(v, (float, int, np.floating, np.integer)) else v for v in t[c]]
        for oid, n, c, i, kind in actions:
            t = tables[n]
            before = t.at[i, c]
            bval = None if before is None else float(before)
            if kind == "missing":
                t.at[i, c] = None if t[c].dtype == object else np.nan
            else:
                t.at[i, c] = encode_value(bval)
            self.ledger.append({"partition": part, "defect": f"cell_{kind}", "table": n, "observation_id": oid,
                                "column": c, "before": bval, "after": None if kind == "missing" else t.at[i, c],
                                "denominator": N, "reversible": kind == "encoded"})
        return {"denominator": N, "affected": k, "missing": n_missing, "encoded": k - n_missing}

    def corrupt_keys(self, tables: dict, rate: float, part: str) -> dict:
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:key:{part}")
        ids = np.sort(self.split.loc[self.split["partition"] == part, "observation_id"].to_numpy())
        N = len(ids)
        k = int(math.floor(rate * N + 0.5))
        pick = ids[rng.permutation(N)[:k]]
        n_dup = (k + 1) // 2
        obs_tables = self._obs_tables(tables)
        index_of = {n: tables[n].groupby("observation_id").indices for n in obs_tables}
        dups: dict = {n: [] for n in obs_tables}
        for j, oid in enumerate(pick):
            tname = obs_tables[int(rng.integers(0, len(obs_tables)))]
            t = tables[tname]
            rows = index_of[tname][oid]
            if j < n_dup:
                r = int(rows[int(rng.integers(0, len(rows)))])
                dups[tname].append(r)
                self.ledger.append({"partition": part, "defect": "duplicate", "table": tname, "observation_id": oid,
                                    "denominator": N, "reversible": True})
            else:
                al = _alias(oid)
                t.loc[t.index[rows], "observation_id"] = al
                self.ledger.append({"partition": part, "defect": "alias", "table": tname, "observation_id": oid,
                                    "alias": al, "rows": int(len(rows)), "denominator": N, "reversible": True})
        for tname, rs in dups.items():
            if rs:
                t = tables[tname]
                pos = np.arange(len(t), dtype=float)
                extra = t.iloc[rs].copy()
                order = np.concatenate([pos, np.array(rs, dtype=float) + 0.5])
                merged = pd.concat([t, extra], ignore_index=True)
                tables[tname] = merged.iloc[np.argsort(order, kind="mergesort")].reset_index(drop=True)
        return {"denominator": N, "affected": k, "duplicates": n_dup, "aliases": k - n_dup}

    # ----- Tier 2 defects (outside the declared specification; every one is recoverable)
    def add_ingest_batch(self, tables: dict):
        """Tier-2 instances carry a packaging column on observation-grain tables (identical in all Tier-2 conditions,
        so its presence does not reveal which defect is present)."""
        for n in self._cell_tables():
            t = tables[n]
            t.insert(list(t.columns).index("_shard"), "_ingest_batch", "batch-01")

    def near_duplicates(self, tables: dict, rate: float, part: str) -> dict:
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:near_dup:{part}")
        ids = np.sort(self.split.loc[self.split["partition"] == part, "observation_id"].to_numpy())
        N = len(ids)
        k = int(math.floor(rate * N + 0.5))
        pick = ids[rng.permutation(N)[:k]]
        names = self._cell_tables()
        index_of = {n: tables[n].groupby("observation_id").indices for n in names}
        dups: dict = {n: [] for n in names}
        for oid in pick:
            tname = names[int(rng.integers(0, len(names)))]
            rows = index_of[tname][oid]
            dups[tname].append(int(rows[0]))
            self.ledger.append({"partition": part, "defect": "near_duplicate", "table": tname, "observation_id": oid,
                                "denominator": N, "reversible": True})
        for tname, rs in dups.items():
            if rs:
                t = tables[tname]
                extra = t.iloc[rs].copy()
                extra["_ingest_batch"] = "batch-02"
                order = np.concatenate([np.arange(len(t), dtype=float), np.array(rs, dtype=float) + 0.5])
                merged = pd.concat([t, extra], ignore_index=True)
                tables[tname] = merged.iloc[np.argsort(order, kind="mergesort")].reset_index(drop=True)
        return {"denominator": N, "affected": k}

    def key_variants(self, tables: dict, rate: float, part: str) -> dict:
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:key_variant:{part}")
        ids = np.sort(self.split.loc[self.split["partition"] == part, "observation_id"].to_numpy())
        N = len(ids)
        k = int(math.floor(rate * N + 0.5))
        pick = ids[rng.permutation(N)[:k]]
        names = self._obs_tables(tables)
        index_of = {n: tables[n].groupby("observation_id").indices for n in names}
        kinds = ["lower", "unpadded", "padded_space"]
        for oid in pick:
            tname = names[int(rng.integers(0, len(names)))]
            kind = kinds[int(rng.integers(0, len(kinds)))]
            prefix, num = oid.rsplit("-", 1)
            variant = {"lower": oid.lower(), "unpadded": f"{prefix}-{int(num)}", "padded_space": f" {oid} "}[kind]
            rows = index_of[tname][oid]
            t = tables[tname]
            t.loc[t.index[rows], "observation_id"] = variant
            self.ledger.append({"partition": part, "defect": "key_variant", "table": tname, "observation_id": oid,
                                "variant": kind, "rows": int(len(rows)), "denominator": N, "reversible": True})
        return {"denominator": N, "affected": k}

    def undeclared_representations(self, tables: dict, n_shards: int, part: str, shard_reps: dict) -> dict:
        """Store k shards per partition in a documented representation that catalog.shard_representations does not
        declare. Only (shard, column, representation) choices that are detectable against the contract value ranges
        and uniquely invertible are used, so the correct canonical values are recoverable."""
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:undeclared_rep:{part}")
        ranges = self._contract_ranges()

        def frac(v, lo, hi):
            v = v[~np.isnan(v)]
            return float(((v >= lo - 1e-9) & (v <= hi + 1e-9)).mean()) if len(v) else 1.0

        shards = sorted({s for s in self.shards.values if s.startswith(part + "-")})
        chosen = 0
        for sh in [shards[i] for i in rng.permutation(len(shards))]:
            if chosen >= n_shards:
                break
            cands = []
            for tname in self._cell_tables():
                for f in self.layout[tname]["cols"]:
                    if f in self.spec.numeric and f in ranges and f not in shard_reps.get(tname, {}).get(sh, {}):
                        for r in ("R1", "R2"):
                            cands.append((tname, f, r))
            for j in rng.permutation(len(cands)):
                tname, f, r = cands[j]
                t = tables[tname]
                m = (t["_shard"] == sh).to_numpy()
                x = t.loc[m, f].to_numpy(dtype=float)
                lo, hi = ranges[f]
                opts = self.reps[f]["options"]
                stored = opts[r]["a"] * x + opts[r]["b"]
                others = [o for k2, o in opts.items() if k2 not in ("R0", r)]
                if (m.sum() == 0 or frac(x, lo, hi) < 0.99 or frac(stored, lo, hi) >= 0.5
                        or any(frac((stored - o["b"]) / o["a"], lo, hi) >= 0.5 for o in others)):
                    continue
                t.loc[m, f] = stored
                self.ledger.append({"partition": part, "defect": "undeclared_representation", "table": tname,
                                    "shard": sh, "column": f, "to": r, "denominator": N_SHARDS, "reversible": True})
                chosen += 1
                break
        if chosen < n_shards:
            raise RuntimeError(f"{self.task}/{self.condition}/{part}: only {chosen} of {n_shards} detectable "
                               "undeclared-representation shards could be placed")
        return {"denominator": N_SHARDS, "affected": chosen}

    def plan_content_drift(self, n_shards: int, part: str) -> dict:
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:content_drift:{part}")
        shards = sorted({s for s in self.shards.values if s.startswith(part + "-")})
        names = self._cell_tables()
        for sh in [shards[i] for i in rng.permutation(len(shards))[:n_shards]]:
            tname = names[int(rng.integers(0, len(names)))]
            self.drift.setdefault((tname, part), []).append(sh)
            self.ledger.append({"partition": part, "defect": "content_drift", "table": tname, "shard": sh,
                                "denominator": N_SHARDS, "reversible": True})
        return {"denominator": N_SHARDS, "affected": n_shards}

    def _drifted(self, df: pd.DataFrame, name: str, part: str) -> pd.DataFrame:
        """Primary-file content after drift: numeric features of the drifted shards perturbed independently of labels."""
        out = df.copy()
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:drift_values:{name}:{part}")
        for sh in self.drift.get((name, part), []):
            m = (out["_shard"] == sh).to_numpy()
            for c in [c for c in out.columns if c in self.spec.numeric]:
                v = out[c].to_numpy(dtype=float)
                sd = float(np.nanstd(v)) or 1.0
                v = v.copy()
                v[m] = v[m] + rng.normal(0.0, 0.1 * sd, size=int(m.sum()))
                out[c] = v
        return out

    def semantic_plan(self, part: str) -> list:
        k = SEMANTIC_SHARDS[self.condition]
        rng = make_rng("corruption", self.family, self.task, self.rep, f"{self.condition}:semantic:{part}")
        shards = sorted({s for s in self.shards.values if s.startswith(part + "-")})
        pick = [shards[i] for i in rng.permutation(len(shards))[:k]]
        bti = base_task_index(self.family, self.template) if self.suite == "evaluation" else TEMPLATES.index(self.template)
        plan = []
        for rank, sh in enumerate(pick):
            subtype = "unit" if (bti + self.rep + rank) % 2 == 0 else "stale"
            plan.append((sh, subtype, rng))
        return plan

    # ----- build
    def build(self) -> dict:
        dirs = instance_dirs(self.paths, self.suite, self.family, self.template, self.condition, self.rep)
        for d in dirs.values():
            if d.exists():
                shutil.rmtree(d)
            d.mkdir(parents=True, exist_ok=True)
        kind, rate = CONDITION_SPEC[self.condition]
        tables = self.clean_tables()
        ref_all = reference_features(self.df, self.spec, self.template, self.suite)
        shard_reps = self.apply_representations(tables)
        stale: dict = {}  # (table, shard) -> DataFrame
        realized = {}
        if kind == "semantic":
            for part in PARTS:
                plan = self.semantic_plan(part)
                realized[part] = {"shards": len(plan), "unit": 0, "stale": 0, "denominator": N_SHARDS}
                for sh, subtype, rng in plan:
                    obs_tables = [n for n in self._cell_tables()]
                    tname = obs_tables[int(rng.integers(0, len(obs_tables)))]
                    if subtype == "unit":
                        cand = [c for c in self.layout[tname]["cols"] if c in self.spec.numeric]
                        if not cand:
                            subtype = "stale"
                        else:
                            f = cand[int(rng.integers(0, len(cand)))]
                            cur = shard_reps.get(tname, {}).get(sh, {}).get(f, "R0")
                            choices = [r for r in ["R1", "R2", "R0"] if r != cur][:2]
                            newr = choices[int(rng.integers(0, len(choices)))]
                            shard_reps.setdefault(tname, {}).setdefault(sh, {})[f] = newr
                            self.ledger.append({"partition": part, "defect": "unit_representation", "table": tname,
                                                "shard": sh, "column": f, "from": cur, "to": newr,
                                                "denominator": N_SHARDS, "reversible": True})
                            realized[part]["unit"] += 1
                    if subtype == "stale":
                        stale[(tname, sh)] = None
                        self.ledger.append({"partition": part, "defect": "stale_candidate", "table": tname,
                                            "shard": sh, "denominator": N_SHARDS, "reversible": True})
                        realized[part]["stale"] += 1
        # stored representations
        for tname, sr in shard_reps.items():
            tables[tname] = self._apply_affine(tables[tname], sr, self.reps)
        self.drift = {}
        if self.tier2:
            self.add_ingest_batch(tables)
            for part in PARTS:
                if kind == "near_dup":
                    realized[part] = self.near_duplicates(tables, rate, part)
                elif kind == "key_variant":
                    realized[part] = self.key_variants(tables, rate, part)
                elif kind == "content_drift":
                    realized[part] = self.plan_content_drift(int(rate), part)
                else:
                    realized[part] = self.undeclared_representations(tables, int(rate), part, shard_reps)
        # other snapshots and stale candidates derive from the published (pre-defect) representation
        versions: dict = {}
        for name, t in tables.items():
            grain = self.layout.get(name, {}).get("grain")
            vs = self.versions_for.get(name, self.versions if grain == "observation" else ["v2"])
            versions[name] = {v: _snapshot(t, self.spec, v, self.suite, [c for c in t.columns if c in self.spec.numeric])
                              for v in vs if v != "v2"}
        for (tname, sh) in list(stale):
            rows = tables[tname][tables[tname]["_shard"] == sh]
            stale[(tname, sh)] = _snapshot(rows, self.spec, "v1", self.suite, [c for c in rows.columns if c in self.spec.numeric])
        if kind == "cell":
            for part in PARTS:
                realized[part] = self.corrupt_cells(tables, rate, part)
        if kind == "key":
            for part in PARTS:
                realized[part] = self.corrupt_keys(tables, rate, part)
        # missing cells -> NaN in reference (unknowable originals)
        ref = ref_all.set_index("observation_id")
        for e in self.ledger:
            if e["defect"] == "cell_missing":
                ref.at[e["observation_id"], e["column"]] = np.nan
        ref = ref.reset_index()
        for name, t in tables.items():
            versions[name] = dict(sorted({**versions[name], "v2": t}.items()))
        # write files and catalogs
        catalogs = {}
        for scope, parts, root in [("visible", VISIBLE_PARTS, dirs["visible"]), ("test", ["test"], dirs["private"])]:
            catalogs[scope] = self._write_scope(root, parts, versions, stale, shard_reps)
        # negative controls
        neg = None
        if self.condition in ("N1", "N2"):
            neg = self._apply_negative(catalogs, dirs)
        # references, labels, splits
        lab = self.df[["observation_id", self.spec.target]].rename(columns={self.spec.target: "target"})
        lab = lab.merge(self.split[["observation_id", "partition", "fold"]], on="observation_id")
        lab[lab["partition"].isin(VISIBLE_PARTS)].to_parquet(dirs["trusted"] / "labels.parquet", index=False)
        lab[lab["partition"] == "test"][["observation_id", "target"]].to_parquet(dirs["private"] / "labels_test.parquet", index=False)
        for part in PARTS:
            ids = set(self.split.loc[self.split["partition"] == part, "observation_id"])
            r = ref[ref["observation_id"].isin(ids)].sort_values("observation_id", kind="mergesort").reset_index(drop=True)
            r.to_parquet(dirs["private"] / f"reference_{part}.parquet", index=False)
        with gzip.open(dirs["private"] / "ledger.json.gz", "wt") as f:
            f.write(dumps(self.ledger))
        contract = self.contract(neg)
        write_json(dirs["visible"] / "contract.json", contract)
        (dirs["visible"] / "contract.md").write_text(render_contract(contract))
        vis_manifest = {
            "partitions": {p: sorted(self.split.loc[self.split["partition"] == p, "observation_id"].tolist()) for p in VISIBLE_PARTS},
            "cv_policy": {"rule": self.spec.split_rule, "folds": 3,
                          "description": "Inner folds are fixed and applied by the trusted fit_evaluate service."},
        }
        write_json(dirs["visible"] / "visible_partition_manifest.json", vis_manifest, indent=None)
        meta = {
            "instance": f"{self.suite}/{self.task}/{self.condition}/r{self.rep}",
            "suite": self.suite, "family": self.family, "template": self.template, "condition": self.condition,
            "repetition": self.rep, "spec": self.spec.to_dict(), "required_features": required_features(self.spec, self.template),
            "realized_severity": realized, "declared_severity": rate, "negative": neg,
            "ledger_counts": pd.Series([e["defect"] for e in self.ledger]).value_counts().to_dict() if self.ledger else {},
            "primary_table": self.primary, "generator_version": GENERATOR_VERSION,
        }
        write_json(dirs["private"] / "instance_meta.json", meta)
        meta["hashes"] = {k: _dir_digest(v) for k, v in dirs.items()}
        write_json(dirs["private"] / "instance_meta.json", meta)
        return meta

    def _write_scope(self, root: Path, parts: list, versions: dict, stale: dict, shard_reps: dict) -> dict:
        src = root / "sources"
        src.mkdir(parents=True, exist_ok=True)
        cat_tables, authority = {}, []
        for name, vmap in versions.items():
            meta = self.table_meta(name)
            files = []
            auth_hashes = {}
            cols_meta = None
            for v, t in vmap.items():
                for p in parts:
                    tp = t[t["_partition"] == p].drop(columns=["_partition"]).reset_index(drop=True)
                    if tp.empty:
                        continue
                    fid = f"{name}__{v}__{p}"
                    path = src / f"{fid}.parquet"
                    tp.to_parquet(path, index=False)
                    back = pd.read_parquet(path)
                    sh = {s: df_hash(g.reset_index(drop=True)) for s, g in back.groupby("_shard", sort=True)}
                    replica = self.tier2 and v == "v2" and name in self._cell_tables()
                    if replica and self.drift.get((name, p)):
                        # the primary file's content drifts; its catalog entry keeps declaring the authorised hashes
                        self._drifted(back, name, p).to_parquet(path, index=False)
                        bad = pd.read_parquet(path)
                        for s in self.drift[(name, p)]:
                            assert df_hash(bad[bad["_shard"] == s].reset_index(drop=True)) != sh[s]
                    files.append({"file_id": fid, "table": name, "version": v, "snapshot_date": SNAPSHOTS[v],
                                  "modified": MODIFIED[v], "partition": p, "path": f"sources/{fid}.parquet",
                                  "shards": sorted(sh), "shard_hashes": sh, "rows": int(len(back))})
                    if replica:
                        # every Tier-2 instance has a replica of the authorised content, so its presence reveals nothing
                        rfid = f"{fid}__replica"
                        back.to_parquet(src / f"{rfid}.parquet", index=False)
                        rb = pd.read_parquet(src / f"{rfid}.parquet")
                        rsh = {s: df_hash(g.reset_index(drop=True)) for s, g in rb.groupby("_shard", sort=True)}
                        assert rsh == sh
                        files.append({"file_id": rfid, "table": name, "version": v, "snapshot_date": SNAPSHOTS[v],
                                      "modified": REPLICA_MODIFIED, "partition": p, "path": f"sources/{rfid}.parquet",
                                      "shards": sorted(rsh), "shard_hashes": rsh, "rows": int(len(rb))})
                    if v == "v2":
                        auth_hashes.update(sh)
                    if cols_meta is None:
                        cols_meta = _column_types(back, self.spec)
            for (tname, shard), sdf in stale.items():
                if tname != name or shard.split("-")[0] not in parts:
                    continue
                fid = f"{name}__{shard}__refresh"
                path = src / f"{fid}.parquet"
                sdf.drop(columns=["_partition"]).reset_index(drop=True).to_parquet(path, index=False)
                back = pd.read_parquet(path)
                files.append({"file_id": fid, "table": name, "version": "v1", "snapshot_date": SNAPSHOTS["v1"],
                              "modified": MODIFIED["refresh"], "partition": shard.split("-")[0],
                              "path": f"sources/{fid}.parquet", "shards": [shard],
                              "shard_hashes": {shard: df_hash(back)}, "rows": int(len(back))})
            cat_tables[name] = {**meta, "columns": cols_meta or {}, "files": files,
                                "versions": sorted({f["version"] for f in files})}
            authority.append({"record_id": f"AUTH-{name}", "registry": "data-governance", "table": name,
                              "as_of": AS_OF, "version": "v2", "shard_hashes": auth_hashes})
        relationships = []
        if self.template == "T2":
            relationships.append({"left": "block_a", "right": "block_b", "keys": ["observation_id"], "cardinality": "one_to_one"})
        if self.template == "T3":
            relationships += [{"left": "observations", "right": "measurements_b", "keys": ["observation_id"], "cardinality": "one_to_one"},
                              {"left": "observations", "right": "profiles", "keys": ["profile_key"], "cardinality": "many_to_one"}]
        if self.template == "T6":
            relationships.append({"left": "observations", "right": "readings", "keys": ["observation_id"], "cardinality": "one_to_many"})
        sr = {t: {s: m for s, m in v.items() if s.split("-")[0] in parts} for t, v in shard_reps.items()}
        catalog = {
            "instance": f"{self.suite}/{self.task}/{self.condition}/r{self.rep}",
            "partitions": parts,
            "tables": cat_tables,
            "authority": authority,
            "authority_policy": "Each authority record lists the version and per-shard SHA-256 content hashes that are authorised for the stated as-of date.",
            "representations": {f: v for f, v in self.reps.items()},
            "shard_representations": sr,
            "relationships": relationships,
            "alias_table": "key_aliases",
            "numeric_format": "Numeric cells may be stored as numbers or strings: a plain decimal literal or 'value:<decimal literal>'. Null means missing.",
        }
        if self.tier2:
            catalog["value_ranges"] = self._contract_ranges()
            catalog["packaging_columns"] = list(PACKAGING_COLUMNS)
        write_json(root / "catalog.json", catalog)
        return catalog

    def _apply_negative(self, catalogs: dict, dirs: dict) -> dict:
        name = self.primary
        for scope, root in [("visible", dirs["visible"]), ("test", dirs["private"])]:
            cat = catalogs[scope]
            t = cat["tables"][name]
            if self.condition == "N1":
                for f in [f for f in t["files"] if f["version"] == "v2"]:
                    (root / f["path"]).unlink(missing_ok=True)
                t["files"] = [f for f in t["files"] if f["version"] != "v2"]
                t["versions"] = sorted({f["version"] for f in t["files"]})
            else:
                v3 = {}
                for f in t["files"]:
                    if f["version"] == "v3":
                        v3.update(f["shard_hashes"])
                cat["authority"].append({"record_id": f"AUTH-{name}-plant", "registry": "plant-registry", "table": name,
                                         "as_of": AS_OF, "version": "v3", "shard_hashes": v3})
            write_json(root / "catalog.json", cat)
        reason = "REQUIRED_SOURCE_MISSING" if self.condition == "N1" else "AMBIGUOUS_AUTHORITY"
        return {"condition": self.condition, "table": name, "expected_reason": reason}

    def _contract_ranges(self) -> dict:
        """Plausible canonical value ranges published in the contract (training partition, padded by 25% of span)."""
        req = required_features(self.spec, self.template)
        train = self.df[self.df["observation_id"].map(self.part_of) == "train"]
        ranges = _ranges(train, [f for f in self.spec.numeric])
        if self.template == "T6":
            r = reference_features(train, self.spec, "T6", self.suite)
            ranges = {k: v for k, v in ranges.items() if k in req}
            ranges.update(_ranges(r, [c for c in req if c.endswith(("_mean", "_std"))]))
            for s in self.spec.numeric[:2]:
                ranges[f"{s}_count"] = [READINGS_PER_SERIES, READINGS_PER_SERIES]
        return {k: [round(v[0], 6), round(v[1], 6)] for k, v in ranges.items()}

    def contract(self, neg) -> dict:
        req = required_features(self.spec, self.template)
        cat = [f for f in self.spec.categorical]
        num = [f for f in req if f not in cat]
        layout = self.layout
        joins = []
        if self.template == "T2":
            joins = [{"left": "block_a", "right": "block_b", "keys": ["observation_id"], "required_cardinality": "one_to_one"}]
        if self.template == "T3":
            joins = [{"left": "observations", "right": "measurements_b", "keys": ["observation_id"], "required_cardinality": "one_to_one"},
                     {"left": "observations", "right": "profiles", "keys": ["profile_key"], "required_cardinality": "many_to_one"}]
        aggregates = []
        if self.template == "T6":
            sel = layout["readings"]["series"]
            aggregates = [{"child_table": "readings", "key": "observation_id", "series_column": "series",
                           "value_column": "value", "series": sel, "stats": ["mean", "pop_std", "count"],
                           "output_columns": {s: [f"{s}_mean", f"{s}_std", f"{s}_count"] for s in sel},
                           "readings_per_series": READINGS_PER_SERIES,
                           "note": "std is the population standard deviation (ddof=0); join aggregates to observations one-to-one."}]
        ranges = self._contract_ranges()
        sizes = self.split[self.split["partition"].isin(VISIBLE_PARTS)]["partition"].value_counts().to_dict()
        from .modeling import candidate_ids
        return {
            "task_id": self.task,
            "instance_id": f"{self.suite}/{self.task}/{self.condition}/r{self.rep}",
            "template": self.template,
            "template_description": TEMPLATE_DESCRIPTIONS[self.template],
            "objective": f"Build a leakage-free feature table for every observation and select the best permitted model "
                         f"for predicting '{self.spec.target}' ({self.spec.task_type}).",
            "task_type": self.spec.task_type,
            "target": self.spec.target,
            "target_access": "Labels are held by the trusted fit_evaluate service; they are not a source table.",
            "prediction_key": "observation_id",
            "feature_grain": "observation",
            "selection_metric": self.spec.selection_metric,
            "required_features": req,
            "numeric_features": num,
            "categorical_features": cat,
            "excluded_predictors": list(self.spec.excluded) + ["observation_id", "profile_key", "_shard"]
                                   + (["_ingest_batch"] if self.tier2 else []),
            "source_versions": {"rule": "Use, for every table and shard, the content authorised by the catalog authority "
                                        f"record for as-of date {AS_OF}. File names and modification times do not establish authority.",
                                "as_of": AS_OF},
            "keys": {"canonical_key": "observation_id", "alias_table": "key_aliases",
                     "rule": "Some records may carry a permitted alias instead of the canonical observation_id; map aliases through key_aliases."},
            "duplicates": "Exact duplicate records are packaging errors. Each observation must appear exactly once in the output.",
            "joins": joins,
            "aggregates": aggregates,
            "units": "All numeric features must be in canonical units. Shard-level stored representations are declared in "
                     "catalog.shard_representations with stored = a*canonical + b.",
            "numeric_format": "Numeric cells may be numbers, plain decimal strings, or 'value:<decimal>' strings; null means missing.",
            "value_ranges": ranges,
            "missing_numeric_policy": "Keep rows with missing values; the shared pipeline applies training-fitted median imputation (zero if a column is entirely missing).",
            "visible_partition_sizes": {k: int(v) for k, v in sizes.items()},
            "final_refit_partitions": ["train", "validation"],
            "candidate_models": candidate_ids(self.spec.task_type),
            "abstention_reasons": ["REQUIRED_SOURCE_MISSING", "AMBIGUOUS_AUTHORITY", "CONTRACT_UNSATISFIABLE", "BUDGET_INSUFFICIENT"],
            "output": {"submission": "submit(handle, candidate) freezes the transform graph and the selected candidate",
                       "abstain": "abstain(reason, detail) when the contract cannot be satisfied"},
        }


def _column_types(df: pd.DataFrame, spec: FamilySpec) -> dict:
    out = {}
    for c in df.columns:
        if c == "_shard":
            out[c] = "string (packaging shard)"
        elif c == "_ingest_batch":
            out[c] = "string (packaging batch)"
        elif pd.api.types.is_numeric_dtype(df[c]):
            out[c] = "numeric"
        else:
            out[c] = "string"
    return out


def _dir_digest(root: Path) -> str:
    """Content digest: parquet files are hashed by their decoded table content so that writer
    metadata (library version strings) does not change the digest."""
    h = hashlib.sha256()
    for p in sorted(Path(root).rglob("*")):
        if not p.is_file():
            continue
        h.update(p.relative_to(root).as_posix().encode())
        if p.suffix == ".parquet":
            h.update(df_hash(pd.read_parquet(p)).encode())
        elif p.name == "instance_meta.json":
            continue
        elif p.suffix == ".gz":
            with gzip.open(p, "rb") as f:
                h.update(hashlib.sha256(f.read()).hexdigest().encode())
        else:
            h.update(sha256_file(p).encode())
    return h.hexdigest()


def render_contract(c: dict) -> str:
    """Fixed natural-language rendering of the machine-readable contract."""
    lines = [f"# Task {c['task_id']} ({c['instance_id']})", "", c["objective"], "", c["template_description"], "",
             f"- Target: {c['target']} ({c['task_type']}); {c['target_access']}",
             f"- Selection metric: {c['selection_metric']} on the validation partition.",
             f"- Output grain: one row per observation_id. Visible partition sizes: {c['visible_partition_sizes']}.",
             f"- Required features (exact names): {', '.join(c['required_features'])}",
             f"- Categorical features: {', '.join(c['categorical_features']) or 'none'}",
             f"- Excluded predictors (never use as features): {', '.join(c['excluded_predictors'])}",
             f"- Source versions: {c['source_versions']['rule']}",
             f"- Keys: {c['keys']['rule']}",
             f"- Duplicates: {c['duplicates']}",
             f"- Units: {c['units']}",
             f"- Numeric format: {c['numeric_format']}",
             f"- Missing values: {c['missing_numeric_policy']}"]
    for j in c["joins"]:
        lines.append(f"- Join {j['left']} to {j['right']} on {j['keys']} ({j['required_cardinality']}).")
    for a in c["aggregates"]:
        lines.append(f"- Aggregate {a['child_table']}.{a['value_column']} by {a['key']} and {a['series_column']} for series "
                     f"{a['series']} with stats {a['stats']} -> columns {a['output_columns']}. {a['note']}")
    lines.append(f"- Candidate models: {', '.join(c['candidate_models'])} (at most six evaluations).")
    lines.append(f"- If the contract cannot be satisfied, abstain with one of: {', '.join(c['abstention_reasons'])}.")
    return "\n".join(lines) + "\n"


# --------------------------------------------------------------------------- suite helpers
def build_instance(paths: Paths, suite: str, family: str, template: str, condition: str, rep: int) -> dict:
    return InstanceBuilder(paths, suite, family, template, condition, rep).build()


def instance_record_path(paths: Paths) -> Path:
    return paths.manifests / "instances.json"


def ensure_instance(paths: Paths, suite: str, family: str, template: str, condition: str, rep: int) -> dict:
    """Build an instance if absent in this session's cache (no manifest side effects)."""
    dirs = instance_dirs(paths, suite, family, template, condition, rep)
    meta_path = dirs["private"] / "instance_meta.json"
    if meta_path.exists() and (dirs["visible"] / "catalog.json").exists():
        meta = read_json(meta_path)
        if "hashes" in meta and meta.get("generator_version", 1) == GENERATOR_VERSION:
            return meta
    return build_instance(paths, suite, family, template, condition, rep)


def ensure_instances(paths: Paths, keys: list, workers: int = 1, log=print) -> dict:
    """Build (in parallel) and verify a list of (suite, family, template, condition, rep) instances.
    The first build records the content digest; every later build must reproduce it exactly."""
    from concurrent.futures import ProcessPoolExecutor
    keys = sorted(set(tuple(k) for k in keys))
    metas = {}
    if workers > 1 and len(keys) > 1:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            futs = {k: ex.submit(ensure_instance, paths, *k) for k in keys}
            for k, f in futs.items():
                metas[k] = f.result()
    else:
        for k in keys:
            metas[k] = ensure_instance(paths, *k)
    rec_path = instance_record_path(paths)
    recs = read_json(rec_path) if rec_path.exists() else {}
    changed = False
    for k, meta in metas.items():
        key = f"{k[0]}/{k[1]}-{k[2]}/{k[3]}/r{k[4]}"
        if key in recs:
            if recs[key] != meta["hashes"]:
                raise RuntimeError(f"instance {key} does not reproduce its recorded content digest; "
                                   "the generator, raw data or library versions changed")
        else:
            recs[key] = meta["hashes"]
            changed = True
    if changed:
        write_json(rec_path, recs)
    return metas


### Declarative transform graph engine

In [ ]:
%%writefile adm_code/adm/engine.py
"""Declarative transform graph: validation, source resolution and execution on any partition.

A graph never names files. ``load`` names a logical table and a version policy, so the same
frozen graph can be replayed by the evaluator on hidden test shards."""
from __future__ import annotations

import copy
import re
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from .core import df_hash, read_json

POLICIES = ["authoritative", "authoritative_verified", "latest_modified", "latest_snapshot"]  # plus "version:<v>"
STAT_LABELS = {"mean": "mean", "pop_std": "std", "sample_std": "std", "count": "count", "min": "min",
               "max": "max", "median": "median"}
OP_ARGS = {
    "load": {"required": ["table", "policy"], "optional": []},
    "parse_numeric": {"required": ["input"], "optional": ["columns"]},
    "to_canonical": {"required": ["input"], "optional": []},
    "map_alias": {"required": ["input"], "optional": ["key"]},
    "normalize_key": {"required": ["input"], "optional": ["key"]},
    "infer_representation": {"required": ["input"], "optional": []},
    "dedupe": {"required": ["input"], "optional": ["mode", "key"]},
    "join": {"required": ["left", "right", "on"], "optional": ["how", "cardinality"]},
    "aggregate": {"required": ["input", "key", "series_column", "value_column", "stats"], "optional": []},
    "drop_missing_rows": {"required": ["input"], "optional": ["columns"]},
    "project": {"required": ["input", "columns"], "optional": []},
}
MAX_NODES = 40
PACKAGING_COLUMNS = ("_shard", "_ingest_batch")
KEY_PATTERN = re.compile(r"([A-Z0-9]+)-(\d+)")
KEY_DIGITS = 6


def normalize_key_value(v):
    """Canonical observation_id format: upper-case <PREFIX>-<six digits>. Values that do not match the pattern after
    trimming and upper-casing (e.g. permitted aliases) are returned unchanged."""
    if v is None:
        return v
    m = KEY_PATTERN.fullmatch(str(v).strip().upper())
    return f"{m.group(1)}-{int(m.group(2)):0{KEY_DIGITS}d}" if m else v


class GraphError(ValueError):
    pass


class ResolutionError(GraphError):
    def __init__(self, msg: str, reason: str):
        super().__init__(msg)
        self.reason = reason


@dataclass
class CatalogView:
    """A catalog plus its source directory, restricted to one partition."""
    root: Path
    catalog: dict
    partition: str
    _cache: dict = field(default_factory=dict)

    @classmethod
    def open(cls, root, partition: str) -> "CatalogView":
        root = Path(root)
        return cls(root, read_json(root / "catalog.json"), partition)

    def read(self, f: dict) -> pd.DataFrame:
        key = f["file_id"]
        if key not in self._cache:
            self._cache[key] = pd.read_parquet(self.root / f["path"])
        return self._cache[key].copy()


# --------------------------------------------------------------------------- validation
def validate_graph(graph) -> dict:
    if not isinstance(graph, dict):
        raise GraphError("graph must be an object with 'nodes' and 'output'")
    nodes = graph.get("nodes")
    if not isinstance(nodes, list) or not nodes:
        raise GraphError("graph.nodes must be a non-empty list")
    if len(nodes) > MAX_NODES:
        raise GraphError(f"graph has {len(nodes)} nodes; the limit is {MAX_NODES}")
    seen = set()
    for n in nodes:
        if not isinstance(n, dict):
            raise GraphError("every node must be an object")
        nid, op = n.get("id"), n.get("op")
        if not isinstance(nid, str) or not nid:
            raise GraphError("every node needs a string 'id'")
        if nid in seen:
            raise GraphError(f"duplicate node id {nid!r}")
        if op not in OP_ARGS:
            raise GraphError(f"node {nid}: unknown op {op!r}; allowed ops: {sorted(OP_ARGS)}")
        spec = OP_ARGS[op]
        for a in spec["required"]:
            if a not in n:
                raise GraphError(f"node {nid} ({op}) is missing argument {a!r}")
        extra = set(n) - set(spec["required"]) - set(spec["optional"]) - {"id", "op"}
        if extra:
            raise GraphError(f"node {nid} ({op}) has unknown arguments {sorted(extra)}")
        for ref in ("input", "left", "right"):
            if ref in n and n[ref] not in seen:
                raise GraphError(f"node {nid}: {ref} {n[ref]!r} must refer to an earlier node id")
        if op == "load":
            p = n["policy"]
            if not (p in POLICIES or (isinstance(p, str) and p.startswith("version:"))):
                raise GraphError(f"node {nid}: policy must be one of {POLICIES} or 'version:<v>'")
        if op == "dedupe" and n.get("mode", "exact") not in ("exact", "key"):
            raise GraphError(f"node {nid}: dedupe mode must be 'exact' or 'key'")
        if op == "join":
            if n.get("how", "inner") not in ("inner", "left"):
                raise GraphError(f"node {nid}: join how must be 'inner' or 'left'")
            if n.get("cardinality") not in (None, "one_to_one", "many_to_one", "one_to_many"):
                raise GraphError(f"node {nid}: cardinality must be one_to_one, many_to_one or one_to_many")
        if op == "aggregate":
            st = n["stats"]
            if not isinstance(st, list) or not st or any(s not in STAT_LABELS for s in st):
                raise GraphError(f"node {nid}: stats must be a non-empty list from {sorted(STAT_LABELS)}")
            labels = [STAT_LABELS[s] for s in st]
            if len(set(labels)) != len(labels):
                raise GraphError(f"node {nid}: stats {st} produce duplicate output labels")
        if op == "project" and (not isinstance(n["columns"], list) or not n["columns"]):
            raise GraphError(f"node {nid}: project columns must be a non-empty list")
        seen.add(nid)
    if graph.get("output") not in seen:
        raise GraphError("graph.output must name a node id")
    return graph


# --------------------------------------------------------------------------- resolution
def _candidates(view: CatalogView, table: str) -> dict:
    t = view.catalog["tables"].get(table)
    if t is None:
        raise GraphError(f"unknown table {table!r}; available: {sorted(view.catalog['tables'])}")
    out: dict = {}
    for f in t["files"]:
        if f["partition"] != view.partition:
            continue
        for s in f["shards"]:
            out.setdefault(s, []).append(f)
    return out


def authority_records(catalog: dict, table: str) -> list:
    return [a for a in catalog.get("authority", []) if a["table"] == table]


def shard_content_hash(view: CatalogView, f: dict, shard: str) -> str:
    d = view.read(f)
    return df_hash(d[d["_shard"] == shard].reset_index(drop=True))


def resolve(view: CatalogView, table: str, policy: str) -> tuple[pd.DataFrame, dict]:
    # tools such as profile_data pass the agent's policy straight through; an invalid one must be a normal tool error
    if not isinstance(policy, str) or not (policy in POLICIES or (policy.startswith("version:") and len(policy) > 8)):
        raise GraphError(f"policy {policy!r} is not valid; use one of {POLICIES} or 'version:<v>'")
    cands = _candidates(view, table)
    chosen: dict = {}
    if policy in ("authoritative", "authoritative_verified"):
        recs = authority_records(view.catalog, table)
        if not recs:
            raise ResolutionError(f"no authority record for table {table!r}", "REQUIRED_SOURCE_MISSING")
        distinct = {(r["version"], tuple(sorted(r["shard_hashes"].items()))) for r in recs}
        if len(distinct) > 1:
            raise ResolutionError(
                f"conflicting authority records for table {table!r}: " +
                "; ".join(f"{r['record_id']} ({r['registry']}) authorises version {r['version']}" for r in recs) +
                ". The catalog defines no precedence rule.", "AMBIGUOUS_AUTHORITY")
        rec = recs[0]
        wanted = {s: h for s, h in rec["shard_hashes"].items() if s.startswith(view.partition + "-")}
        if not wanted:
            raise ResolutionError(f"authority record for {table!r} lists no shards for partition {view.partition}",
                                  "REQUIRED_SOURCE_MISSING")
        for s, h in sorted(wanted.items()):
            match = [f for f in cands.get(s, []) if f["version"] == rec["version"] and f["shard_hashes"].get(s) == h]
            if policy == "authoritative_verified":
                # do not trust declared file metadata: keep only candidates whose actual content has the authorised hash
                match = [f for f in match if shard_content_hash(view, f, s) == h]
            if not match:
                raise ResolutionError(
                    f"table {table!r}: authorised version {rec['version']} for shard {s} is not available "
                    f"(available versions: {sorted({f['version'] for f in cands.get(s, [])})})", "REQUIRED_SOURCE_MISSING")
            chosen[s] = match[0]
    else:
        for s, fl in sorted(cands.items()):
            if policy == "latest_modified":
                chosen[s] = sorted(fl, key=lambda f: (f["modified"], f["file_id"]))[-1]
            elif policy == "latest_snapshot":
                chosen[s] = sorted(fl, key=lambda f: (f["snapshot_date"], f["modified"], f["file_id"]))[-1]
            else:
                v = policy.split(":", 1)[1]
                m = [f for f in fl if f["version"] == v]
                if not m:
                    raise ResolutionError(f"table {table!r}: shard {s} has no candidate with version {v}", "REQUIRED_SOURCE_MISSING")
                if len(m) > 1:
                    raise GraphError(f"table {table!r}: shard {s} has {len(m)} candidates with version {v}; "
                                     "use a policy that selects by authority")
                chosen[s] = m[0]
        if not chosen:
            raise GraphError(f"table {table!r} has no files for partition {view.partition}")
    frames, record = [], {}
    for s, f in sorted(chosen.items()):
        d = view.read(f)
        d = d[d["_shard"] == s]
        frames.append(d)
        record[s] = {"file_id": f["file_id"], "version": f["version"], "declared_sha256": f["shard_hashes"].get(s),
                     "content_sha256": df_hash(d.reset_index(drop=True))}
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    rejected = sorted({f["file_id"] for fl in cands.values() for f in fl} - {r["file_id"] for r in record.values()})
    return df, {"table": table, "policy": policy, "shards": record, "rejected_files": rejected}


# --------------------------------------------------------------------------- execution helpers
def parse_cell(v):
    if v is None:
        return np.nan, False
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v), False
    s = str(v).strip()
    if s == "" or s.lower() in ("nan", "none", "null", "<na>"):
        return np.nan, False
    enc = False
    if s.startswith("value:"):
        s, enc = s[6:].strip(), True
    return float(s), enc


def _shard_tables(lineage: dict, cols: list) -> set:
    out = set()
    for c in cols:
        out |= lineage.get(c, set())
    return out


def execute(graph: dict, view: CatalogView) -> tuple[pd.DataFrame, dict]:
    """Execute a validated graph on one partition. Returns the output frame and an execution trace."""
    validate_graph(graph)
    frames: dict = {}
    lineage: dict = {}   # node -> {column: set(tables)}
    single: dict = {}    # node -> table name when lineage is a single loaded table without joins
    trace = {"partition": view.partition, "nodes": [], "loads": [], "operations": []}
    for n in graph["nodes"]:
        op, nid = n["op"], n["id"]
        info: dict = {"id": nid, "op": op}
        try:
            if op == "load":
                df, rec = resolve(view, n["table"], n["policy"])
                trace["loads"].append({"node": nid, **rec})
                lineage[nid] = {c: {n["table"]} for c in df.columns}
                single[nid] = n["table"]
            else:
                src = n.get("input") or n.get("left")
                df = frames[src].copy()
                lin = copy.deepcopy(lineage[src])
                single[nid] = single.get(src) if op not in ("join", "aggregate") else None
                if op == "parse_numeric":
                    cols = n.get("columns", "all")
                    if cols in ("all", "all_numeric", None):
                        tab = single.get(src)
                        cat_types = view.catalog["tables"].get(tab, {}).get("columns", {}) if tab else {}
                        reps = view.catalog.get("representations", {})
                        cols = [c for c in df.columns if c in reps or cat_types.get(c) == "numeric"]
                    missing = [c for c in cols if c not in df.columns]
                    if missing:
                        raise GraphError(f"parse_numeric: columns not found {missing}")
                    parsed_locs = []
                    for c in cols:
                        if pd.api.types.is_numeric_dtype(df[c]):
                            df[c] = df[c].astype(float)
                            continue
                        vals, bad = [], []
                        for oid, v in zip(df["observation_id"] if "observation_id" in df else range(len(df)), df[c]):
                            try:
                                x, enc = parse_cell(v)
                            except (ValueError, TypeError):
                                bad.append(v)
                                x, enc = np.nan, False
                            if enc:
                                parsed_locs.append([str(oid), c])
                            vals.append(x)
                        if bad:
                            raise GraphError(f"parse_numeric: column {c!r} has unparseable values, e.g. {bad[:3]}")
                        df[c] = np.array(vals, dtype=float)
                    info["parsed_encoded"] = len(parsed_locs)
                    trace["operations"].append({"node": nid, "op": op, "locations": parsed_locs})
                elif op == "to_canonical":
                    tab = single.get(src)
                    if not tab:
                        raise GraphError("to_canonical must be applied to a single loaded table before any join or aggregate")
                    sr = view.catalog.get("shard_representations", {}).get(tab, {})
                    reps = view.catalog.get("representations", {})
                    converted = []
                    for shard, fmap in sr.items():
                        if not shard.startswith(view.partition + "-"):
                            continue
                        m = (df["_shard"] == shard).to_numpy()
                        if not m.any():
                            continue
                        for f, r in fmap.items():
                            if f not in df.columns:
                                continue
                            if not pd.api.types.is_numeric_dtype(df[f]):
                                raise GraphError(f"to_canonical: column {f!r} is not numeric; apply parse_numeric first")
                            o = reps[f]["options"][r]
                            df.loc[m, f] = (df.loc[m, f].astype(float) - o["b"]) / o["a"]
                            converted.append([shard, f, r])
                    info["converted"] = len(converted)
                    trace["operations"].append({"node": nid, "op": op, "table": tab, "locations": converted})
                elif op == "map_alias":
                    key = n.get("key", "observation_id")
                    if key not in df.columns:
                        raise GraphError(f"map_alias: key column {key!r} not found")
                    al, rec = resolve(view, view.catalog.get("alias_table", "key_aliases"), "authoritative")
                    trace["loads"].append({"node": nid + ":aliases", **rec})
                    amap = dict(zip(al["alias"], al["observation_id"]))
                    before = df[key].astype(str)
                    after = before.map(lambda x: amap.get(x, x))
                    changed = (before != after).to_numpy()
                    df[key] = after.to_numpy()
                    info["remapped"] = int(changed.sum())
                    trace["operations"].append({"node": nid, "op": op,
                                                "locations": sorted(set(after[changed].tolist()))})
                elif op == "normalize_key":
                    key = n.get("key", "observation_id")
                    if key not in df.columns:
                        raise GraphError(f"normalize_key: key column {key!r} not found")
                    before = df[key]
                    after = before.map(normalize_key_value)
                    changed = (before.astype(str) != after.astype(str)).to_numpy()
                    df[key] = after.to_numpy()
                    info["normalized"] = int(changed.sum())
                    trace["operations"].append({"node": nid, "op": op,
                                                "locations": sorted(set(after[changed].astype(str).tolist()))})
                elif op == "infer_representation":
                    tab = single.get(src)
                    if not tab:
                        raise GraphError("infer_representation must be applied to a single loaded table before any join "
                                         "or aggregate")
                    ranges = view.catalog.get("value_ranges") or {}
                    reps = view.catalog.get("representations", {})
                    declared = view.catalog.get("shard_representations", {}).get(tab, {})
                    converted = []
                    if "_shard" in df.columns:
                        for shard in sorted(df["_shard"].astype(str).unique()):
                            m = (df["_shard"] == shard).to_numpy()
                            for f in [c for c in df.columns if c in ranges and c in reps]:
                                if f in declared.get(shard, {}) or not pd.api.types.is_numeric_dtype(df[f]):
                                    continue
                                lo, hi = ranges[f]
                                v = df.loc[m, f].to_numpy(dtype=float)
                                v = v[~np.isnan(v)]
                                if not len(v):
                                    continue
                                inside = lambda x: float(((x >= lo - 1e-9) & (x <= hi + 1e-9)).mean())
                                if inside(v) >= 0.5:
                                    continue
                                best, best_f = None, 0.0
                                for r, o in reps[f]["options"].items():
                                    if r == "R0":
                                        continue
                                    fr = inside((v - o["b"]) / o["a"])
                                    if fr > best_f:
                                        best, best_f = r, fr
                                if best is not None and best_f >= 0.95:
                                    o = reps[f]["options"][best]
                                    df.loc[m, f] = (df.loc[m, f].astype(float) - o["b"]) / o["a"]
                                    converted.append([shard, f, best])
                    info["converted"] = len(converted)
                    if not ranges:
                        info["note"] = "catalog declares no value_ranges; nothing to infer"
                    trace["operations"].append({"node": nid, "op": op, "table": tab, "locations": converted})
                elif op == "dedupe":
                    mode = n.get("mode", "exact")
                    before = len(df)
                    if mode == "exact":
                        dmask = df.astype(str).duplicated(keep="first").to_numpy()
                    else:
                        key = n.get("key", "observation_id")
                        keys = [key] if isinstance(key, str) else list(key)
                        dmask = df.duplicated(subset=keys, keep="first").to_numpy()
                    removed = df.loc[dmask, "observation_id"].astype(str).tolist() if "observation_id" in df else []
                    df = df.loc[~dmask].reset_index(drop=True)
                    info["removed"] = before - len(df)
                    trace["operations"].append({"node": nid, "op": op, "mode": mode, "locations": sorted(set(removed))})
                elif op == "join":
                    right = frames[n["right"]].copy()
                    rlin = lineage[n["right"]]
                    on = n["on"] if isinstance(n["on"], list) else [n["on"]]
                    for k in on:
                        if k not in df.columns or k not in right.columns:
                            raise GraphError(f"join: key {k!r} missing on one side")
                    pack = [c for c in PACKAGING_COLUMNS if c in right.columns and c not in on]
                    if pack:
                        right = right.drop(columns=pack)
                    overlap = (set(df.columns) & set(right.columns)) - set(on)
                    if overlap:
                        raise GraphError(f"join: non-key columns present on both sides {sorted(overlap)}; project first")
                    for k in on:
                        if df[k].dtype != right[k].dtype:
                            df[k] = df[k].astype(str)
                            right[k] = right[k].astype(str)
                    validate = {"one_to_one": "1:1", "many_to_one": "m:1", "one_to_many": "1:m"}.get(n.get("cardinality"))
                    try:
                        df = df.merge(right, on=on, how=n.get("how", "inner"), validate=validate)
                    except pd.errors.MergeError as e:
                        raise GraphError(f"join: declared cardinality {n.get('cardinality')} violated: {e}")
                    for c in right.columns:
                        lin[c] = lin.get(c, set()) | rlin.get(c, set())
                    info["rows"] = len(df)
                elif op == "aggregate":
                    key, sc, vc = n["key"], n["series_column"], n["value_column"]
                    for c in (key, sc, vc):
                        if c not in df.columns:
                            raise GraphError(f"aggregate: column {c!r} not found")
                    if not pd.api.types.is_numeric_dtype(df[vc]):
                        raise GraphError(f"aggregate: value column {vc!r} is not numeric")
                    g = df.groupby([key, sc])[vc]
                    parts = []
                    for s in n["stats"]:
                        if s == "mean":
                            v = g.mean()
                        elif s == "pop_std":
                            v = g.std(ddof=0)
                        elif s == "sample_std":
                            v = g.std(ddof=1)
                        elif s == "count":
                            v = g.count().astype(float)
                        elif s == "min":
                            v = g.min()
                        elif s == "max":
                            v = g.max()
                        else:
                            v = g.median()
                        w = v.unstack(sc)
                        w.columns = [f"{c}_{STAT_LABELS[s]}" for c in w.columns]
                        parts.append(w)
                    df = pd.concat(parts, axis=1).reset_index()
                    src_tabs = lin.get(vc, set())
                    lin = {c: set(src_tabs) for c in df.columns}
                    info["rows"] = len(df)
                elif op == "drop_missing_rows":
                    cols = n.get("columns")
                    before = len(df)
                    df = df.dropna(subset=cols).reset_index(drop=True) if cols else df.dropna().reset_index(drop=True)
                    info["removed"] = before - len(df)
                elif op == "project":
                    missing = [c for c in n["columns"] if c not in df.columns]
                    if missing:
                        raise GraphError(f"project: columns not found {missing}; available {list(df.columns)[:40]}")
                    df = df[list(dict.fromkeys(n["columns"]))].copy()
                lineage[nid] = {c: lin.get(c, set()) for c in df.columns}
        except ResolutionError as e:
            raise ResolutionError(f"node {nid} ({op}): {e}", e.reason) from None
        except GraphError as e:
            raise GraphError(f"node {nid} ({op}): {e}") from None
        frames[nid] = df
        info["rows_out"] = int(len(df))
        trace["nodes"].append(info)
    out = frames[graph["output"]]
    if "observation_id" not in out.columns:
        raise GraphError("graph output must contain the observation_id column")
    trace["output_columns"] = list(out.columns)
    trace["column_lineage"] = {c: sorted(v) for c, v in lineage[graph["output"]].items()}
    trace["output_hash"] = df_hash(out)
    return out.reset_index(drop=True), trace


def summarize(df: pd.DataFrame, expected_ids: int | None = None, sample_rows: int = 3) -> dict:
    """Bounded summary of a materialised frame returned to agents (no verdicts)."""
    out = {
        "rows": int(len(df)),
        "distinct_observation_ids": int(df["observation_id"].nunique()) if "observation_id" in df else None,
        "columns": {c: str(df[c].dtype) for c in df.columns},
        "null_counts": {c: int(v) for c, v in df.isna().sum().items() if v},
    }
    if expected_ids is not None:
        out["visible_partition_size"] = expected_ids
    if sample_rows:
        out["sample"] = df.head(sample_rows).astype(object).where(df.head(sample_rows).notna(), None).to_dict("records")
    return out


### Fixed contract-driven workflow F

In [ ]:
%%writefile adm_code/adm/fixed.py
"""Fixed contract-driven workflow F (Section 11.1): no LLM, no task-ID branches."""
from __future__ import annotations


def plan_tables(contract: dict, catalog: dict) -> dict:
    """Pick the tables the contract requires, using only contract and catalog metadata."""
    req = set(contract["required_features"])
    joins = contract.get("joins", [])
    aggs = contract.get("aggregates", [])
    tables = []
    for j in joins:
        for t in (j["left"], j["right"]):
            if t not in tables:
                tables.append(t)
    child = [a["child_table"] for a in aggs]
    if not tables:
        best, best_cov = None, -1
        for name, t in sorted(catalog["tables"].items()):
            if t.get("grain") != "observation":
                continue
            cov = len(req & set(t.get("columns", {})))
            if cov > best_cov:
                best, best_cov = name, cov
        tables = [best]
    return {"tables": tables, "child": child}


def contract_graph(contract: dict, catalog: dict, extended: bool = False) -> dict:
    """F's graph. ``extended=True`` builds the oracle reference solver instead (solvability check, ABC II.7-II.8):
    it adds the general repairs for Tier-2 defects outside the declared specification. F itself never uses it."""
    plan = plan_tables(contract, catalog)
    nodes = []
    prepared = {}

    def prep(t: str) -> str:
        cols = catalog["tables"][t].get("columns", {})
        base = f"{t}_load"
        nodes.append({"id": base, "op": "load", "table": t,
                      "policy": "authoritative_verified" if extended else "authoritative"})
        cur = base
        nodes.append({"id": f"{t}_parse", "op": "parse_numeric", "input": cur, "columns": "all"})
        cur = f"{t}_parse"
        nodes.append({"id": f"{t}_canon", "op": "to_canonical", "input": cur})
        cur = f"{t}_canon"
        if extended:
            nodes.append({"id": f"{t}_infer", "op": "infer_representation", "input": cur})
            cur = f"{t}_infer"
        if "observation_id" in cols:
            nodes.append({"id": f"{t}_alias", "op": "map_alias", "input": cur, "key": "observation_id"})
            cur = f"{t}_alias"
            if extended:
                nodes.append({"id": f"{t}_normkey", "op": "normalize_key", "input": cur, "key": "observation_id"})
                cur = f"{t}_normkey"
        nodes.append({"id": f"{t}_dedupe", "op": "dedupe", "input": cur, "mode": "exact"})
        cur = f"{t}_dedupe"
        if extended and catalog["tables"][t].get("grain") == "observation":
            nodes.append({"id": f"{t}_dedupe_key", "op": "dedupe", "input": cur, "mode": "key", "key": "observation_id"})
            cur = f"{t}_dedupe_key"
        return cur

    for t in plan["tables"]:
        prepared[t] = prep(t)
    for t in plan["child"]:
        prepared[t] = prep(t)
    if contract.get("joins"):
        cur = prepared[contract["joins"][0]["left"]]
        for k, j in enumerate(contract["joins"]):
            nid = f"join_{k}"
            nodes.append({"id": nid, "op": "join", "left": cur, "right": prepared[j["right"]], "on": j["keys"],
                          "how": "left", "cardinality": j["required_cardinality"]})
            cur = nid
    else:
        cur = prepared[plan["tables"][0]]
    for k, a in enumerate(contract.get("aggregates", [])):
        aid = f"agg_{k}"
        nodes.append({"id": aid, "op": "aggregate", "input": prepared[a["child_table"]], "key": a["key"],
                      "series_column": a["series_column"], "value_column": a["value_column"], "stats": a["stats"]})
        jid = f"agg_join_{k}"
        nodes.append({"id": jid, "op": "join", "left": cur, "right": aid, "on": [a["key"]], "how": "left",
                      "cardinality": "one_to_one"})
        cur = jid
    nodes.append({"id": "features", "op": "project", "input": cur,
                  "columns": ["observation_id"] + list(contract["required_features"])})
    return {"nodes": nodes, "output": "features"}


def oracle_graph(contract: dict, catalog: dict) -> dict:
    return contract_graph(contract, catalog, extended=True)


### Internal quality and provenance gates (Section 9.6)

In [ ]:
%%writefile adm_code/adm/gates.py
"""Internal treatment gates (Section 9.6). They use only agent-permitted information:
the contract, the visible catalog, the visible partition manifest and the materialised artifact."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .engine import authority_records


def _check(name: str, ok: bool, detail: str = "") -> dict:
    return {"check": name, "passed": bool(ok), "detail": detail}


def quality_gate(handle: dict, contract: dict, catalog: dict, manifest: dict) -> dict:
    checks = []
    req = list(contract["required_features"])
    excluded = set(contract["excluded_predictors"]) - {"observation_id"}
    ranges = contract.get("value_ranges", {})
    num = set(contract.get("numeric_features", []))
    graph = handle["graph"]
    for part, df in handle["frames"].items():
        cols = [c for c in df.columns if c != "observation_id"]
        missing = [c for c in req if c not in df.columns]
        extra = [c for c in cols if c not in req]
        checks.append(_check(f"{part}:required_columns", not missing and not extra,
                             f"missing={missing[:8]} extra={extra[:8]}" if (missing or extra) else ""))
        forb = [c for c in cols if c in excluded]
        checks.append(_check(f"{part}:excluded_predictors", not forb, f"present={forb}" if forb else ""))
        bad_types = [c for c in cols if c in num and not pd.api.types.is_numeric_dtype(df[c])]
        checks.append(_check(f"{part}:numeric_types", not bad_types, f"non-numeric={bad_types}" if bad_types else ""))
        ids = df["observation_id"].astype(str)
        dup = int(ids.duplicated().sum())
        checks.append(_check(f"{part}:key_uniqueness", dup == 0, f"{dup} duplicated observation_id rows" if dup else ""))
        expected = set(manifest["partitions"][part])
        got = set(ids)
        miss, unexpected = len(expected - got), len(got - expected)
        checks.append(_check(f"{part}:record_coverage", miss == 0 and unexpected == 0,
                             f"{miss} expected observation_ids missing, {unexpected} unexpected ids "
                             f"(examples: {sorted(got - expected)[:3]})" if (miss or unexpected) else ""))
        viol = {}
        for c, (lo, hi) in ranges.items():
            if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                v = df[c].to_numpy(dtype=float)
                v = v[~np.isnan(v)]
                n = int(((v < lo - 1e-9) | (v > hi + 1e-9)).sum())
                if n:
                    viol[c] = n
        checks.append(_check(f"{part}:value_ranges", not viol, f"out-of-range value counts {dict(list(viol.items())[:6])}" if viol else ""))
        for a in contract.get("aggregates", []):
            for s in a["series"]:
                cc = f"{s}_count"
                sd = f"{s}_std"
                if cc in df.columns and pd.api.types.is_numeric_dtype(df[cc]):
                    bad = int((df[cc].dropna() != a["readings_per_series"]).sum())
                    checks.append(_check(f"{part}:aggregate_count:{s}", bad == 0,
                                         f"{bad} rows with count != {a['readings_per_series']}" if bad else ""))
                if sd in df.columns and pd.api.types.is_numeric_dtype(df[sd]):
                    neg = int((df[sd].dropna() < 0).sum())
                    checks.append(_check(f"{part}:aggregate_std:{s}", neg == 0, f"{neg} negative std" if neg else ""))
    # canonical units: every loaded table with declared non-canonical shards must pass through to_canonical
    trace = handle["traces"]["train"]
    loaded = {l["table"] for l in trace["loads"]}
    converted = {o.get("table") for o in trace["operations"] if o["op"] == "to_canonical"}
    need = sorted(t for t in loaded if catalog.get("shard_representations", {}).get(t))
    lacking = [t for t in need if t not in converted]
    checks.append(_check("canonical_units", not lacking,
                         f"tables with declared shard-level representations not converted: {lacking}" if lacking else ""))
    drops = [n["id"] for n in graph["nodes"] if n["op"] == "drop_missing_rows"]
    checks.append(_check("missing_value_policy", not drops, f"rows dropped by {drops}; the contract requires keeping rows" if drops else ""))
    return {"gate": "quality", "artifact": handle["id"], "passed": all(c["passed"] for c in checks),
            "failures": [c for c in checks if not c["passed"]], "n_checks": len(checks)}


def provenance_gate(handle: dict, contract: dict, catalog: dict, provenance: list, stage: str = "materialize",
                    candidate: str | None = None) -> dict:
    checks = []
    for part, trace in handle["traces"].items():
        for load in trace["loads"]:
            recs = authority_records(catalog, load["table"])
            if not recs:
                checks.append(_check(f"{part}:authority:{load['table']}", False, "no authority record"))
                continue
            distinct = {(r["version"], tuple(sorted(r["shard_hashes"].items()))) for r in recs}
            if len(distinct) > 1:
                checks.append(_check(f"{part}:authority:{load['table']}", False,
                                     f"AMBIGUOUS_AUTHORITY: {len(recs)} conflicting records, no precedence rule"))
                continue
            rec = recs[0]
            wrong = [s for s, v in load["shards"].items()
                     if v["version"] != rec["version"] or v["declared_sha256"] != rec["shard_hashes"].get(s)]
            wanted = {s for s in rec["shard_hashes"] if s.startswith(part + "-")}
            absent = sorted(wanted - set(load["shards"]))
            ok = not wrong and not absent
            checks.append(_check(f"{part}:authority:{load['table']}", ok,
                                 (f"{len(wrong)} shards not matching authorised version {rec['version']} "
                                  f"(examples {wrong[:3]}); {len(absent)} authorised shards absent") if not ok else ""))
            drift = [s for s, v in load["shards"].items()
                     if v.get("content_sha256") and v["content_sha256"] != rec["shard_hashes"].get(s)]
            checks.append(_check(f"{part}:content_hash:{load['table']}", not drift,
                                 (f"{len(drift)} loaded shards have content whose SHA-256 differs from the authority "
                                  f"record although the file metadata declares the authorised hash (examples {drift[:3]})")
                                 if drift else ""))
        lin = trace.get("column_lineage", {})
        orphan = [c for c in contract["required_features"] if c in lin and not lin[c]]
        checks.append(_check(f"{part}:lineage_complete", not orphan, f"columns without source lineage {orphan}" if orphan else ""))
    fits = [p for p in provenance if p.get("activity") == "fit_evaluate" and p.get("handle") == handle["id"]]
    bad_scope = [f for f in fits if f.get("fit_partitions") != ["train"]]
    checks.append(_check("fit_scope", not bad_scope, f"{len(bad_scope)} fits used non-training partitions" if bad_scope else ""))
    if stage == "submit":
        prod = [p for p in provenance if p.get("activity") == "materialize_features" and p.get("handle") == handle["id"]]
        consistent = bool(prod) and prod[-1].get("graph_hash") == handle["graph_hash"]
        checks.append(_check("artifact_consistency", consistent, "" if consistent else "submitted handle not produced in this run"))
        if candidate is not None:
            ev = any(f.get("candidate") == candidate and f.get("status") == "ok" for f in fits)
            checks.append(_check("candidate_evaluated", ev, "" if ev else f"candidate {candidate} was not evaluated on {handle['id']}"))
    return {"gate": "provenance", "artifact": handle["id"], "stage": stage, "passed": all(c["passed"] for c in checks),
            "failures": [c for c in checks if not c["passed"]], "n_checks": len(checks)}


def fit_quality_gate(result: dict, handle_id: str) -> dict:
    rows = result.get("rows", {})
    tr = rows.get("train", {})
    cov = rows.get("validation_coverage", 1.0)
    checks = [_check("label_match:train", tr.get("unmatched_rows", 0) == 0,
                     f"{tr.get('unmatched_rows')} training rows have no canonical label match" if tr.get("unmatched_rows") else ""),
              _check("validation_coverage", abs(cov - 1.0) < 1e-12, f"validation coverage {cov:.4f}" if abs(cov - 1.0) >= 1e-12 else "")]
    return {"gate": "quality", "artifact": f"{handle_id}/{result.get('candidate')}", "stage": "fit",
            "passed": all(c["passed"] for c in checks), "failures": [c for c in checks if not c["passed"]], "n_checks": len(checks)}


### Instrumented tools, budgets and trusted provenance (Section 13)

In [ ]:
%%writefile adm_code/adm/tools.py
"""Instrumented tool layer shared by every system (Section 13). Trusted wrappers, not LLM
statements, record sources, dependencies, row counts, parameters and hashes."""
from __future__ import annotations

import json
import pickle
import threading
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from . import gates, modeling
from .core import (ABSTENTION_REASONS, append_jsonl, derive_seed, df_hash, dumps, json_hash, read_json,
                   sha256_file, write_json)
from .datasets import cv_splits
from .engine import CatalogView, GraphError, ResolutionError, execute, summarize, validate_graph

TOOL_NAMES = ["list_sources", "inspect_schema", "profile_data", "query_data", "materialize_features",
              "fit_evaluate", "quality_check", "provenance_check", "submit", "abstain"]
MAX_OBS_CHARS = 3500


class BudgetExceeded(Exception):
    pass


class Terminal(Exception):
    """Raised by submit/abstain to end a run."""

    def __init__(self, state: str, info: dict):
        super().__init__(state)
        self.state, self.info = state, info


class Budget:
    def __init__(self, limits: dict, clock=time.time):
        self.limits = dict(limits)
        self.clock = clock
        self.start = clock()
        self.tokens_in = 0
        self.tokens_out = 0
        self.model_calls = 0
        self.tool_calls = 0
        self.gate_calls = 0
        self.repairs = 0
        self.evaluations = 0
        self.fits = 0
        self.wait_seconds = 0.0
        self.llm_seconds = 0.0

    @property
    def tokens(self) -> int:
        return self.tokens_in + self.tokens_out

    def active_seconds(self) -> float:
        return self.clock() - self.start - self.wait_seconds

    def exceeded(self) -> str | None:
        L = self.limits
        if self.tokens > L["aggregate_tokens"]:
            return "aggregate_tokens"
        if self.model_calls > L["model_calls"]:
            return "model_calls"
        if self.tool_calls > L["tool_calls"]:
            return "tool_calls"
        if self.active_seconds() > L["wall_seconds"]:
            return "wall_seconds"
        return None

    def check(self):
        r = self.exceeded()
        if r:
            raise BudgetExceeded(r)

    def remaining(self) -> dict:
        L = self.limits
        return {"tokens": L["aggregate_tokens"] - self.tokens, "model_calls": L["model_calls"] - self.model_calls,
                "tool_calls": L["tool_calls"] - self.tool_calls, "repairs": L["repair_cycles"] - self.repairs,
                "evaluations": L["pipeline_evaluations"] - self.evaluations,
                "seconds": round(L["wall_seconds"] - self.active_seconds(), 1)}

    def usage(self) -> dict:
        return {"input_tokens": self.tokens_in, "output_tokens": self.tokens_out, "aggregate_tokens": self.tokens,
                "model_calls": self.model_calls, "tool_calls": self.tool_calls, "gate_calls": self.gate_calls,
                "repair_cycles": self.repairs, "pipeline_evaluations": self.evaluations, "fit_count": self.fits,
                "active_seconds": round(self.active_seconds(), 3), "wait_seconds": round(self.wait_seconds, 3),
                "llm_seconds": round(self.llm_seconds, 3)}


def _truncate(obj, limit: int = MAX_OBS_CHARS) -> str:
    s = dumps(obj)
    if len(s) <= limit:
        return s
    return s[: limit - 40] + f' ..."[truncated {len(s) - limit + 40} chars]"'


class ToolSession:
    """One run's tool environment. Holds no reference to the private evaluation directory."""

    def __init__(self, run_id: str, visible: Path, trusted: Path, submission_dir: Path, system: dict,
                 budget: Budget, seeds: dict, fit_slot=None, query_timeout: float = 10.0):
        self.run_id = run_id
        self.visible, self.trusted = Path(visible), Path(trusted)
        self.sub_dir = Path(submission_dir)
        self.sub_dir.mkdir(parents=True, exist_ok=True)
        self.system = system
        self.q, self.p = bool(system["q"]), bool(system["p"])
        self.budget = budget
        self.seeds = seeds
        self.fit_slot = fit_slot
        self.query_timeout = query_timeout
        self.contract = read_json(self.visible / "contract.json")
        self.catalog = read_json(self.visible / "catalog.json")
        self.manifest = read_json(self.visible / "visible_partition_manifest.json")
        self.labels = pd.read_parquet(self.trusted / "labels.parquet")
        self.views = {p: CatalogView(self.visible, self.catalog, p) for p in ("train", "validation")}
        self.handles: dict = {}
        self.fit_cache: dict = {}
        self.fit_results: list = []
        self.provenance: list = []
        self.prov_path = self.sub_dir / "trusted_provenance.jsonl"
        self.last_failed = False
        self.blocked_attempts: list = []
        self.gate_log: list = []
        self._duck = None
        split = self.labels[["observation_id", "partition", "fold"]].copy()
        self.cv = cv_splits(split, self.contract_rule())

    def contract_rule(self) -> str:
        return self.manifest["cv_policy"]["rule"]

    # ------------------------------------------------------------------ provenance
    def _prov(self, activity: str, **kw):
        rec = {"seq": len(self.provenance), "time": round(time.time(), 3), "activity": activity,
               "agent": f"tool:{activity}", **kw}
        self.provenance.append(rec)
        append_jsonl(self.prov_path, rec)

    # ------------------------------------------------------------------ dispatch
    def call(self, name: str, args: dict, charge: bool = True) -> dict:
        """Execute one agent-requested tool call. Returns an observation dict."""
        if name not in TOOL_NAMES:
            return {"ok": False, "error": f"unknown tool {name!r}; tools: {TOOL_NAMES}"}
        if charge:
            self.budget.tool_calls += 1
            self.budget.check()
        fn = getattr(self, "t_" + name)
        try:
            res = fn(**(args or {}))
            return {"ok": "error" not in res, **res}
        except Terminal:
            raise
        except BudgetExceeded:
            raise
        except TypeError as e:
            return {"ok": False, "error": f"bad arguments for {name}: {e}"}
        except (GraphError, ValueError, KeyError, duckdb.Error) as e:
            return {"ok": False, "error": f"{type(e).__name__}: {e}"}

    # ------------------------------------------------------------------ tools
    def t_list_sources(self) -> dict:
        out = []
        for name, t in sorted(self.catalog["tables"].items()):
            shards: dict = {}
            for f in t["files"]:
                for s in f["shards"]:
                    shards.setdefault(s, []).append(f["version"])
            multi = sorted(s for s, v in shards.items() if len(v) > 1)
            sr = self.catalog.get("shard_representations", {}).get(name, {})
            out.append({"table": name, "grain": t["grain"], "key": t["key"], "description": t["description"],
                        "columns": list(t["columns"]), "versions": t["versions"], "files": len(t["files"]),
                        "shards": len(shards), "shards_with_multiple_candidates": len(multi),
                        "shards_with_declared_representations": len(sr)})
        auth = [{"record_id": a["record_id"], "registry": a["registry"], "table": a["table"], "as_of": a["as_of"],
                 "version": a["version"], "shards": len(a["shard_hashes"])} for a in self.catalog["authority"]]
        self._prov("list_sources")
        return {"tables": out, "authority": auth, "alias_table": self.catalog.get("alias_table"),
                "relationships": self.catalog.get("relationships", [])}

    def t_inspect_schema(self, table: str) -> dict:
        t = self.catalog["tables"].get(table)
        if t is None:
            raise KeyError(f"unknown table {table!r}")
        files = [{k: f[k] for k in ("file_id", "version", "snapshot_date", "modified", "partition", "rows")}
                 | {"shards": len(f["shards"])} for f in t["files"]]
        reps = {c: self.catalog["representations"][c] for c in t["columns"] if c in self.catalog.get("representations", {})}
        sr = self.catalog.get("shard_representations", {}).get(table, {})
        auth = [{"record_id": a["record_id"], "registry": a["registry"], "version": a["version"],
                 "as_of": a["as_of"], "shards": len(a["shard_hashes"])} for a in self.catalog["authority"] if a["table"] == table]
        self._prov("inspect_schema", used=[table])
        return {"table": table, "grain": t["grain"], "key": t["key"], "columns": t["columns"], "files": files[:16],
                "representations": reps, "shard_representations": dict(list(sr.items())[:12]),
                "shard_representations_total": len(sr), "authority": auth}

    def t_profile_data(self, table: str, policy: str = "authoritative", partition: str = "train",
                       sample_rows: int = 5) -> dict:
        if partition not in self.views:
            raise ValueError("partition must be 'train' or 'validation'")
        from .engine import resolve
        df, rec = resolve(self.views[partition], table, policy)
        sample_rows = max(0, min(int(sample_rows), 20))
        cols = {}
        for c in df.columns:
            s = df[c]
            info = {"dtype": str(s.dtype), "nulls": int(s.isna().sum()), "distinct": int(s.nunique())}
            if pd.api.types.is_numeric_dtype(s):
                info.update({"min": float(s.min()) if s.notna().any() else None,
                             "max": float(s.max()) if s.notna().any() else None,
                             "mean": float(s.mean()) if s.notna().any() else None})
            else:
                vals = s.dropna().astype(str)
                info["examples"] = vals.head(3).tolist()
                info["prefixed_values"] = int(vals.str.startswith("value:").sum())
            cols[c] = info
        dup = int(df.duplicated().sum())
        keydup = int(df["observation_id"].duplicated().sum()) if "observation_id" in df.columns and \
            self.catalog["tables"][table]["grain"] == "observation" else None
        self._prov("profile_data", used=[f"{s}:{v['file_id']}" for s, v in rec["shards"].items()], policy=policy)
        smp = df.head(sample_rows)
        return {"table": table, "partition": partition, "policy": policy, "rows": int(len(df)),
                "exact_duplicate_rows": dup, "duplicate_keys": keydup,
                "versions_loaded": sorted({v["version"] for v in rec["shards"].values()}),
                "columns": cols, "sample": smp.astype(object).where(smp.notna(), None).to_dict("records")}

    def _duckdb(self):
        if self._duck is None:
            con = duckdb.connect(":memory:")
            for name, t in self.catalog["tables"].items():
                for f in t["files"]:
                    df = pd.read_parquet(self.visible / f["path"])
                    con.register("tmp_df", df)
                    con.execute(f'CREATE TABLE "{f["file_id"]}" AS SELECT * FROM tmp_df')
                    con.unregister("tmp_df")
            con.execute("SET enable_external_access=false")
            try:
                con.execute("SET autoload_known_extensions=false")
                con.execute("SET autoinstall_known_extensions=false")
            except duckdb.Error:
                pass
            con.execute("SET lock_configuration=true")
            self._duck = con
        return self._duck

    def t_query_data(self, sql: str, max_rows: int = 50) -> dict:
        s = str(sql).strip().rstrip(";")
        low = s.lower()
        if not (low.startswith("select") or low.startswith("with")):
            raise ValueError("only SELECT/WITH queries over registered file tables are permitted")
        if ";" in s:
            raise ValueError("a single statement is permitted")
        con = self._duckdb()
        timer = threading.Timer(self.query_timeout, con.interrupt)
        timer.start()
        try:
            rel = con.execute(s)
            cols = [d[0] for d in rel.description]
            rows = rel.fetchmany(max(1, min(int(max_rows), 50)))
        finally:
            timer.cancel()
        self._prov("query_data", sql=s[:500])
        return {"columns": cols, "rows": [list(r) for r in rows], "row_count_returned": len(rows),
                "tables_hint": "Tables are named by file_id, e.g. observations__v2__train."}

    def t_materialize_features(self, graph: dict) -> dict:
        repair = self.last_failed
        if repair:
            if self.budget.repairs >= self.budget.limits["repair_cycles"]:
                return {"error": "repair limit reached: submit the best valid artifact or abstain",
                        "repairs_used": self.budget.repairs}
            self.budget.repairs += 1
        try:
            validate_graph(graph)
            frames, traces = {}, {}
            for p, v in self.views.items():
                frames[p], traces[p] = execute(graph, v)
        except ResolutionError as e:
            self.last_failed = True
            self._prov("materialize_features", status="error", error=str(e)[:300], reason=e.reason)
            return {"materialized": False, "error": str(e), "resolution_failure": e.reason}
        except GraphError as e:
            self.last_failed = True
            self._prov("materialize_features", status="error", error=str(e)[:300])
            return {"materialized": False, "error": str(e)}
        hid = f"H{len(self.handles) + 1}"
        handle = {"id": hid, "graph": graph, "graph_hash": json_hash(graph), "frames": frames, "traces": traces}
        self.handles[hid] = handle
        self._prov("materialize_features", status="ok", handle=hid, graph_hash=handle["graph_hash"],
                   used=sorted({f"{l['table']}:{s}:{v['file_id']}" for t in traces.values() for l in t["loads"]
                                for s, v in l["shards"].items()})[:400],
                   generated={p: traces[p]["output_hash"] for p in traces}, repair=repair)
        out = {"materialized": True, "handle": hid,
               "summary": {p: summarize(frames[p], len(self.manifest["partitions"][p]), sample_rows=2 if p == "train" else 0)
                           for p in frames},
               "operations": {n["id"]: {k: v for k, v in n.items() if k not in ("id",)} for n in traces["train"]["nodes"]
                              if n["op"] != "project"}}
        failed = False
        gate_out = []
        if self.q:
            v = gates.quality_gate(handle, self.contract, self.catalog, self.manifest)
            self._gate(v)
            gate_out.append(v)
            failed |= not v["passed"]
        if self.p:
            v = gates.provenance_gate(handle, self.contract, self.catalog, self.provenance)
            self._gate(v)
            gate_out.append(v)
            failed |= not v["passed"]
        handle["gates"] = gate_out
        if gate_out:
            out["gate_verdicts"] = gate_out
        self.last_failed = failed
        return out

    def _gate(self, verdict: dict):
        self.budget.gate_calls += 1
        self.gate_log.append(verdict)
        self._prov("gate", gate=verdict["gate"], artifact=verdict["artifact"], passed=verdict["passed"],
                   failures=[f["check"] for f in verdict["failures"]])

    def t_fit_evaluate(self, handle: str, candidate: str) -> dict:
        if handle not in self.handles:
            raise KeyError(f"unknown handle {handle!r}; materialize_features first")
        key = (handle, candidate)
        if key in self.fit_cache:
            return {"cached": True, **self.fit_cache[key]}
        if self.budget.evaluations >= self.budget.limits["pipeline_evaluations"]:
            return {"error": "pipeline evaluation limit (6) reached", "evaluations_used": self.budget.evaluations}
        self.budget.evaluations += 1
        h = self.handles[handle]
        seed = self.seeds["estimator"]
        t_wait = time.time()
        if self.fit_slot is not None:
            self.fit_slot.acquire()
        self.budget.wait_seconds += time.time() - t_wait
        try:
            res = modeling.fit_evaluate(candidate, h["frames"]["train"], h["frames"]["validation"], self.labels,
                                        self.cv, self.contract, seed)
            status = "ok"
        except Exception as e:  # failed attempts consume an evaluation slot
            res = {"candidate": candidate, "error": f"{type(e).__name__}: {e}"}
            status = "error"
        finally:
            if self.fit_slot is not None:
                self.fit_slot.release()
        self.budget.fits += res.get("fits", 0)
        self._prov("fit_evaluate", handle=handle, candidate=candidate, status=status, fit_partitions=["train"],
                   score_partitions=["train_cv", "validation"], validation=res.get("validation"))
        res["evaluations_used"] = self.budget.evaluations
        self.fit_results.append({"handle": handle, **res})
        if status == "error":
            self.last_failed = True
            self.fit_cache[key] = res
            return res
        gate_out = []
        if self.q:
            v = gates.fit_quality_gate(res, handle)
            self._gate(v)
            gate_out.append(v)
        if self.p:
            v = gates.provenance_gate(h, self.contract, self.catalog, self.provenance, stage="fit")
            self._gate(v)
            gate_out.append(v)
        if gate_out:
            res["gate_verdicts"] = gate_out
            self.last_failed = not all(g["passed"] for g in gate_out)
        slim = {k: res[k] for k in ("candidate", "metric", "cv_mean", "validation", "validation_reason", "rows",
                                    "evaluations_used") if k in res}
        if gate_out:
            slim["gate_verdicts"] = gate_out
        self.fit_cache[key] = slim
        return slim

    def t_quality_check(self, handle: str) -> dict:
        if not self.q:
            return {"error": "quality_check is not available in this configuration"}
        h = self.handles.get(handle)
        if h is None:
            raise KeyError(f"unknown handle {handle!r}")
        return {"verdicts": [g for g in h.get("gates", []) if g["gate"] == "quality"]}

    def t_provenance_check(self, handle: str) -> dict:
        if not self.p:
            return {"error": "provenance_check is not available in this configuration"}
        h = self.handles.get(handle)
        if h is None:
            raise KeyError(f"unknown handle {handle!r}")
        return {"verdicts": [g for g in h.get("gates", []) if g["gate"] == "provenance"]}

    def t_submit(self, handle: str, candidate: str, findings: list | None = None) -> dict:
        if handle not in self.handles:
            raise KeyError(f"unknown handle {handle!r}")
        if candidate not in self.contract["candidate_models"]:
            raise ValueError(f"candidate {candidate!r} is not permitted")
        h = self.handles[handle]
        gate_out = []
        if self.q:
            v = gates.quality_gate(h, self.contract, self.catalog, self.manifest)
            v["stage"] = "submit"
            self._gate(v)
            gate_out.append(v)
        if self.p:
            v = gates.provenance_gate(h, self.contract, self.catalog, self.provenance, stage="submit", candidate=candidate)
            self._gate(v)
            gate_out.append(v)
        if gate_out and not all(g["passed"] for g in gate_out):
            self.last_failed = True
            self._prov("submit", status="rejected_by_gates", handle=handle, candidate=candidate)
            return {"submitted": False, "error": "final gate checks failed; repair or abstain", "gate_verdicts": gate_out}
        seed = self.seeds["estimator"]
        t_wait = time.time()
        if self.fit_slot is not None:
            self.fit_slot.acquire()
        self.budget.wait_seconds += time.time() - t_wait
        try:
            model, info = modeling.refit(candidate, h["frames"]["train"], h["frames"]["validation"], self.labels,
                                         self.contract, seed)
        except Exception as e:
            self.last_failed = True
            self._prov("submit", status="refit_error", handle=handle, candidate=candidate, error=str(e)[:300])
            return {"submitted": False, "error": f"final refit failed: {type(e).__name__}: {e}"}
        finally:
            if self.fit_slot is not None:
                self.fit_slot.release()
        self.budget.fits += 1
        clean_findings = []
        for f in (findings or [])[:50]:
            if isinstance(f, dict):
                clean_findings.append({k: str(v)[:200] for k, v in f.items()})
            else:
                clean_findings.append({"finding": str(f)[:200]})
        files = {}
        write_json(self.sub_dir / "graph.json", h["graph"])
        with open(self.sub_dir / "model.pkl", "wb") as fh:
            pickle.dump(model, fh, protocol=4)
        write_json(self.sub_dir / "model_choice.json", {"candidate": candidate, "handle": handle,
                                                         "effective_params": modeling.effective_params(candidate), **info})
        write_json(self.sub_dir / "findings.json", {"findings": clean_findings,
                                                     "operations": {p: t["operations"] for p, t in h["traces"].items()}})
        for fn in ("graph.json", "model.pkl", "model_choice.json", "findings.json"):
            files[fn] = sha256_file(self.sub_dir / fn)
        bundle = {"run_id": self.run_id, "handle": handle, "candidate": candidate, "graph_hash": h["graph_hash"],
                  "files": files, "visible_output_hashes": {p: t["output_hash"] for p, t in h["traces"].items()},
                  "fit_partitions": info["fit_partitions"]}
        write_json(self.sub_dir / "bundle.json", bundle)
        self._prov("submit", status="ok", handle=handle, candidate=candidate, bundle_hash=json_hash(bundle),
                   fit_partitions=info["fit_partitions"])
        raise Terminal("SUBMITTED", {"handle": handle, "candidate": candidate})

    def t_abstain(self, reason: str, detail: str = "") -> dict:
        if reason not in ABSTENTION_REASONS:
            raise ValueError(f"reason must be one of {ABSTENTION_REASONS}")
        self._prov("abstain", reason=reason, detail=str(detail)[:300])
        raise Terminal("ABSTAINED", {"reason": reason, "detail": str(detail)[:300]})

    def close(self):
        if self._duck is not None:
            try:
                self._duck.close()
            except Exception:
                pass
            self._duck = None


### Independent evaluator (Section 14)

In [ ]:
%%writefile adm_code/adm/evaluator.py
"""Independent evaluator (Section 14). Runs in a separate process with the private directory.

It does not call the internal gates. Critical checks (records, values, authority hashes, fit scope,
predictions) are implemented here against independently computed hidden references."""
from __future__ import annotations

import gzip
import json
import pickle
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

from . import modeling
from .core import df_hash, read_json, read_jsonl, sha256_file, write_json, json_hash
from .engine import CatalogView, GraphError, ResolutionError, authority_records, execute

ATOL, RTOL = 1e-8, 1e-6
FAILURE_ORDER = ["ARTIFACT_MISMATCH", "EXECUTION", "SOURCE_VERSION", "FORBIDDEN_FEATURE", "MISSING_FEATURE",
                 "WRONG_JOIN", "MISSING_RECORD", "ROW_MULTIPLICITY", "PARSE", "UNIT_MISMATCH", "BAD_AGGREGATE",
                 "VALUE_MISMATCH", "FIT_SCOPE", "INVALID_PREDICTION", "BUDGET"]


def _close(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.isclose(a, b, rtol=RTOL, atol=ATOL, equal_nan=True)


def _authority_content_check(view: CatalogView, trace: dict) -> list:
    """Recompute content hashes of every loaded shard and compare with the unique authority record."""
    problems = []
    cache: dict = {}
    for load in trace["loads"]:
        recs = authority_records(view.catalog, load["table"])
        distinct = {(r["version"], tuple(sorted(r["shard_hashes"].items()))) for r in recs}
        if len(distinct) != 1:
            problems.append(f"{load['table']}: authority is {'missing' if not recs else 'ambiguous'}")
            continue
        rec = recs[0]
        files = {f["file_id"]: f for f in view.catalog["tables"][load["table"]]["files"]}
        wanted = {s: h for s, h in rec["shard_hashes"].items() if s.startswith(view.partition + "-")}
        if set(load["shards"]) != set(wanted):
            problems.append(f"{load['table']}: loaded shards differ from authorised shards")
        for s, v in load["shards"].items():
            f = files[v["file_id"]]
            if f["file_id"] not in cache:
                cache[f["file_id"]] = pd.read_parquet(view.root / f["path"])
            d = cache[f["file_id"]]
            content = df_hash(d[d["_shard"] == s].reset_index(drop=True))
            if content != wanted.get(s) or v["version"] != rec["version"]:
                problems.append(f"{load['table']}:{s} uses {v['file_id']} (version {v['version']}), not the authorised content")
    return problems


def compare_features(out: pd.DataFrame, ref: pd.DataFrame, required: list, catalog: dict, graph: dict,
                     aggregate_cols: set) -> dict:
    """Compare a replayed feature table to the hidden reference. Returns failure codes and details."""
    codes, details = [], {}
    cols = [c for c in out.columns if c != "observation_id"]
    excluded_extra = [c for c in cols if c not in required]
    missing_cols = [c for c in required if c not in out.columns]
    if excluded_extra:
        codes.append("FORBIDDEN_FEATURE")
        details["extra_columns"] = excluded_extra[:20]
    if missing_cols:
        codes.append("MISSING_FEATURE")
        details["missing_columns"] = missing_cols[:20]
    ids = out["observation_id"].astype(str)
    ref_ids = set(ref["observation_id"])
    counts = ids.value_counts()
    missing_ids = ref_ids - set(ids)
    unexpected = set(ids) - ref_ids
    dup_ids = counts[counts > 1]
    if missing_ids or unexpected:
        codes.append("MISSING_RECORD")
        details["missing_ids"] = len(missing_ids)
        details["unexpected_ids"] = len(unexpected)
    if len(dup_ids):
        has_join = any(n["op"] == "join" for n in graph["nodes"])
        codes.append("ROW_MULTIPLICITY")
        details["duplicated_ids"] = int(len(dup_ids))
        if has_join and dup_ids.max() > 2:
            codes.append("WRONG_JOIN")
    o = out.assign(observation_id=ids).drop_duplicates("observation_id").set_index("observation_id")
    r = ref.set_index("observation_id")
    common = sorted(set(o.index) & set(r.index))
    o, r = o.loc[common], r.loc[common]
    value_report = {}
    total, matched = 0, 0
    reps = catalog.get("representations", {})
    has_left = any(n["op"] == "join" and n.get("how", "inner") == "left" for n in graph["nodes"])
    for c in required:
        if c not in o.columns:
            continue
        rv = r[c]
        if not pd.api.types.is_numeric_dtype(rv):
            ok = (o[c].astype(str).to_numpy() == rv.astype(str).to_numpy())
            total += len(ok)
            matched += int(ok.sum())
            if not ok.all():
                value_report[c] = {"mismatches": int((~ok).sum()), "kind": "VALUE_MISMATCH"}
            continue
        if not pd.api.types.is_numeric_dtype(o[c]):
            value_report[c] = {"mismatches": int(len(o)), "kind": "PARSE"}
            total += len(o)
            continue
        a = o[c].to_numpy(dtype=float)
        b = rv.to_numpy(dtype=float)
        ok = _close(a, b)
        total += len(ok)
        matched += int(ok.sum())
        if ok.all():
            continue
        bad = ~ok
        kind = "VALUE_MISMATCH"
        nan_for_value = bad & np.isnan(a) & ~np.isnan(b)
        if c in aggregate_cols:
            kind = "BAD_AGGREGATE"
        elif nan_for_value.sum() == bad.sum() and has_left:
            kind = "WRONG_JOIN"
        elif c in reps:
            fin = bad & ~np.isnan(a) & ~np.isnan(b)
            if fin.any():
                for opt in reps[c]["options"].values():
                    if opt["a"] == 1.0 and opt["b"] == 0.0:
                        continue
                    if _close(a[fin], opt["a"] * b[fin] + opt["b"]).mean() > 0.9 or \
                       _close(a[fin], (b[fin] - opt["b"]) / opt["a"]).mean() > 0.9:
                        kind = "UNIT_MISMATCH"
                        break
        value_report[c] = {"mismatches": int(bad.sum()), "kind": kind}
    for c, v in value_report.items():
        if v["kind"] not in codes:
            codes.append(v["kind"])
    details["value_mismatches"] = value_report
    return {"codes": codes, "details": details, "value_match_rate": (matched / total) if total else None,
            "record_integrity": (sum(1 for i in ref_ids if counts.get(i, 0) == 1) / len(ref_ids)) if ref_ids else None}


def _data_management(ledger: list, traces: dict, outs: dict, refs: dict, catalog_vis: dict, required: list) -> dict:
    """Detection, repair and unintended-edit metrics over all partitions (Section 14.4)."""
    rep_ops = {"encoded": set(), "duplicate": set(), "alias": set(), "unit": set(), "key_format": set()}
    for part, tr in traces.items():
        for op in tr["operations"]:
            if op["op"] == "parse_numeric":
                rep_ops["encoded"] |= {(part, a, b) for a, b in op["locations"]}
            elif op["op"] == "dedupe":
                rep_ops["duplicate"] |= {(part, a) for a in op["locations"]}
            elif op["op"] == "map_alias":
                rep_ops["alias"] |= {(part, a) for a in op["locations"]}
            elif op["op"] == "normalize_key":
                rep_ops["key_format"] |= {(part, a) for a in op["locations"]}
            elif op["op"] in ("to_canonical", "infer_representation"):
                rep_ops["unit"] |= {(part, op.get("table"), s, f) for s, f, _ in op["locations"]}
    injected = {"encoded": set(), "duplicate": set(), "alias": set(), "unit": set(), "key_format": set()}
    for e in ledger:
        if e["defect"] == "cell_encoded":
            injected["encoded"].add((e["partition"], e["observation_id"], e["column"]))
        elif e["defect"] in ("duplicate", "near_duplicate"):
            injected["duplicate"].add((e["partition"], e["observation_id"]))
        elif e["defect"] == "alias":
            injected["alias"].add((e["partition"], e["observation_id"]))
        elif e["defect"] == "key_variant":
            injected["key_format"].add((e["partition"], e["observation_id"]))
        elif e["defect"] in ("unit_representation", "undeclared_representation"):
            injected["unit"].add((e["partition"], e["table"], e["shard"], e["column"]))
    # required template conversions (T4 baseline) are not defect detections
    baseline_unit = set()
    for t, shards in catalog_vis.get("_all_shard_representations", {}).items():
        for s, fmap in shards.items():
            for f in fmap:
                baseline_unit.add((s.split("-")[0], t, s, f))
    baseline_unit -= injected["unit"]
    det = {}
    for k in injected:
        rep = rep_ops[k] - (baseline_unit if k == "unit" else set())
        inj = injected[k]
        tp = len(rep & inj)
        det[k] = {"injected": len(inj), "reported": len(rep), "true_positive": tp,
                  "precision": (tp / len(rep)) if rep else None,
                  "recall": (tp / len(inj)) if inj else None}
    # exact reversible repair: check affected values/records in replayed outputs against references
    repaired, reversible = 0, 0
    unintended, unaffected = 0, 0
    affected_cells = {(e["partition"], e["observation_id"], e["column"]) for e in ledger if e["defect"].startswith("cell_")}
    for part in outs:
        o = outs[part]
        r = refs[part]
        if o is None:
            continue
        o2 = o.assign(observation_id=o["observation_id"].astype(str))
        cnt = o2["observation_id"].value_counts()
        o2 = o2.drop_duplicates("observation_id").set_index("observation_id")
        r2 = r.set_index("observation_id")
        common = r2.index.intersection(o2.index)
        cols = [c for c in required if c in o2.columns and pd.api.types.is_numeric_dtype(r2[c])]
        for e in ledger:
            if e["partition"] != part or not e.get("reversible"):
                continue
            reversible += 1
            if e["defect"] == "cell_encoded":
                oid, c = e["observation_id"], e["column"]
                if oid in common and c in o2.columns and pd.api.types.is_numeric_dtype(o2[c]):
                    if _close(np.array([float(o2.at[oid, c])]), np.array([float(r2.at[oid, c])]))[0]:
                        repaired += 1
            elif e["defect"] in ("duplicate", "alias", "near_duplicate", "key_variant"):
                oid = e["observation_id"]
                if cnt.get(oid, 0) == 1 and oid in common and all(
                        _close(np.array([float(o2.at[oid, c]) if pd.api.types.is_numeric_dtype(o2[c]) else np.nan]),
                               np.array([float(r2.at[oid, c])]))[0] for c in cols):
                    repaired += 1
            elif e["defect"] in ("unit_representation", "undeclared_representation", "stale_candidate", "content_drift"):
                tr = traces.get(part)
                if tr is None:
                    continue
                if e["defect"] == "stale_candidate":
                    ok = any(l["table"] == e["table"] and e["shard"] in l["shards"] and
                             not l["shards"][e["shard"]]["file_id"].endswith("refresh") for l in tr["loads"])
                elif e["defect"] == "content_drift":
                    ok = any(l["table"] == e["table"] and e["shard"] in l["shards"] and
                             l["shards"][e["shard"]]["file_id"].endswith("__replica") for l in tr["loads"])
                else:
                    ok = (part, e["table"], e["shard"], e["column"]) in rep_ops["unit"]
                    if ok and e["column"] in o2.columns and pd.api.types.is_numeric_dtype(o2[e["column"]]):
                        a = o2.loc[common, e["column"]].to_numpy(dtype=float)
                        b = r2.loc[common, e["column"]].to_numpy(dtype=float)
                        ok = bool(_close(a, b).all())
                repaired += int(ok)
        for c in cols:
            if not pd.api.types.is_numeric_dtype(o2[c]):
                continue
            a = o2.loc[common, c].to_numpy(dtype=float)
            b = r2.loc[common, c].to_numpy(dtype=float)
            mask = np.array([(part, i, c) not in affected_cells for i in common])
            unaffected += int(mask.sum())
            unintended += int((~_close(a[mask], b[mask])).sum())
    return {"detection": det, "exact_reversible_repair_rate": (repaired / reversible) if reversible else None,
            "reversible_defects": reversible, "unintended_edit_rate": (unintended / unaffected) if unaffected else None}


def evaluate(job: dict) -> dict:
    """Grade one run. ``job`` keys: run_id, terminal_state, abstention_reason, instance (dict of dirs),
    submission_dir, usage, limits, negative (expected reason or None), retain_model."""
    t0 = time.time()
    rec = {"run_id": job["run_id"], "evaluator_version": 1, "verified_completion": None,
           "external_failure_codes": [], "first_failure": None, "test_metrics": None}
    negative = job.get("negative")
    state = job["terminal_state"]
    if negative:
        rec["negative_control"] = True
        rec["correct_refusal"] = bool(state == "ABSTAINED" and job.get("abstention_reason") == negative)
        rec["false_submission"] = bool(state == "SUBMITTED")
        rec["verified_completion"] = None
        rec["evaluator_seconds"] = round(time.time() - t0, 3)
        return rec
    if state != "SUBMITTED":
        rec["verified_completion"] = 0
        rec["external_failure_codes"] = ["EXECUTION" if state in ("EXECUTION_ERROR",) else
                                         "BUDGET" if state == "BUDGET_EXCEEDED" else state]
        rec["first_failure"] = rec["external_failure_codes"][0]
        rec["evaluator_seconds"] = round(time.time() - t0, 3)
        return rec
    sub = Path(job["submission_dir"])
    inst = {k: Path(v) for k, v in job["instance"].items()}
    codes, predicates = [], {}
    # 1. artifacts and hashes
    try:
        bundle = read_json(sub / "bundle.json")
        ok = all((sub / fn).exists() and sha256_file(sub / fn) == h for fn, h in bundle["files"].items())
        graph = read_json(sub / "graph.json")
        choice = read_json(sub / "model_choice.json")
        ok = ok and json_hash(graph) == bundle["graph_hash"]
    except Exception as e:
        ok, bundle, graph, choice = False, None, None, None
        predicates["artifacts_error"] = str(e)[:200]
    predicates["artifacts"] = ok
    if not ok:
        codes.append("ARTIFACT_MISMATCH")
        return _finish(rec, codes, predicates, t0)
    contract = read_json(inst["visible"] / "contract.json")
    meta = read_json(inst["private"] / "instance_meta.json")
    required = meta["required_features"]
    with open(sub / "model.pkl", "rb") as fh:
        model = pickle.load(fh)
    permitted = choice["candidate"] in contract["candidate_models"]
    est = model.named_steps.get("model") if hasattr(model, "named_steps") else None
    expected_est = type(modeling.make_estimator(choice["candidate"], 0)).__name__ if permitted else None
    predicates["permitted_estimator"] = bool(permitted and est is not None and type(est).__name__ == expected_est)
    if not predicates["permitted_estimator"]:
        codes.append("ARTIFACT_MISMATCH")
    # 2. trusted provenance consistency
    prov = read_jsonl(sub / "trusted_provenance.jsonl")
    mat = [p for p in prov if p["activity"] == "materialize_features" and p.get("handle") == bundle["handle"]]
    sub_rec = [p for p in prov if p["activity"] == "submit" and p.get("status") == "ok"]
    prov_ok = bool(mat) and mat[-1]["graph_hash"] == bundle["graph_hash"] and len(sub_rec) == 1
    predicates["provenance_consistent"] = prov_ok
    if not prov_ok:
        codes.append("ARTIFACT_MISMATCH")
    fits = [p for p in prov if p["activity"] == "fit_evaluate"]
    scope_ok = all(p.get("fit_partitions") == ["train"] for p in fits) and \
        bundle.get("fit_partitions") == ["train", "validation"] == choice.get("fit_partitions")
    predicates["fit_scope"] = scope_ok
    if not scope_ok:
        codes.append("FIT_SCOPE")
    # 3. replay on hidden test and visible partitions
    vis_cat = read_json(inst["visible"] / "catalog.json")
    test_cat = read_json(inst["private"] / "catalog.json")
    outs, traces = {}, {}
    replay_error = None
    for part, root, cat in [("train", inst["visible"], vis_cat), ("validation", inst["visible"], vis_cat),
                            ("test", inst["private"], test_cat)]:
        try:
            view = CatalogView(root, cat, part)
            outs[part], traces[part] = execute(graph, view)
            # replay must reproduce the visible artifacts exactly
            if part != "test" and traces[part]["output_hash"] != bundle["visible_output_hashes"].get(part):
                codes.append("ARTIFACT_MISMATCH")
                predicates["visible_replay_identical"] = False
        except ResolutionError as e:
            outs[part], traces[part] = None, None
            replay_error = ("SOURCE_VERSION", str(e))
        except GraphError as e:
            outs[part], traces[part] = None, None
            replay_error = ("EXECUTION", str(e))
    predicates.setdefault("visible_replay_identical", True)
    if replay_error:
        codes.append(replay_error[0])
        predicates["replay_error"] = replay_error[1][:300]
    # 4. authority (independent content hashes)
    auth_problems = []
    for part, cat, root in [("train", vis_cat, inst["visible"]), ("validation", vis_cat, inst["visible"]),
                            ("test", test_cat, inst["private"])]:
        if traces.get(part):
            auth_problems += [f"{part}: {p}" for p in _authority_content_check(CatalogView(root, cat, part), traces[part])]
    predicates["source_authority"] = not auth_problems
    if auth_problems:
        codes.append("SOURCE_VERSION")
        predicates["authority_problems"] = auth_problems[:8]
    refs = {p: pd.read_parquet(inst["private"] / f"reference_{p}.parquet") for p in ("train", "validation", "test")}
    agg_cols = {c for c in required if c.endswith(("_mean", "_std", "_count"))} if meta["template"] == "T6" else set()
    comp = None
    if outs.get("test") is not None:
        comp = compare_features(outs["test"], refs["test"], required, test_cat, graph, agg_cols)
        codes += comp["codes"]
        predicates["feature_comparison"] = comp["details"]
        rec["test_value_match_rate"] = comp["value_match_rate"]
        rec["record_integrity_rate"] = comp["record_integrity"]
    # 5. predictions
    y = pd.read_parquet(inst["private"] / "labels_test.parquet")
    pred_ok = False
    if outs.get("test") is not None:
        X = outs["test"].copy()
        X["observation_id"] = X["observation_id"].astype(str)
        X = X.sort_values("observation_id", kind="mergesort").reset_index(drop=True)
        try:
            feats = choice["numeric"] + choice["categorical"]
            missing = [c for c in feats if c not in X.columns]
            if missing:
                raise ValueError(f"replayed features lack model inputs {missing}")
            for c in choice["numeric"]:
                if not pd.api.types.is_numeric_dtype(X[c]):
                    X[c] = pd.to_numeric(X[c], errors="coerce")
            pr = modeling.predict_scores(model, X[feats], contract["task_type"])
            if contract["task_type"] == "multiclass":
                labels, proba = pr
                valid = np.isfinite(proba).all() and (proba >= -1e-9).all() and np.allclose(proba.sum(axis=1), 1, atol=1e-6)
                yhat = np.asarray(labels)[np.argmax(proba, axis=1)]
                pdf = pd.DataFrame({"observation_id": X["observation_id"], "prediction": yhat})
                for i, l in enumerate(labels):
                    pdf[f"p_{l}"] = proba[:, i]
            else:
                arr = np.asarray(pr, dtype=float)
                valid = bool(np.isfinite(arr).all())
                if contract["task_type"] == "binary":
                    valid = valid and bool(((arr >= 0) & (arr <= 1)).all())
                pdf = pd.DataFrame({"observation_id": X["observation_id"], "prediction": arr})
            one_per = (pdf["observation_id"].value_counts().max() == 1) and set(pdf["observation_id"]) == set(y["observation_id"])
            pred_ok = bool(valid and one_per)
            pdf.to_parquet(sub / "test_predictions.parquet", index=False)
            # metrics on the reference-id rows that have predictions
            m = y.merge(pdf.drop_duplicates("observation_id"), on="observation_id", how="inner")
            rec["test_prediction_coverage"] = len(m) / max(1, len(y))
            if len(m):
                if contract["task_type"] == "multiclass":
                    pm = (list(labels), m[[f"p_{l}" for l in labels]].to_numpy())
                else:
                    pm = m["prediction"].to_numpy(dtype=float)
                rec["test_metrics"] = modeling.full_metrics(contract["task_type"], m["target"].to_numpy(), pm)
                rec["test_metrics"]["n"] = int(len(m))
                # replay determinism: predict a second time on the frozen artifact
                pr2 = modeling.predict_scores(model, X[feats], contract["task_type"])
                a1 = pr[1] if contract["task_type"] == "multiclass" else np.asarray(pr)
                a2 = pr2[1] if contract["task_type"] == "multiclass" else np.asarray(pr2)
                rec["replay_status"] = "identical" if np.array_equal(a1, a2) else "different"
        except Exception as e:
            predicates["prediction_error"] = f"{type(e).__name__}: {e}"[:300]
    predicates["predictions_valid"] = pred_ok
    if not pred_ok:
        codes.append("INVALID_PREDICTION")
    # 6. resources
    usage, limits = job.get("usage", {}), job.get("limits", {})
    within = (usage.get("aggregate_tokens", 0) <= limits.get("aggregate_tokens", float("inf")) and
              usage.get("model_calls", 0) <= limits.get("model_calls", float("inf")) and
              usage.get("tool_calls", 0) <= limits.get("tool_calls", float("inf")) and
              usage.get("pipeline_evaluations", 0) <= limits.get("pipeline_evaluations", float("inf")) and
              usage.get("active_seconds", 0) <= limits.get("wall_seconds", float("inf")))
    predicates["within_limits"] = within
    if not within:
        codes.append("BUDGET")
    # data-management metrics
    try:
        with gzip.open(inst["private"] / "ledger.json.gz", "rt") as f:
            ledger = json.load(f)
        allsr = {}
        for cat in (vis_cat, test_cat):
            for t, shards in cat.get("shard_representations", {}).items():
                allsr.setdefault(t, {}).update(shards)
        vis_cat2 = dict(vis_cat, _all_shard_representations=allsr)
        rec["data_management"] = _data_management(ledger, {k: v for k, v in traces.items() if v},
                                                  {k: v for k, v in outs.items()}, refs, vis_cat2, required)
        rec["version_selection_correct"] = predicates["source_authority"]
    except Exception as e:
        rec["data_management"] = {"error": f"{type(e).__name__}: {e}"[:300]}
    if not job.get("retain_model", False):
        try:
            (sub / "model.pkl").unlink()
            rec["model_retained"] = False
        except OSError:
            pass
    return _finish(rec, codes, predicates, t0)


def _finish(rec: dict, codes: list, predicates: dict, t0: float) -> dict:
    uniq = []
    for c in codes:
        if c not in uniq:
            uniq.append(c)
    uniq.sort(key=lambda c: FAILURE_ORDER.index(c) if c in FAILURE_ORDER else 99)
    rec["external_failure_codes"] = uniq
    rec["first_failure"] = uniq[0] if uniq else None
    rec["verified_completion"] = 0 if uniq else 1
    rec["predicates"] = predicates
    rec["evaluator_seconds"] = round(time.time() - t0, 3)
    return rec


def evaluate_safe(job: dict) -> dict:
    try:
        return evaluate(job)
    except Exception as e:
        return {"run_id": job["run_id"], "verified_completion": 0 if not job.get("negative") else None,
                "external_failure_codes": ["EVALUATOR_ERROR"], "first_failure": "EVALUATOR_ERROR",
                "evaluator_error": traceback.format_exc()[-2000:]}


if __name__ == "__main__":  # python -m adm.evaluator job.json out.json
    job = read_json(sys.argv[1])
    write_json(sys.argv[2], evaluate_safe(job))


### Language-model clients (vLLM and the scripted MockLLM)

In [ ]:
%%writefile adm_code/adm/llm.py
"""Language-model clients: an OpenAI-compatible vLLM client with schema-constrained JSON, and a
scripted MockLLM used to test the full pipeline without a GPU (mock results are never study results)."""
from __future__ import annotations

import json
import random
import re
import time
import urllib.error
import urllib.request

from .core import dumps
from .fixed import contract_graph

ROLE_ACTIONS = {
    "single": ["list_sources", "inspect_schema", "profile_data", "query_data", "materialize_features", "fit_evaluate",
               "quality_check", "provenance_check", "submit", "abstain"],
    "coordinator": ["delegate", "list_sources", "inspect_schema", "profile_data", "query_data", "materialize_features",
                    "fit_evaluate", "quality_check", "provenance_check", "submit", "abstain"],
    "data_manager": ["list_sources", "inspect_schema", "profile_data", "query_data", "materialize_features",
                     "quality_check", "provenance_check", "report"],
    "miner": ["fit_evaluate", "quality_check", "provenance_check", "report"],
}


def action_schema(role: str) -> dict:
    return {"type": "object",
            "properties": {"thought": {"type": "string"},
                           "action": {"type": "string", "enum": ROLE_ACTIONS[role]},
                           "args": {"type": "object"}},
            "required": ["thought", "action", "args"], "additionalProperties": False}


def estimate_tokens(text: str) -> int:
    return max(1, int(len(text) / 3.2))


class LLMError(RuntimeError):
    def __init__(self, msg: str, status: int | None = None):
        super().__init__(msg)
        self.status = status

    @property
    def context_overflow(self) -> bool:
        m = str(self).lower()
        return self.status == 400 and ("context" in m or "maximum" in m or "too long" in m or "max_model_len" in m)


class VLLMClient:
    """Calls a local vLLM OpenAI-compatible server."""

    def __init__(self, base_url: str, model: str, generation: dict, structured_mode: str = "response_format",
                 timeout: float = 600.0):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.gen = generation
        self.mode = structured_mode
        self.timeout = timeout

    def _post(self, payload: dict) -> dict:
        data = json.dumps(payload).encode()
        last = None
        for attempt in range(3):  # transient connection resets are retried; HTTP errors are not
            req = urllib.request.Request(self.base_url + "/v1/chat/completions", data=data,
                                         headers={"Content-Type": "application/json"})
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    return json.loads(r.read())
            except urllib.error.HTTPError as e:
                body = e.read().decode(errors="replace")[:500]
                raise LLMError(f"HTTP {e.code}: {body}", e.code) from None
            except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
                last = e
                time.sleep(1 + 2 * attempt)
        raise LLMError(f"connection error: {last}")

    def complete(self, messages: list, role: str, seed: int) -> dict:
        payload = {"model": self.model, "messages": messages, "temperature": self.gen["temperature"],
                   "top_p": self.gen["top_p"], "top_k": self.gen["top_k"],
                   "max_tokens": self.gen["max_output_tokens_per_call"], "seed": int(seed)}
        if "enable_thinking" in self.gen:
            payload["chat_template_kwargs"] = {"enable_thinking": bool(self.gen["enable_thinking"])}
        schema = action_schema(role)
        if self.mode == "response_format":
            payload["response_format"] = {"type": "json_schema",
                                          "json_schema": {"name": f"{role}_action", "schema": schema, "strict": True}}
        elif self.mode == "structured_outputs":
            payload["structured_outputs"] = {"json": schema}
        elif self.mode == "guided_json":
            payload["guided_json"] = schema
        t0 = time.time()
        resp = self._post(payload)
        dt = time.time() - t0
        ch = resp["choices"][0]
        usage = resp.get("usage") or {}
        return {"text": ch["message"].get("content") or "", "input_tokens": int(usage.get("prompt_tokens", 0)),
                "output_tokens": int(usage.get("completion_tokens", 0)), "seconds": dt,
                "finish_reason": ch.get("finish_reason"),
                "cached_tokens": int(((usage.get("prompt_tokens_details") or {}).get("cached_tokens")) or 0)}


# --------------------------------------------------------------------------- mock
MOCK_MISTAKES = {
    "latest_policy": 0.30, "skip_alias": 0.25, "skip_dedupe": 0.25, "skip_parse": 0.20, "skip_canonical": 0.30,
    "sample_std": 0.30, "keep_excluded": 0.15, "drop_missing": 0.10,
}
GATE_FIXES = {
    "authority": ["latest_policy"], "record_coverage": ["skip_alias", "drop_missing"], "key_uniqueness": ["skip_dedupe"],
    "numeric_types": ["skip_parse"], "canonical_units": ["skip_canonical"], "value_ranges": ["skip_canonical"],
    "aggregate": ["sample_std"], "excluded_predictors": ["keep_excluded"], "required_columns": ["keep_excluded"],
    "missing_value_policy": ["drop_missing"], "label_match": ["skip_alias"], "validation_coverage": ["skip_alias", "skip_dedupe"],
}


class MockLLM:
    """Deterministic scripted policy that reads the conversation and emits schema-valid actions.

    Each run draws a set of workflow mistakes. Ordinary execution errors and (when the configuration has
    them) gate verdicts let it repair specific mistakes, so the full pipeline exercises passes, failures,
    repairs, refusals and false submissions. It exists only to test plumbing."""

    def __init__(self, seed: int, invalid_rate: float = 0.03):
        self.rng = random.Random(seed)
        self.mistakes = {k for k, p in MOCK_MISTAKES.items() if self.rng.random() < p}
        self.invalid_rate = invalid_rate
        self.contract = None
        self.catalog_tables = None
        self.handle = None
        self.evals: dict = {}
        self.plan = ["CL3", "CL5", "CL1"]
        self.phase: dict = {}
        self.repairs = 0
        self.abstain_reason = None

    @staticmethod
    def _last_obs(messages: list) -> dict | None:
        for m in reversed(messages):
            if m["role"] != "user":
                continue
            # a user message may hold several merged observations; use the last real (non-format) one
            chunks = [c for c in m["content"].split("\n\n") if c.startswith("Observation") and
                      not c.startswith("Observation (format_error)")]
            if not chunks and "Observation (format_error)" in m["content"]:
                continue
            if chunks:
                try:
                    return json.loads(chunks[-1].split("\n", 1)[1])
                except Exception:
                    return None
            return None
        return None

    def _contract(self, messages):
        if self.contract is None:
            for m in messages:
                mm = re.search(r"<contract_json>(.*?)</contract_json>", m["content"], re.S)
                if mm:
                    self.contract = json.loads(mm.group(1))
                    break
        return self.contract


    def _graph(self) -> dict:
        g = contract_graph(self.contract, {"tables": self.catalog_tables})
        M = self.mistakes
        alias, nodes = {}, []
        for n in g["nodes"]:
            n = dict(n)
            for r in ("input", "left", "right"):
                if r in n:
                    n[r] = alias.get(n[r], n[r])
            op = n["op"]
            if (op == "map_alias" and "skip_alias" in M) or (op == "dedupe" and "skip_dedupe" in M) or \
               (op == "parse_numeric" and "skip_parse" in M) or (op == "to_canonical" and "skip_canonical" in M):
                alias[n["id"]] = n["input"]
                continue
            if op == "load" and "latest_policy" in M:
                n["policy"] = "latest_modified"
            if op == "aggregate" and "sample_std" in M:
                n["stats"] = ["mean", "sample_std", "count"]
            if op == "project" and n["id"] == "features":
                if "keep_excluded" in M:
                    ex = [c for c in self.contract["excluded_predictors"] if c not in ("observation_id", "profile_key", "_shard")]
                    if ex:
                        n["columns"] = n["columns"] + ex[:1]
                if "drop_missing" in M:
                    nodes.append({"id": "dropna", "op": "drop_missing_rows", "input": n["input"]})
                    n["input"] = "dropna"
            nodes.append(n)
        return {"nodes": nodes, "output": alias.get(g["output"], g["output"])}

    def _apply_feedback(self, obs: dict | None) -> bool:
        if not obs:
            return False
        texts = [f["check"] + " " + f.get("detail", "") for v in (obs.get("gate_verdicts") or []) for f in v.get("failures", [])]
        if obs.get("error"):
            texts.append(str(obs["error"]))
        fixed = False
        for t in texts:
            keys = [k for k in GATE_FIXES if k in t]
            if "parse_numeric" in t or "non-numeric" in t or "not numeric" in t:
                keys.append("numeric_types")
            for k in keys:
                for m in GATE_FIXES[k]:
                    if m in self.mistakes:
                        self.mistakes.discard(m)
                        fixed = True
        return fixed

    @staticmethod
    def _act(thought, action, args):
        return {"thought": thought, "action": action, "args": args}

    def _cands(self):
        return list((self.contract or {}).get("candidate_models", ["CL1"]))

    def _best(self):
        metric = (self.contract or {}).get("selection_metric", "average_precision")
        if self.evals:
            return (min if metric in ("rmse", "mae") else max)(sorted(self.evals), key=self.evals.get)
        return self._cands()[0]

    def _data_step(self, role: str, obs: dict | None):
        """Shared by the single agent and the data manager. Returns an action or None to continue."""
        st = self.phase.get(role, "start")
        if st == "start":
            self.phase[role] = "listed"
            return self._act("List the catalog.", "list_sources", {})
        if st == "listed" and obs and "tables" in obs:
            self.phase[role] = "materialize"
            return self._act("Build the feature graph from the contract.", "materialize_features", {"graph": self._graph()})
        if st == "materialize" and obs is not None and ("materialized" in obs or "repairs_used" in obs):
            if obs.get("resolution_failure"):
                r = obs["resolution_failure"]
                if role == "data_manager":
                    return self._act("Authority cannot be resolved.", "report",
                                     {"summary": str(obs.get("error", ""))[:200], "handle": None, "problem": r})
                return self._act("Authority cannot be resolved.", "abstain", {"reason": r, "detail": str(obs.get("error", ""))[:200]})
            if "repair limit" in str(obs.get("error", "")):
                self.phase[role] = "data_done"
                return None
            if obs.get("materialized"):
                self.handle = obs["handle"]
            bad = [v for v in (obs.get("gate_verdicts") or []) if not v["passed"]]
            if (bad or not obs.get("materialized")) and self._apply_feedback(obs):
                return self._act("Repair the graph using the feedback.", "materialize_features", {"graph": self._graph()})
            if not obs.get("materialized") and self.handle is None:
                if role == "data_manager":
                    return self._act("No valid handle.", "report", {"summary": str(obs.get("error", ""))[:200], "handle": None,
                                                                   "problem": "CONTRACT_UNSATISFIABLE"})
                return self._act("No valid handle.", "abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": str(obs.get("error", ""))[:200]})
            self.phase[role] = "data_done"
            return None
        return None

    def decide(self, messages: list, role: str) -> dict:
        self._contract(messages)
        obs = self._last_obs(messages)
        if obs and isinstance(obs.get("tables"), list):
            self.catalog_tables = {t["table"]: {"grain": t["grain"], "columns": {c: "x" for c in t["columns"]}}
                                   for t in obs["tables"]}
        if role == "coordinator":
            if obs and obs.get("report_from") == "data_manager":
                if obs.get("problem"):
                    reason = obs["problem"] if obs["problem"] in ("REQUIRED_SOURCE_MISSING", "AMBIGUOUS_AUTHORITY") else "CONTRACT_UNSATISFIABLE"
                    return self._act("The data cannot satisfy the contract.", "abstain", {"reason": reason, "detail": obs.get("summary", "")})
                self.handle = obs.get("handle") or self.handle
                self.phase["miner_attempts"] = 0
                self.evals = {}
                return self._act("Features ready; ask the miner.", "delegate",
                                 {"role": "miner", "instruction": f"Evaluate candidates on {self.handle} and recommend one."})
            if obs and obs.get("report_from") == "miner":
                if obs.get("candidate") is None:
                    if self.repairs >= 2:
                        return self._act("Modeling keeps failing.", "abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "no valid model"})
                    self.repairs += 1
                    self.phase["data_manager"] = "repair"
                    return self._act("Modeling failed; repair features.", "delegate",
                                     {"role": "data_manager", "instruction": "Repair the feature graph: " + str(obs.get("summary"))[:300]})
                return self._act("Submit the recommended model.", "submit",
                                 {"handle": obs.get("handle") or self.handle, "candidate": obs["candidate"],
                                  "findings": [{"type": "summary", "detail": str(obs.get("summary", ""))[:100]}]})
            if obs and obs.get("submitted") is False:
                if self._apply_feedback(obs) and self.repairs < 2:
                    self.repairs += 1
                    self.phase["data_manager"] = "repair"
                    self.phase["miner_attempts"] = 0
                    return self._act("Gate rejected the submission; repair.", "delegate",
                                     {"role": "data_manager", "instruction": "Repair the graph using: " + str(obs)[:300]})
                return self._act("Gates cannot be satisfied.", "abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "final gates failed"})
            return self._act("Start with data preparation.", "delegate",
                             {"role": "data_manager", "instruction": "Discover sources and materialize a contract-compliant handle."})
        if role == "data_manager":
            if self.phase.get("data_manager") == "repair":
                self.phase["data_manager"] = "materialize"
                return self._act("Rebuild the graph.", "materialize_features", {"graph": self._graph()})
            a = self._data_step(role, obs)
            if a:
                return a
            return self._act("Report the handle.", "report", {"summary": f"materialized {self.handle}", "handle": self.handle, "problem": None})
        if role == "single":
            if self.phase.get("single") != "data_done":
                a = self._data_step(role, obs)
                if a:
                    return a
                if self.phase.get("single") != "data_done":
                    return self._act("Continue.", "list_sources", {})
            if obs and obs.get("submitted") is False:
                if self._apply_feedback(obs):
                    self.phase["single"] = "materialize"
                    self.evals = {}
                    self.phase["single_attempts"] = 0
                    return self._act("Repair after gate rejection.", "materialize_features", {"graph": self._graph()})
                return self._act("Gates cannot be satisfied.", "abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "final gates failed"})
        if role in ("single", "miner"):
            if role == "miner":
                for m in reversed(messages):
                    if m["role"] == "user" and not m["content"].startswith("Observation"):
                        h = re.findall(r"\bH\d+\b", m["content"])
                        if h:
                            self.handle = h[-1]
                        break
            if obs and obs.get("candidate") and obs.get("validation") is not None:
                self.evals[obs["candidate"]] = obs["validation"]
            if obs and "non-numeric" in str(obs.get("error", "")):
                if role == "single" and self._apply_feedback(obs):
                    self.phase["single"] = "materialize"
                    return self._act("Parse numeric columns.", "materialize_features", {"graph": self._graph()})
                if role == "miner":
                    self._apply_feedback(obs)
                    return self._act("Features are not numeric.", "report", {"summary": str(obs["error"])[:200], "handle": self.handle, "candidate": None})
            plan = [c for c in self._cands() if c[-1] in "351"]
            plan = sorted(plan, key=lambda c: "351".index(c[-1]))
            key = role + "_attempts"
            todo = [c for c in plan if c not in self.evals]
            if todo and self.phase.get(key, 0) < 3:
                self.phase[key] = self.phase.get(key, 0) + 1
                return self._act("Evaluate a candidate.", "fit_evaluate", {"handle": self.handle, "candidate": todo[0]})
            best = self._best()
            if role == "miner":
                return self._act("Recommend the best candidate.", "report",
                                 {"summary": f"best {best} {self.evals.get(best)}", "handle": self.handle, "candidate": best if self.evals else None})
            return self._act("Submit the best candidate.", "submit",
                             {"handle": self.handle, "candidate": best, "findings": [{"type": "summary", "detail": "mock"}]})
        return self._act("Fallback.", "list_sources", {})

    def complete(self, messages: list, role: str, seed: int) -> dict:
        prompt_chars = sum(len(m["content"]) for m in messages)
        if self.rng.random() < self.invalid_rate:
            text = '{"thought": "unterminated'
        else:
            text = dumps(self.decide(messages, role))
        return {"text": text, "input_tokens": max(1, int(prompt_chars / 3.2)), "output_tokens": estimate_tokens(text),
                "seconds": 0.0, "finish_reason": "stop", "cached_tokens": 0}


### Agent state machines: F, single agent, three-role team

In [ ]:
%%writefile adm_code/adm/agents.py
"""Agent state machines: fixed workflow F, single agent (S0/SQP) and three-role team (M0/MQ/MP/MQP)."""
from __future__ import annotations

import json
import time
from pathlib import Path

from . import modeling
from .core import dumps
from .fixed import contract_graph
from .llm import ROLE_ACTIONS, LLMError
from .tools import BudgetExceeded, Terminal, ToolSession, _truncate

PROMPT_DIR = Path(__file__).parent / "prompts"
WORKER_STEP_CAP = 12
MAX_CONSECUTIVE_INVALID = 3
CHARS_PER_TOKEN = 2.8  # conservative for JSON-heavy text; overflows are also handled by re-trimming


def load_prompt(name: str) -> str:
    return (PROMPT_DIR / f"{name}.md").read_text()


def est_tokens(s: str) -> int:
    return int(len(s) / CHARS_PER_TOKEN) + 4


class InfrastructureFailure(Exception):
    pass


class Conversation:
    """Role-specific history with deterministic trimming to the context window."""

    def __init__(self, system: str, task: str):
        self.system, self.task = system, task
        self.entries: list = []   # alternating assistant/user dicts with 'summary'

    def add_assistant(self, text: str, summary: str):
        self.entries.append({"role": "assistant", "content": text, "summary": summary})

    def add_user(self, text: str):
        if self.entries and self.entries[-1]["role"] == "user":
            self.entries[-1]["content"] += "\n\n" + text
        elif not self.entries:
            self.task = self.task + "\n\n" + text
        else:
            self.entries.append({"role": "user", "content": text})

    def render(self, max_prompt_tokens: int) -> list:
        fixed = est_tokens(self.system) + est_tokens(self.task) + 200
        budget = max_prompt_tokens - fixed
        pairs = []
        i = 0
        while i < len(self.entries):
            a = self.entries[i]
            u = self.entries[i + 1] if i + 1 < len(self.entries) else None
            pairs.append((a, u))
            i += 2
        keep = []
        for a, u in reversed(pairs):
            cost = est_tokens(a["content"]) + (est_tokens(u["content"]) if u else 0)
            if cost > budget and keep:
                break
            if cost > budget and not keep:
                # always keep the latest turn, truncating it if needed
                u = dict(u) if u else None
                if u:
                    u["content"] = u["content"][: max(400, int(budget * CHARS_PER_TOKEN) - len(a["content"]))]
                keep.insert(0, (a, u))
                break
            keep.insert(0, (a, u))
            budget -= cost
        omitted = pairs[: len(pairs) - len(keep)]
        task = self.task
        if omitted:
            task += "\n\nEarlier steps (details omitted to fit the context window):\n" + "\n".join(
                f"- {a['summary']}" for a, _ in omitted)
        msgs = [{"role": "system", "content": self.system}, {"role": "user", "content": task}]
        for a, u in keep:
            msgs.append({"role": "assistant", "content": a["content"]})
            if u:
                msgs.append({"role": "user", "content": u["content"]})
        return msgs


def parse_action(text: str, role: str) -> tuple[dict | None, str | None]:
    s = (text or "").strip()
    obj = None
    try:
        obj = json.loads(s)
    except Exception:
        a, b = s.find("{"), s.rfind("}")
        if a >= 0 and b > a:
            try:
                obj = json.loads(s[a:b + 1])
            except Exception:
                obj = None
    if not isinstance(obj, dict):
        return None, "response is not a JSON object"
    act = obj.get("action")
    if act not in ROLE_ACTIONS[role]:
        return None, f"action must be one of {ROLE_ACTIONS[role]}"
    args = obj.get("args", {})
    if not isinstance(args, dict):
        return None, "args must be an object"
    return {"thought": str(obj.get("thought", ""))[:600], "action": act, "args": args}, None


class AgentRun:
    def __init__(self, session: ToolSession, llm, system_code: str, spec: dict, generation: dict, agent_seed: int,
                 trace: list):
        self.s = session
        self.llm = llm
        self.code = system_code
        self.spec = spec
        self.gen = generation
        self.seed = agent_seed
        self.trace = trace
        self.calls = 0
        self.max_prompt = generation["max_context_tokens"] - generation["max_output_tokens_per_call"] - 256
        c = session.contract
        contract_md = (session.visible / "contract.md").read_text()
        gates_line = (f"Configuration: quality gates are {'ON' if spec['q'] else 'OFF'}; provenance gates are "
                      f"{'ON' if spec['p'] else 'OFF'}. When a gate is ON, verdicts are attached automatically to "
                      "materialize, fit and submit observations.")
        self.task_text = (f"{contract_md}\n{gates_line}\n\nMachine-readable contract:\n<contract_json>{dumps(c)}</contract_json>")
        self.common = load_prompt("common")

    # ---------------------------------------------------------------- model calls
    def ask(self, role: str, conv: Conversation) -> dict:
        invalid = 0
        while True:
            b = self.s.budget
            if b.model_calls + 1 > b.limits["model_calls"]:
                raise BudgetExceeded("model_calls")
            if b.active_seconds() > b.limits["wall_seconds"]:
                raise BudgetExceeded("wall_seconds")
            seed = (self.seed + self.calls) % (2**31 - 1)
            res = None
            prompt_budget = self.max_prompt
            failures = 0
            while res is None:
                msgs = conv.render(prompt_budget)
                try:
                    res = self.llm.complete(msgs, role, seed)
                except LLMError as e:
                    self.trace.append({"event": "llm_error", "role": role, "error": str(e)[:300], "attempt": failures,
                                       "prompt_budget": prompt_budget})
                    if e.context_overflow and prompt_budget > 3000:
                        prompt_budget = int(prompt_budget * 0.8)  # the token estimate was too low: trim further
                        continue
                    failures += 1
                    if failures >= 3:
                        raise InfrastructureFailure(str(e))
                    time.sleep(5 * failures)
            self.calls += 1
            b.model_calls += 1
            b.tokens_in += res["input_tokens"]
            b.tokens_out += res["output_tokens"]
            b.llm_seconds += res["seconds"]
            act, err = parse_action(res["text"], role)
            self.trace.append({"event": "model", "role": role, "call": self.calls, "input_tokens": res["input_tokens"],
                               "output_tokens": res["output_tokens"], "cached_tokens": res.get("cached_tokens", 0),
                               "seconds": round(res["seconds"], 3), "finish_reason": res.get("finish_reason"),
                               "text": res["text"][:4000], "parse_error": err})
            b.check()
            if act is not None:
                return act
            invalid += 1
            conv.add_assistant(res["text"][:300], f"invalid response ({err})")
            conv.add_user("Observation (format_error):\n" + dumps({"error": err, "hint": "Reply with one JSON object "
                                                                   '{"thought": ..., "action": ..., "args": {...}}'}))
            if invalid >= MAX_CONSECUTIVE_INVALID:
                raise _ExecutionError(f"{invalid} consecutive invalid responses")

    def tool(self, role: str, act: dict, conv: Conversation) -> dict:
        name = act["action"]
        obs = self.s.call(name, act["args"])
        self.trace.append({"event": "tool", "role": role, "tool": name, "args": _slim_args(act["args"]),
                           "ok": obs.get("ok"), "obs": _truncate(obs, 2500)})
        conv.add_assistant(dumps(act), f"{role}: {name}({_short(act['args'])}) -> {'ok' if obs.get('ok') else 'error'}"
                           + (f" handle={obs.get('handle')}" if obs.get("handle") else "")
                           + (f" val={obs.get('validation')}" if obs.get("validation") is not None else ""))
        conv.add_user(f"Observation ({name}) | remaining budget {dumps(self.s.budget.remaining())}:\n" + _truncate(obs))
        return obs

    # ---------------------------------------------------------------- single agent
    def run_single(self):
        conv = Conversation(self.common + "\n" + load_prompt("single"), self.task_text)
        while True:
            act = self.ask("single", conv)
            self.tool("single", act, conv)

    # ---------------------------------------------------------------- multi agent
    def run_multi(self):
        coord = Conversation(self.common + "\n" + load_prompt("coordinator"), self.task_text)
        workers = {r: Conversation(self.common + "\n" + load_prompt(r), self.task_text) for r in ("data_manager", "miner")}
        board = {"handles": {}, "evaluations": {}, "reports": []}
        while True:
            act = self.ask("coordinator", coord)
            if act["action"] == "delegate":
                role = act["args"].get("role")
                instr = str(act["args"].get("instruction", ""))[:1500]
                if role not in workers:
                    obs = {"ok": False, "error": "role must be data_manager or miner"}
                    coord.add_assistant(dumps(act), "coordinator: invalid delegate")
                    coord.add_user("Observation (delegate):\n" + dumps(obs))
                    continue
                self.trace.append({"event": "delegate", "role": role, "instruction": instr[:500]})
                report = self.run_worker(role, workers[role], instr, board)
                board["reports"].append({"from": role, **{k: report.get(k) for k in ("summary", "handle", "candidate", "problem")}})
                coord.add_assistant(dumps(act), f"coordinator: delegate {role} -> report handle={report.get('handle')} "
                                                f"candidate={report.get('candidate')} problem={report.get('problem')}")
                coord.add_user(f"Observation (report from {role}) | remaining budget {dumps(self.s.budget.remaining())}:\n"
                               + _truncate({"report_from": role, **report, "blackboard": _board(board)}))
            else:
                obs = self.tool("coordinator", act, coord)
                _update_board(board, act, obs)

    def run_worker(self, role: str, conv: Conversation, instruction: str, board: dict) -> dict:
        conv.add_user(f"Instruction from coordinator: {instruction}\nShared blackboard: {dumps(_board(board))}")
        for _ in range(WORKER_STEP_CAP):
            act = self.ask(role, conv)
            if act["action"] == "report":
                a = act["args"]
                rep = {"summary": str(a.get("summary", ""))[:600], "handle": a.get("handle"),
                       "candidate": a.get("candidate"), "problem": a.get("problem")}
                conv.add_assistant(dumps(act), f"{role}: report")
                return rep
            obs = self.tool(role, act, conv)
            _update_board(board, act, obs)
        return {"summary": f"{role} reached its {WORKER_STEP_CAP}-step delegation cap", "handle": None,
                "candidate": None, "problem": None}


class _ExecutionError(Exception):
    pass


def _short(args: dict) -> str:
    s = dumps({k: (v if k != "graph" else f"<graph {len(v.get('nodes', [])) if isinstance(v, dict) else '?'} nodes>")
               for k, v in (args or {}).items()})
    return s[:160]


def _slim_args(args: dict) -> dict:
    return json.loads(dumps(args))


def _board(board: dict) -> dict:
    return {"handles": board["handles"], "evaluations": board["evaluations"], "last_reports": board["reports"][-3:]}


def _update_board(board: dict, act: dict, obs: dict):
    if act["action"] == "materialize_features" and obs.get("handle"):
        gv = obs.get("gate_verdicts")
        board["handles"][obs["handle"]] = {"rows": {p: v.get("rows") for p, v in obs.get("summary", {}).items()},
                                           "gates_passed": (all(g["passed"] for g in gv) if gv else None)}
    if act["action"] == "fit_evaluate" and obs.get("candidate"):
        board["evaluations"][f"{act['args'].get('handle')}/{obs['candidate']}"] = obs.get("validation")


# --------------------------------------------------------------------------- drivers
def run_fixed(session: ToolSession, trace: list):
    c, cat = session.contract, session.catalog
    graph = contract_graph(c, cat)
    obs = session.call("materialize_features", {"graph": graph})
    trace.append({"event": "tool", "role": "F", "tool": "materialize_features", "ok": obs.get("ok"), "obs": _truncate(obs, 2500)})
    if not obs.get("materialized"):
        reason = obs.get("resolution_failure") or "CONTRACT_UNSATISFIABLE"
        session.call("abstain", {"reason": reason, "detail": str(obs.get("error", ""))[:200]})
    if any(not g["passed"] for g in obs.get("gate_verdicts", [])):
        session.call("abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "fixed workflow gates failed"})
    h = obs["handle"]
    results = []
    for cid in c["candidate_models"]:
        r = session.call("fit_evaluate", {"handle": h, "candidate": cid})
        trace.append({"event": "tool", "role": "F", "tool": "fit_evaluate", "ok": r.get("ok"), "obs": _truncate(r, 800)})
        if r.get("ok"):
            results.append(r)
    best = modeling.select_best(results, c["selection_metric"])
    if best is None:
        session.call("abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "no candidate could be evaluated"})
    r = session.call("submit", {"handle": h, "candidate": best, "findings": [{"type": "fixed_workflow", "detail": "contract-driven"}]})
    trace.append({"event": "tool", "role": "F", "tool": "submit", "ok": r.get("ok"), "obs": _truncate(r, 1500)})
    session.call("abstain", {"reason": "CONTRACT_UNSATISFIABLE", "detail": "submission rejected"})


def execute_run(session: ToolSession, system_code: str, spec: dict, llm, generation: dict, agent_seed: int) -> dict:
    """Run one system on one task instance until a terminal state."""
    trace: list = []
    out = {"terminal_state": None, "abstention_reason": None, "error": None, "submission": None}
    try:
        if spec["planner"] == "fixed":
            run_fixed(session, trace)
        else:
            ar = AgentRun(session, llm, system_code, spec, generation, agent_seed, trace)
            ar.run_single() if spec["planner"] == "single" else ar.run_multi()
    except Terminal as t:
        out["terminal_state"] = t.state
        if t.state == "ABSTAINED":
            out["abstention_reason"] = t.info["reason"]
        else:
            out["submission"] = t.info
    except BudgetExceeded as e:
        out["terminal_state"] = "BUDGET_EXCEEDED"
        out["error"] = f"limit exceeded: {e}"
    except InfrastructureFailure as e:
        out["terminal_state"] = "INFRASTRUCTURE_ERROR"
        out["error"] = str(e)[:500]
    except _ExecutionError as e:
        out["terminal_state"] = "EXECUTION_ERROR"
        out["error"] = str(e)[:500]
    except Exception as e:  # unexpected failure inside the agent loop is an execution error of the run
        import traceback
        out["terminal_state"] = "EXECUTION_ERROR"
        out["error"] = f"{type(e).__name__}: {e}"[:300]
        trace.append({"event": "exception", "traceback": traceback.format_exc()[-3000:]})
    finally:
        session.close()
    out["trace"] = trace
    out["blocked_attempts"] = session.blocked_attempts
    out["gate_log"] = [{"gate": g["gate"], "artifact": g["artifact"], "stage": g.get("stage", "materialize"),
                        "passed": g["passed"], "failures": [f["check"] for f in g["failures"]]} for g in session.gate_log]
    return out


### Runner: manifests, worker processes, retries, resumable records

In [ ]:
%%writefile adm_code/adm/runner.py
"""Experiment manifests, scheduling, per-run worker processes, evaluation and resumable records."""
from __future__ import annotations

import gzip
import json
import multiprocessing as mp
import os
import random
import resource
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from .core import (AGENTIC_SYSTEMS, CONDITIONS, FAMILIES, NEGATIVE_CONDITIONS, REPLICATION_SYSTEMS,
                   REPLICATION_TASKS, SYSTEM_SPEC, SYSTEMS, TEMPLATES, TIER2_CONDITIONS, TIER2_DESIGN, Paths, SessionClock,
                   derive_seed, dumps,
                   instance_dirs, now_iso, read_json, write_json, write_json_once, sha256_tree, json_hash, small_seed)

THREAD_VARS = ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS",
               "VECLIB_MAXIMUM_THREADS")


def set_thread_caps(n: int = 1) -> None:
    """Numerical-library threads are capped consistently for every worker (Section 10.2)."""
    for v in THREAD_VARS:
        os.environ[v] = str(n)


# --------------------------------------------------------------------------- manifests
def run_id(r: dict) -> str:
    bid = f"-{r['budget_profile']}" if r.get("budget_profile", "standard") != "standard" else ""
    return f"{r['experiment']}-{r['family']}-{r['template']}-{r['condition']}-r{r['repetition']}-{r['system']}-{r['backbone']}{bid}"


def _block(exp, suite, fam, tmpl, cond, rep, systems, backbone, budget="standard"):
    rng = random.Random(derive_seed("agent", fam, f"{fam}-{tmpl}", rep, f"order:{exp}:{cond}:{budget}"))
    order = list(systems)
    rng.shuffle(order)
    out = []
    for k, s in enumerate(order):
        r = {"experiment": exp, "suite": suite, "family": fam, "template": tmpl, "condition": cond, "repetition": rep,
             "system": s, "backbone": backbone if s != "F" else "none", "block": f"{exp}/{fam}-{tmpl}/{cond}/r{rep}/{budget}",
             "order_in_block": k, "budget_profile": budget}
        r["run_id"] = run_id(r)
        out.append(r)
    return out


def experiment_runs(exp: str, repetitions: list, rehearsal: dict | None = None, tier2: list | None = None) -> list:
    """Run lists for each experiment (Section 12). ``rehearsal`` restricts the design for local testing.
    ``tier2`` lists the frozen Tier-2 conditions (main-plan-v2.1 Section 7.6) added to E1 and E4."""
    fams, tmpls, conds = FAMILIES, TEMPLATES, CONDITIONS
    if rehearsal:
        fams, tmpls, conds = rehearsal.get("families", fams), rehearsal.get("templates", tmpls), rehearsal.get("conditions", conds)
    conds = list(conds) + [c for c in (tier2 or []) if c not in conds]
    runs = []
    if exp == "E0S":  # supplementary pilot (recorded separately, Section 12 E0): development tasks under C1-C5
        for fam in ["D5", "D6"]:
            for t in TEMPLATES:
                for c in ["C1", "C2", "C3", "C4", "C5"]:
                    for r in repetitions:
                        runs += _block("E0S", "development", fam, t, c, r, SYSTEMS, "A")
        return runs
    if exp == "E0X":  # Tier-2 calibration pilot: development tasks, X1-X3 at three levels, one repetition, backbone A.
        # Template is the outer loop so that a partial session still covers every defect and level evenly.
        for t in TEMPLATES:
            for fam in ["D5", "D6"]:
                for c in TIER2_CONDITIONS:
                    for r in repetitions:
                        runs += _block("E0X", "development", fam, t, c, r, SYSTEMS, "A")
        return runs
    if exp == "E0X2":  # second Tier-2 calibration (X2, X3, X4 at three levels), backbone A, repetitions 1 and 2
        for t in TEMPLATES:
            for r in repetitions:
                for fam in ["D5", "D6"]:
                    for c in TIER2_DESIGN:
                        runs += _block("E0X2", "development", fam, t, c, r, SYSTEMS, "A")
        return runs
    if exp == "E0SB":  # supplementary pilot, backbone B: development tasks, C0-C6, replication systems (E4 headroom)
        for fam in ["D5", "D6"]:
            for t in TEMPLATES:
                for c in CONDITIONS:
                    for r in repetitions:
                        runs += _block("E0SB", "development", fam, t, c, r, REPLICATION_SYSTEMS, "B")
        return runs
    if exp == "E0B":  # unscored backbone-B smoke test during the pilot
        for fam, t, c in [("D5", "T2", "C0"), ("D6", "T4", "C6")]:
            runs += _block("E0B", "development", fam, t, c, 0, REPLICATION_SYSTEMS, "B")
        return runs
    if exp == "E0":
        for fam in ["D5", "D6"]:
            for t in TEMPLATES if not rehearsal else tmpls:
                for c in ["C0", "C6"]:
                    for r in [0, 1]:
                        runs += _block("E0", "development", fam, t, c, r, SYSTEMS, "A")
    elif exp == "E1":
        for r in repetitions:
            for fam in fams:
                for t in tmpls:
                    for c in conds:
                        runs += _block("E1", "evaluation", fam, t, c, r, SYSTEMS, "A")
    elif exp in ("E4", "E5", "E7"):
        tasks = [x for x in REPLICATION_TASKS if (x.split("-")[0] in fams and x.split("-")[1] in tmpls)] if rehearsal else REPLICATION_TASKS
        for r in repetitions:
            for tk in tasks:
                fam, t = tk.split("-")
                if exp == "E4":
                    for c in conds:
                        runs += _block("E4", "evaluation", fam, t, c, r, REPLICATION_SYSTEMS, "B")
                elif exp == "E5":
                    for c in NEGATIVE_CONDITIONS:
                        runs += _block("E5", "evaluation", fam, t, c, r, SYSTEMS, "A")
                else:
                    for c in ["C0", "C2", "C6"]:  # E7 budget sensitivity stays on Tier 1 (unchanged design)
                        if r > 1:
                            continue
                        for b in ["half", "double"]:
                            runs += _block("E7", "evaluation", fam, t, c, r, REPLICATION_SYSTEMS, "A", b)
    return runs


def budget_for(limits: dict, profile: str) -> dict:
    if profile == "standard":
        return dict(limits)
    f = 0.5 if profile == "half" else 2.0
    out = dict(limits)
    for k in ("aggregate_tokens", "model_calls", "tool_calls", "wall_seconds"):
        out[k] = int(round(limits[k] * f))
    return out  # pipeline_evaluations and repair_cycles unchanged


def run_seeds(r: dict) -> dict:
    tk = f"{r['family']}-{r['template']}"
    return {"corruption": derive_seed("corruption", r["family"], tk, r["repetition"], r["condition"]),
            "estimator": small_seed(derive_seed("estimator", r["family"], tk, r["repetition"], r["condition"])),
            "agent": small_seed(derive_seed("agent", r["family"], tk, r["repetition"], r["condition"])),
            "split": derive_seed("split", r["family"], r["suite"], r["repetition"], "outer")}


# --------------------------------------------------------------------------- worker
def worker_main(job_path: str, result_path: str, fit_slot) -> None:
    """Executed in a fresh spawned process for each run."""
    t0 = time.time()
    try:
        job = read_json(job_path)
        from .agents import execute_run
        from .core import Paths, instance_dirs
        from .llm import MockLLM, VLLMClient
        from .tools import Budget, ToolSession
        r = job["run"]
        paths = Paths.from_dict(job["paths"])
        dirs = instance_dirs(paths, r["suite"], r["family"], r["template"], r["condition"], r["repetition"])
        spec = SYSTEM_SPEC[r["system"]]
        budget = Budget(job["limits"])
        sub_dir = Path(job["submission_dir"])
        if sub_dir.exists():
            shutil.rmtree(sub_dir)
        session = ToolSession(r["run_id"], dirs["visible"], dirs["trusted"], sub_dir, spec, budget, job["seeds"],
                              fit_slot=fit_slot)
        llm = None
        if spec["planner"] != "fixed":
            if job["llm"]["mode"] == "mock":
                llm = MockLLM(job["seeds"]["agent"] + hash_system(r["system"]), invalid_rate=job["llm"].get("invalid_rate", 0.03))
            else:
                llm = VLLMClient(job["llm"]["base_url"], job["llm"]["served_model"], job["generation"],
                                 job["llm"].get("structured_mode", "response_format"))
        out = execute_run(session, r["system"], spec, llm, job["generation"], job["seeds"]["agent"])
        trace = out.pop("trace")
        tp = Path(job["trace_path"])
        tp.parent.mkdir(parents=True, exist_ok=True)
        with gzip.open(tp, "wt", encoding="utf-8") as f:
            for e in trace:
                f.write(dumps(e) + "\n")
        out["usage"] = budget.usage()
        out["exception_events"] = sum(1 for e in trace if e.get("event") == "exception")
        out["peak_worker_memory_mb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 if sys.platform != "darwin" else 1024 * 1024)
        out["worker_seconds"] = round(time.time() - t0, 3)
        write_json(result_path, out)
    except Exception:
        write_json(result_path, {"terminal_state": "INFRASTRUCTURE_ERROR", "error": traceback.format_exc()[-3000:],
                                 "usage": {}, "worker_crash": True})


def hash_system(code: str) -> int:
    # mock only: differentiate personas between systems deterministically
    return sum(ord(c) * (i + 1) for i, c in enumerate(code)) * 7919


# --------------------------------------------------------------------------- scheduler
class Runner:
    def __init__(self, paths: Paths, config: dict, clock: SessionClock, log=print):
        self.paths, self.cfg, self.clock, self.log = paths, config, clock, log
        set_thread_caps()
        self.ctx = mp.get_context("spawn")
        self.fit_slot = self.ctx.Semaphore(config.get("fit_slots", 3))
        self.jobs_dir = paths.work / "jobs"
        self.jobs_dir.mkdir(parents=True, exist_ok=True)

    def record_path(self, r: dict) -> Path:
        return self.paths.runs / r["experiment"] / f"{r['run_id']}.json"

    def attempts(self, r: dict) -> list:
        d = self.paths.attempts / r["experiment"]
        return sorted(d.glob(f"{r['run_id']}__a*.json")) if d.exists() else []

    def stale_markers(self, runs: list):
        """A start marker without a result means the previous session ended abruptly: record it as an
        infrastructure attempt so that it is retried at most once (Section 13.4)."""
        for r in runs:
            d = self.paths.attempts / r["experiment"]
            for m in (d.glob(f"{r['run_id']}__a*.started") if d.exists() else []):
                res = m.with_suffix(".json")
                if not res.exists() and not self.record_path(r).exists():
                    write_json(res, {"run_id": r["run_id"], "terminal_state": "INFRASTRUCTURE_ERROR",
                                     "error": "session ended while the run was in progress", "attempt": m.stem.split("__a")[-1]})
                m.unlink(missing_ok=True)

    def pending(self, runs: list) -> list:
        out = []
        for r in runs:
            if self.record_path(r).exists():
                continue
            out.append(r)
        return out

    def _job(self, r: dict, attempt: int) -> dict:
        llm = self.cfg["llm"]
        lim = budget_for(self.cfg["limits"], r.get("budget_profile", "standard"))
        return {"run": r, "paths": self.paths.to_dict(), "limits": lim, "generation": self.cfg["generation"],
                "seeds": run_seeds(r),
                "llm": {**llm, "served_model": llm.get("served_models", {}).get(r["backbone"], llm.get("served_model"))},
                "submission_dir": str(self.paths.work / "submissions" / r["run_id"]),
                "trace_path": str(self.paths.traces / r["experiment"] / f"{r['run_id']}__a{attempt}.jsonl.gz"),
                "attempt": attempt}

    def execute(self, runs: list, concurrency: int, evaluator_workers: int = 1, on_progress=None,
                retain_ids: set | None = None) -> dict:
        from .evaluator import evaluate_safe
        self.retain_ids = set(retain_ids or ())
        self.stale_markers(runs)
        todo = self.pending(runs)
        stats = {"scheduled": len(runs), "pending_at_start": len(todo), "completed": 0, "infra_attempts": 0,
                 "stopped_for_time": False}
        if not todo:
            return stats
        active: dict = {}
        queue = list(todo)
        eval_pool = ProcessPoolExecutor(max_workers=evaluator_workers, mp_context=self.ctx)
        eval_futs: dict = {}
        avg_dur = [self.cfg["limits"]["wall_seconds"] / 4]
        t_last = time.time()
        try:
            while queue or active or eval_futs:
                # dispatch
                while queue and len(active) < concurrency:
                    if not self.clock.may_start_new(min(avg_dur[0], self.cfg["limits"]["wall_seconds"])):
                        stats["stopped_for_time"] = True
                        queue = []
                        break
                    r = queue.pop(0)
                    att = len(self.attempts(r)) + 1
                    job = self._job(r, att)
                    jp = self.jobs_dir / f"{r['run_id']}__a{att}.job.json"
                    rp = self.jobs_dir / f"{r['run_id']}__a{att}.result.json"
                    rp.unlink(missing_ok=True)
                    write_json(jp, job)
                    marker = self.paths.attempts / r["experiment"] / f"{r['run_id']}__a{att}.started"
                    marker.parent.mkdir(parents=True, exist_ok=True)
                    marker.write_text(now_iso())
                    p = self.ctx.Process(target=worker_main, args=(str(jp), str(rp), self.fit_slot), daemon=True)
                    p.start()
                    active[r["run_id"]] = {"run": r, "proc": p, "start": time.time(), "attempt": att, "job": job,
                                           "result": rp, "marker": marker, "queued_at": now_iso()}
                # collect finished runs
                for rid in list(active):
                    a = active[rid]
                    p = a["proc"]
                    hard = a["job"]["limits"]["wall_seconds"] * 1.5 + 180
                    killed = None
                    if p.is_alive():
                        if time.time() - a["start"] > hard:
                            killed = "run exceeded the hard wall-clock guard"
                        elif self.clock.must_stop():
                            killed = "session time limit reached"
                        if killed:
                            p.kill()
                            p.join(10)
                        else:
                            continue
                    p.join(1)
                    del active[rid]
                    res = read_json(a["result"]) if a["result"].exists() else {
                        "terminal_state": "INFRASTRUCTURE_ERROR", "error": killed or "worker exited without a result", "usage": {}}
                    if killed and res.get("terminal_state") != "INFRASTRUCTURE_ERROR":
                        res = {"terminal_state": "INFRASTRUCTURE_ERROR", "error": killed, "usage": res.get("usage", {})}
                    dur = time.time() - a["start"]
                    avg_dur[0] = 0.8 * avg_dur[0] + 0.2 * dur
                    res["wall_including_queue_seconds"] = round(dur, 3)
                    self._finish(a, res, eval_pool, eval_futs, evaluate_safe, stats, queue)
                # collect evaluations
                for rid in list(eval_futs):
                    fut, a, res = eval_futs[rid]
                    if fut.done():
                        del eval_futs[rid]
                        try:
                            ev = fut.result()
                        except Exception as e:
                            ev = {"run_id": rid, "external_failure_codes": ["EVALUATOR_ERROR"], "evaluator_error": str(e)[:500],
                                  "verified_completion": 0}
                        self._write_record(a, res, ev)
                        stats["completed"] += 1
                if on_progress and time.time() - t_last > 30:
                    t_last = time.time()
                    on_progress(stats, len(queue), len(active), len(eval_futs))
                if self.clock.must_stop() and not active:
                    # leave remaining evaluations to complete quickly
                    pass
                time.sleep(0.2)
        finally:
            eval_pool.shutdown(wait=True, cancel_futures=False)
        return stats

    def _finish(self, a, res, eval_pool, eval_futs, evaluate_safe, stats, queue):
        r = a["run"]
        a["marker"].unlink(missing_ok=True)
        if res.get("terminal_state") == "INFRASTRUCTURE_ERROR":
            stats["infra_attempts"] += 1
            att_path = self.paths.attempts / r["experiment"] / f"{r['run_id']}__a{a['attempt']}.json"
            write_json(att_path, {"run_id": r["run_id"], "attempt": a["attempt"], **{k: v for k, v in res.items() if k != "gate_log"}})
            n_att = len(self.attempts(r))
            if n_att < 2 and not self.clock.must_stop():
                queue.insert(0, r)  # one automatic retry with the same frozen inputs
                return
            if n_att < 2:
                return  # retried next session
        dirs = instance_dirs(self.paths, r["suite"], r["family"], r["template"], r["condition"], r["repetition"])
        neg = None
        if r["condition"] in NEGATIVE_CONDITIONS:
            neg = "REQUIRED_SOURCE_MISSING" if r["condition"] == "N1" else "AMBIGUOUS_AUTHORITY"
        job = {"run_id": r["run_id"], "terminal_state": res.get("terminal_state"),
               "abstention_reason": res.get("abstention_reason"),
               "instance": {k: str(v) for k, v in dirs.items()}, "submission_dir": a["job"]["submission_dir"],
               "usage": res.get("usage", {}), "limits": a["job"]["limits"], "negative": neg,
               "retain_model": r["run_id"] in getattr(self, "retain_ids", set())}
        eval_futs[r["run_id"]] = (eval_pool.submit(evaluate_safe, job), a, res)

    def _write_record(self, a: dict, res: dict, ev: dict):
        r = a["run"]
        job = a["job"]
        u = res.get("usage", {}) or {}
        llm = self.cfg["llm"]
        model_lock = self.cfg.get("model_lock", {}).get(r["backbone"], {})
        rec = {
            "run_id": r["run_id"], "protocol_hash": self.cfg.get("protocol_hash"), "code_hash": self.cfg.get("code_hash"),
            "experiment_id": r["experiment"], "suite": r["suite"], "dataset_id": r["family"],
            "task_id": f"{r['family']}-{r['template']}", "template_id": r["template"], "condition_id": r["condition"],
            "repetition_id": r["repetition"], "system_id": r["system"], "backbone_id": r["backbone"],
            "budget_profile": r.get("budget_profile", "standard"), "block": r["block"], "order_in_block": r["order_in_block"],
            "llm_mode": llm["mode"], "model_revision": model_lock.get("revision"), "model_repo": model_lock.get("repo"),
            "tokenizer_hash": model_lock.get("tokenizer_hash"),
            "data_hashes": self.cfg.get("instance_hashes", {}).get(f"{r['suite']}/{r['family']}-{r['template']}/{r['condition']}/r{r['repetition']}"),
            "split_hash": self.cfg.get("split_hashes", {}).get(f"{r['suite']}/{r['family']}/r{r['repetition']}", {}).get("hash"),
            "corruption_seed": job["seeds"]["corruption"], "estimator_seed": job["seeds"]["estimator"],
            "agent_seed": job["seeds"]["agent"], "start_time": a["queued_at"], "attempt": a["attempt"],
            "infrastructure_retry_count": a["attempt"] - 1,
            "terminal_state": res.get("terminal_state"), "abstention_reason": res.get("abstention_reason"),
            "run_error": res.get("error"), "exception_events": res.get("exception_events", 0),
            "verified_completion": ev.get("verified_completion"), "external_failure_codes": ev.get("external_failure_codes"),
            "first_failure": ev.get("first_failure"),
            "correct_refusal": ev.get("correct_refusal"), "false_submission": ev.get("false_submission"),
            "selected_candidate": (res.get("submission") or {}).get("candidate"),
            "test_metrics": ev.get("test_metrics"), "test_prediction_coverage": ev.get("test_prediction_coverage"),
            "test_value_match_rate": ev.get("test_value_match_rate"), "record_integrity_rate": ev.get("record_integrity_rate"),
            "data_management": ev.get("data_management"), "version_selection_correct": ev.get("version_selection_correct"),
            "input_tokens": u.get("input_tokens"), "output_tokens": u.get("output_tokens"),
            "aggregate_tokens": u.get("aggregate_tokens"), "model_calls": u.get("model_calls"),
            "tool_calls": u.get("tool_calls"), "gate_calls": u.get("gate_calls"), "repair_cycles": u.get("repair_cycles"),
            "pipeline_evaluations": u.get("pipeline_evaluations"), "fit_count": u.get("fit_count"),
            "active_seconds": u.get("active_seconds"), "queue_seconds": u.get("wait_seconds"),
            "llm_seconds": u.get("llm_seconds"), "wall_including_queue_seconds": res.get("wall_including_queue_seconds"),
            "peak_worker_memory_mb": res.get("peak_worker_memory_mb"),
            "server_gpu_seconds": None, "server_gpu_seconds_reason": "shared server; see per-experiment GPU accounting",
            "monetary_cost": None, "monetary_cost_reason": f"{self.cfg.get('mode', 'kaggle')} compute has no per-run tariff; GPU-seconds reported instead",
            "replay_status": ev.get("replay_status"), "evaluator_seconds": ev.get("evaluator_seconds"),
            "gate_log": res.get("gate_log"), "limits": job["limits"],
            "predicates": ev.get("predicates"), "evaluator_error": ev.get("evaluator_error"),
            "trace_path": str(Path(job["trace_path"]).relative_to(self.paths.state)),
        }
        # persist the small submission artifacts; the pickled model is deleted after evaluation
        sub = Path(job["submission_dir"])
        keep = self.paths.submissions / r["experiment"] / r["run_id"]
        if sub.exists():
            keep.mkdir(parents=True, exist_ok=True)
            for fn in ("bundle.json", "graph.json", "model_choice.json", "findings.json", "trusted_provenance.jsonl",
                       "test_predictions.parquet", "model.pkl"):
                if (sub / fn).exists():
                    shutil.copy2(sub / fn, keep / fn)
            shutil.rmtree(sub, ignore_errors=True)
        write_json_once(self.record_path(r), rec)


def load_records(paths: Paths, experiment: str | None = None) -> list:
    out = []
    roots = [paths.runs / experiment] if experiment else sorted(paths.runs.glob("*"))
    for d in roots:
        if d.is_dir():
            for p in sorted(d.glob("*.json")):
                out.append(read_json(p))
    return out


### Clean-data reference workflows and dummy models

In [ ]:
%%writefile adm_code/adm/references.py
"""Clean-data reference workflows and dummy models (Section 10.4). Trusted, evaluator-side computations;
agents never see these outputs."""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd

from . import modeling
from .core import Paths, instance_dirs, read_json, write_json, derive_seed, small_seed
from .datasets import cv_splits
from .tasks import ensure_instance


def _labels(dirs) -> tuple[pd.DataFrame, pd.DataFrame]:
    lab = pd.read_parquet(dirs["trusted"] / "labels.parquet")
    test = pd.read_parquet(dirs["private"] / "labels_test.parquet")
    return lab, test


def reference_workflow(paths: Paths, suite: str, family: str, template: str, rep: int) -> dict:
    ensure_instance(paths, suite, family, template, "C0", rep)
    dirs = instance_dirs(paths, suite, family, template, "C0", rep)
    contract = read_json(dirs["visible"] / "contract.json")
    rule = read_json(dirs["visible"] / "visible_partition_manifest.json")["cv_policy"]["rule"]
    refs = {p: pd.read_parquet(dirs["private"] / f"reference_{p}.parquet") for p in ("train", "validation", "test")}
    lab, test = _labels(dirs)
    cv = cv_splits(lab, rule)
    seed = small_seed(derive_seed("estimator", family, f"{family}-{template}", rep, "C0"))
    tt = contract["task_type"]
    results = [modeling.fit_evaluate(c, refs["train"], refs["validation"], lab, cv, contract, seed)
               for c in contract["candidate_models"]]
    best = modeling.select_best(results, contract["selection_metric"])
    model, info = modeling.refit(best, refs["train"], refs["validation"], lab, contract, seed)
    X = refs["test"].merge(test, on="observation_id")
    feats = info["numeric"] + info["categorical"]
    pred = modeling.predict_scores(model, X[feats], tt)
    metrics = modeling.full_metrics(tt, X["target"].to_numpy(), pred)
    # dummy model on the correct canonical workflow
    trv = lab[lab["partition"].isin(["train", "validation"])]
    dm = modeling.dummy_model(tt)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dm.fit(np.zeros((len(trv), 1)), trv["target"].to_numpy())
    Z = np.zeros((len(X), 1))
    if tt == "regression":
        dpred = dm.predict(Z)
    elif tt == "binary":
        classes = list(dm.classes_)
        dpred = dm.predict_proba(Z)[:, classes.index(1)]
    else:
        dpred = (list(dm.classes_), dm.predict_proba(Z))
    dmetrics = modeling.full_metrics(tt, X["target"].to_numpy(), dpred)
    return {"suite": suite, "family": family, "template": template, "repetition": rep, "task_type": tt,
            "selected_candidate": best, "validation": {r["candidate"]: r["validation"] for r in results},
            "test_metrics": metrics, "dummy_test_metrics": dmetrics}


def build_references(paths: Paths, suite: str, families: list, templates: list, reps: list, log=print) -> dict:
    out_path = paths.outputs / f"references_{suite}.json"
    out = read_json(out_path) if out_path.exists() else {}
    for f in families:
        for t in templates:
            for r in reps:
                k = f"{f}-{t}/r{r}"
                if k in out:
                    continue
                out[k] = reference_workflow(paths, suite, f, t, r)
                write_json(out_path, out)
    log(f"reference workflows ({suite}): {len(out)}")
    return out


### Analysis: paired estimates, block bootstrap, Holm, tables, figures (Section 15)

In [ ]:
%%writefile adm_code/adm/analysis.py
"""Metrics, paired dependence-aware statistics, tables and figures (Sections 14-15, 22)."""
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd

from .core import FAMILIES, SYSTEMS, write_json, dumps

CONTRASTS = {
    "H1: MQP - M0": {"MQP": 1, "M0": -1},
    "Q main effect": {"MQ": 0.5, "M0": -0.5, "MQP": 0.5, "MP": -0.5},
    "P main effect": {"MP": 0.5, "M0": -0.5, "MQP": 0.5, "MQ": -0.5},
    "Q x P interaction": {"MQP": 1, "MQ": -1, "MP": -1, "M0": 1},
    "MQP - SQP": {"MQP": 1, "SQP": -1},
    "MQP - F": {"MQP": 1, "F": -1},
}
SECONDARY = ["Q main effect", "P main effect", "Q x P interaction", "MQP - SQP", "MQP - F"]
# Pre-freeze decision (2026-11-12): F abstains on every Tier-2 instance by construction, so the Holm-family MQP - F is
# estimated on Tier-1 conditions only; the all-tier MQP - F is reported as exploratory.
TIER1_ONLY = {"MQP - F"}
EXPLORATORY = {"M0 - S0": {"M0": 1, "S0": -1}, "SQP - S0": {"SQP": 1, "S0": -1},
               "MQP - F (all tiers, exploratory)": {"MQP": 1, "F": -1}}


def records_df(records: list) -> pd.DataFrame:
    rows = []
    for r in records:
        tm = r.get("test_metrics") or {}
        rows.append({
            "run_id": r["run_id"], "experiment": r["experiment_id"], "suite": r.get("suite"), "family": r["dataset_id"],
            "template": r["template_id"], "condition": r["condition_id"], "rep": r["repetition_id"],
            "system": r["system_id"], "backbone": r["backbone_id"], "budget": r.get("budget_profile", "standard"),
            "terminal": r["terminal_state"], "abstention": r.get("abstention_reason"),
            "V": r.get("verified_completion"), "first_failure": r.get("first_failure"),
            "codes": r.get("external_failure_codes") or [], "correct_refusal": r.get("correct_refusal"),
            "false_submission": r.get("false_submission"),
            "tokens": r.get("aggregate_tokens"), "input_tokens": r.get("input_tokens"), "output_tokens": r.get("output_tokens"),
            "model_calls": r.get("model_calls"), "tool_calls": r.get("tool_calls"), "gate_calls": r.get("gate_calls"),
            "repairs": r.get("repair_cycles"), "evaluations": r.get("pipeline_evaluations"), "fits": r.get("fit_count"),
            "active_s": r.get("active_seconds"), "llm_s": r.get("llm_seconds"), "queue_s": r.get("queue_seconds"),
            "memory_mb": r.get("peak_worker_memory_mb"), "exceptions": r.get("exception_events", 0),
            "attempt": r.get("attempt", 1), "replay": r.get("replay_status"),
            "ap": tm.get("average_precision"), "macro_f1": tm.get("macro_f1"), "roc_auc": tm.get("roc_auc"),
            "balanced_accuracy": tm.get("balanced_accuracy"), "rmse": tm.get("rmse"), "mae": tm.get("mae"),
            "value_match": r.get("test_value_match_rate"), "record_integrity": r.get("record_integrity_rate"),
            "repair_rate": (r.get("data_management") or {}).get("exact_reversible_repair_rate"),
            "unintended_edit": (r.get("data_management") or {}).get("unintended_edit_rate"),
            "version_ok": r.get("version_selection_correct"),
        })
    return pd.DataFrame(rows)


def recoverable(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["condition"].isin(["N1", "N2"])].copy()


# --------------------------------------------------------------------------- paired estimation
def block_differences(df: pd.DataFrame, weights: dict) -> pd.DataFrame | None:
    """D_fr: per (family, repetition), mean over templates and conditions of the weighted system contrast.
    Only complete paired cells (all systems in the contrast present) are used."""
    d = recoverable(df)
    d = d[d["system"].isin(weights)]
    if d.empty:
        return None
    piv = d.pivot_table(index=["family", "rep", "template", "condition"], columns="system", values="V", aggfunc="first")
    if not all(s in piv.columns for s in weights):
        return None
    piv = piv.dropna(subset=list(weights))
    if piv.empty:
        return None
    contrast = sum(w * piv[s].astype(float) for s, w in weights.items())
    D = contrast.groupby(level=["family", "rep"]).mean().rename("D").reset_index()
    D["cells"] = contrast.groupby(level=["family", "rep"]).size().to_numpy()
    return D


def estimate(D: pd.DataFrame) -> float:
    return float(D.groupby("family")["D"].mean().mean())


def block_bootstrap(D: pd.DataFrame, draws: int = 10000, seed: int = 20261030, level: float = 0.95) -> dict:
    """Families fixed; repetition blocks resampled with replacement within each family."""
    rng = np.random.default_rng(seed)
    fams = {f: g["D"].to_numpy() for f, g in D.groupby("family")}
    stats = np.empty(draws)
    for b in range(draws):
        stats[b] = np.mean([v[rng.integers(0, len(v), len(v))].mean() for v in fams.values()])
    a = (1 - level) / 2
    return {"ci_low": float(np.quantile(stats, a)), "ci_high": float(np.quantile(stats, 1 - a)),
            "bootstrap_sd": float(stats.std(ddof=1)), "draws": draws,
            "min_reps_per_family": int(min(len(v) for v in fams.values()))}


def sign_flip_test(D: pd.DataFrame, draws: int = 10000, seed: int = 20261031) -> float:
    """Paired block sign-flip test on family-repetition differences (assumes symmetric exchangeability)."""
    rng = np.random.default_rng(seed)
    obs = abs(estimate(D))
    fam = D["family"].to_numpy()
    vals = D["D"].to_numpy()
    uf = np.unique(fam)
    idx = [np.where(fam == f)[0] for f in uf]
    count = 0
    for _ in range(draws):
        s = rng.choice([-1.0, 1.0], size=len(vals))
        v = vals * s
        stat = abs(np.mean([v[i].mean() for i in idx]))
        count += stat >= obs - 1e-15
    return (count + 1) / (draws + 1)


def holm(pvals: dict) -> dict:
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    adj, running = {}, 0.0
    for k, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - k) * p))
        adj[name] = running
    return adj


def contrast_table(df: pd.DataFrame, draws: int = 10000) -> pd.DataFrame:
    rows = []
    pv = {}
    for name, w in {**CONTRASTS, **EXPLORATORY}.items():
        sub = df[df["condition"].map(tier_of) == "Tier 1"] if name in TIER1_ONLY else df
        D = block_differences(sub, w)
        if D is None or D.empty:
            rows.append({"contrast": name, "estimate_pp": None, "note": "insufficient paired data"})
            continue
        est = estimate(D)
        bs = block_bootstrap(D, draws)
        p = sign_flip_test(D, draws)
        lofo = {f: estimate(D[D["family"] != f]) for f in sorted(D["family"].unique())} if D["family"].nunique() > 1 else {}
        rows.append({"contrast": name, "estimate_pp": 100 * est, "ci_low_pp": 100 * bs["ci_low"], "ci_high_pp": 100 * bs["ci_high"],
                     "p_sign_flip": p, "families": int(D["family"].nunique()), "blocks": int(len(D)),
                     "paired_cells": int(D["cells"].sum()),
                     "lofo_min_pp": 100 * min(lofo.values()) if lofo else None, "lofo_max_pp": 100 * max(lofo.values()) if lofo else None,
                     "role": "primary" if name.startswith("H1") else "secondary" if name in SECONDARY else "exploratory"})
        if name in SECONDARY:
            pv[name] = p
    t = pd.DataFrame(rows)
    adj = holm(pv) if pv else {}
    t["p_holm"] = t["contrast"].map(adj)
    return t


def tier_of(condition: str) -> str:
    return "Tier 2" if str(condition).startswith("X") else "Tier 1"


TIER_CONTRASTS = ["H1: MQP - M0", "Q main effect", "P main effect", "Q x P interaction", "MQP - SQP", "MQP - F"]


def tier_difference(df: pd.DataFrame, weights: dict, draws: int = 10000) -> dict | None:
    """Tier-2 minus Tier-1 difference of a contrast, paired by (family, repetition) block; block bootstrap."""
    d = recoverable(df)
    D1 = block_differences(d[d["condition"].map(tier_of) == "Tier 1"], weights)
    D2 = block_differences(d[d["condition"].map(tier_of) == "Tier 2"], weights)
    if D1 is None or D2 is None:
        return None
    m = D2.merge(D1, on=["family", "rep"], suffixes=("_t2", "_t1"))
    if m.empty:
        return None
    D = pd.DataFrame({"family": m["family"], "rep": m["rep"], "D": m["D_t2"] - m["D_t1"]})
    bs = block_bootstrap(D, draws)
    return {"estimate_pp": 100 * estimate(D), "ci_low_pp": 100 * bs["ci_low"], "ci_high_pp": 100 * bs["ci_high"],
            "p_sign_flip": sign_flip_test(D, draws), "blocks": int(len(D))}


def tier_sections(df: pd.DataFrame, draws: int = 10000) -> str:
    """Prespecified tier analyses (main-plan-v2.1 Section 15): per-tier contrasts, Tier-2 minus Tier-1 differences,
    and sensitivity excluding X4."""
    d = recoverable(df)
    if d.empty or not (d["condition"].map(tier_of) == "Tier 2").any():
        return ""
    keep = lambda t: t[t["contrast"].isin(TIER_CONTRASTS)].drop(columns=["p_holm"], errors="ignore")
    parts = ["## Tier-separated analyses (prespecified, main-plan-v2.1)\n"]
    for title, sub in [("Tier 1 (declared defects, C0-C6)", d[d["condition"].map(tier_of) == "Tier 1"]),
                       ("Tier 2 (out-of-specification defects)", d[d["condition"].map(tier_of) == "Tier 2"]),
                       ("Sensitivity: Tier 2 excluding X4", d[(d["condition"].map(tier_of) == "Tier 2")
                                                             & ~d["condition"].str.startswith("X4")]),
                       ("Sensitivity: all recoverable tasks excluding X4", d[~d["condition"].str.startswith("X4")])]:
        parts.append(f"### {title}\n" + md_table(keep(contrast_table(sub, draws))))
    rows = []
    for name in TIER_CONTRASTS:
        r = tier_difference(d, CONTRASTS.get(name) or EXPLORATORY.get(name), draws)
        rows.append({"contrast": name, **(r or {"note": "insufficient paired data"})})
    parts.append("### Tier-2 minus Tier-1 difference of each contrast\n" + md_table(pd.DataFrame(rows)))
    parts.append("Per-tier and sensitivity estimates are prespecified secondary analyses; the primary H1 remains the "
                 "pooled estimate over all recoverable tasks. They are not Holm-adjusted.\n")
    return "\n".join(parts)


# --------------------------------------------------------------------------- descriptive tables
def vcr_tables(df: pd.DataFrame) -> dict:
    d = recoverable(df)
    out = {}
    if d.empty:
        return out
    g = d.groupby("system")["V"]
    fam_w = d.groupby(["system", "family"])["V"].mean().groupby("system").mean()
    out["by_system"] = pd.DataFrame({"attempted": g.size(), "verified": g.sum(), "VCR": g.mean(),
                                     "VCR_family_weighted": fam_w}).reindex([s for s in SYSTEMS if s in g.groups])
    out["by_system_family"] = d.pivot_table(index="system", columns="family", values="V", aggfunc="mean")
    out["by_system_condition"] = d.pivot_table(index="system", columns="condition", values="V", aggfunc="mean")
    out["counts_system_condition"] = d.pivot_table(index="system", columns="condition", values="V", aggfunc="count")
    out["terminal_states"] = d.pivot_table(index="system", columns="terminal", values="run_id", aggfunc="count", fill_value=0)
    out["first_failure"] = d[d["V"] == 0].pivot_table(index="system", columns="first_failure", values="run_id",
                                                       aggfunc="count", fill_value=0)
    sub = d[d["terminal"] == "SUBMITTED"]
    out["reliability"] = pd.DataFrame({
        "invalid_submission_rate": d.groupby("system").apply(lambda x: ((x["terminal"] == "SUBMITTED") & (x["V"] == 0)).mean()),
        "submission_error_rate": sub.groupby("system")["V"].apply(lambda v: 1 - v.mean()) if not sub.empty else None,
        "recoverable_abstention_rate": d.groupby("system")["terminal"].apply(lambda t: (t == "ABSTAINED").mean()),
    })
    return out


def efficiency_table(df: pd.DataFrame) -> pd.DataFrame:
    d = recoverable(df)
    g = d.groupby("system")
    t = pd.DataFrame({
        "runs": g.size(), "mean_tokens": g["tokens"].mean(), "p90_tokens": g["tokens"].quantile(0.9),
        "mean_model_calls": g["model_calls"].mean(), "mean_tool_calls": g["tool_calls"].mean(),
        "mean_gate_calls": g["gate_calls"].mean(), "mean_repairs": g["repairs"].mean(), "mean_fits": g["fits"].mean(),
        "mean_active_s": g["active_s"].mean(), "p90_active_s": g["active_s"].quantile(0.9), "mean_llm_s": g["llm_s"].mean(),
        "verified": g["V"].sum()})
    t["tokens_per_verified"] = (g["tokens"].sum() / t["verified"]).where(t["verified"] > 0)
    t["active_s_per_verified"] = (g["active_s"].sum() / t["verified"]).where(t["verified"] > 0)
    return t


def skill_table(df: pd.DataFrame, references: dict) -> pd.DataFrame:
    """Predictive scores for verified runs, skill relative to the correct-workflow dummy, verified utility."""
    rows = []
    for _, r in recoverable(df).iterrows():
        ref = references.get(f"{r['family']}-{r['template']}/r{r['rep']}")
        tt = ref["task_type"] if ref else None
        score, dummy, skill = None, None, None
        if ref:
            dm = ref["dummy_test_metrics"]
            if tt == "binary":
                score, dummy = r["ap"], dm.get("average_precision")
            elif tt == "multiclass":
                score, dummy = r["macro_f1"], dm.get("macro_f1")
            else:
                score, dummy = r["rmse"], dm.get("rmse")
            if score is not None and dummy is not None and not (isinstance(score, float) and math.isnan(score)):
                if tt == "regression":
                    skill = 1 - score / dummy if dummy else None
                else:
                    skill = (score - dummy) / (1 - dummy) if dummy != 1 else None
        v = r["V"] if r["V"] is not None else 0
        rows.append({"system": r["system"], "family": r["family"], "V": v, "score": score, "skill": skill,
                     "verified_utility": (v * max(0.0, skill)) if skill is not None else 0.0})
    t = pd.DataFrame(rows)
    if t.empty:
        return t
    ver = t[t["V"] == 1]
    out = pd.DataFrame({
        "verified_runs": ver.groupby("system").size(),
        "mean_skill_verified": ver.groupby("system")["skill"].mean(),
        "mean_unclipped_skill_all_scored": t.groupby("system")["skill"].mean(),
        "mean_verified_utility_all_attempts": t.groupby("system")["verified_utility"].mean(),
    })
    return out


def raw_metric_table(df: pd.DataFrame) -> pd.DataFrame:
    d = recoverable(df)
    d = d[d["V"] == 1]
    if d.empty:
        return pd.DataFrame()
    return d.groupby(["family", "system"]).agg(n=("run_id", "size"), ap=("ap", "mean"), macro_f1=("macro_f1", "mean"),
                                               rmse=("rmse", "mean"), mae=("mae", "mean"))


def negative_table(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["condition"].isin(["N1", "N2"])]
    if d.empty:
        return pd.DataFrame()
    g = d.groupby("system")
    return pd.DataFrame({"attempts": g.size(), "correct_refusal": g["correct_refusal"].mean(),
                         "false_submission": g["false_submission"].mean()})


def usage_percentiles(df: pd.DataFrame) -> pd.DataFrame:
    d = df[(df["system"] != "F") & df["terminal"].isin(["SUBMITTED", "ABSTAINED"])]
    cols = ["tokens", "model_calls", "tool_calls", "active_s", "repairs", "evaluations"]
    if d.empty:
        return pd.DataFrame(index=cols)
    return d[cols].astype(float).quantile([0.5, 0.9, 0.95, 1.0]).T.rename(columns=lambda q: f"p{int(q * 100)}")


# --------------------------------------------------------------------------- figures
def figures(df: pd.DataFrame, outdir: Path, prefix: str) -> list:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    outdir.mkdir(parents=True, exist_ok=True)
    d = recoverable(df)
    files = []
    if d.empty:
        return files
    piv = d.pivot_table(index="condition", columns="system", values="V", aggfunc="mean")
    piv = piv[[s for s in SYSTEMS if s in piv.columns]]
    ax = piv.plot(kind="bar", figsize=(10, 4.5), width=0.85)
    ax.set_ylabel("Verified-completion rate")
    ax.set_ylim(0, 1)
    ax.set_title(f"{prefix}: completion by defect condition (C* families are not one severity scale)")
    plt.tight_layout()
    p = outdir / f"{prefix}_completion_by_condition.png"
    plt.savefig(p, dpi=120)
    plt.close()
    files.append(p)
    e = d.groupby("system").agg(V=("V", "mean"), tokens=("tokens", "mean"), active=("active_s", "mean"))
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, lab in [(axs[0], "tokens", "Mean aggregate tokens per run"), (axs[1], "active", "Mean active seconds per run")]:
        ax.scatter(e[col], e["V"])
        for s, row in e.iterrows():
            ax.annotate(s, (row[col], row["V"]))
        ax.set_xlabel(lab)
        ax.set_ylabel("VCR")
    fig.suptitle(f"{prefix}: cost versus completion")
    plt.tight_layout()
    p = outdir / f"{prefix}_cost_vs_completion.png"
    plt.savefig(p, dpi=120)
    plt.close()
    files.append(p)
    ff = d[d["V"] == 0].pivot_table(index="system", columns="first_failure", values="run_id", aggfunc="count", fill_value=0)
    if not ff.empty:
        ax = ff.plot(kind="bar", stacked=True, figsize=(10, 4.5))
        ax.set_ylabel("Runs (first failure category)")
        ax.set_title(f"{prefix}: failure categories")
        plt.tight_layout()
        p = outdir / f"{prefix}_failure_categories.png"
        plt.savefig(p, dpi=120)
        plt.close()
        files.append(p)
    return files


def md_table(t: pd.DataFrame | None, floatfmt: str = "{:.3f}") -> str:
    if t is None or (hasattr(t, "empty") and t.empty):
        return "_no data_\n"
    t = t.copy()
    t = t.reset_index() if not isinstance(t.index, pd.RangeIndex) else t
    cols = [str(c) for c in t.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in t.iterrows():
        vals = []
        for v in r.tolist():
            if isinstance(v, (float, np.floating)):
                vals.append("" if math.isnan(v) else floatfmt.format(v))
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines) + "\n"


def report(df: pd.DataFrame, title: str, outdir: Path, figdir: Path, references: dict, banner: str = "",
           draws: int = 10000, extra_sections: list | None = None) -> str:
    """Write the mandatory result tables (Section 22.1) for a set of runs."""
    outdir.mkdir(parents=True, exist_ok=True)
    parts = [f"# {title}\n", banner + "\n" if banner else ""]
    v = vcr_tables(df)
    parts.append("## 1. Verified-completion rate (recoverable runs)\n")
    parts.append(md_table(v.get("by_system")))
    parts.append("### By dataset family\n" + md_table(v.get("by_system_family")))
    parts.append("### By condition\n" + md_table(v.get("by_system_condition")))
    parts.append("### Attempt counts by condition\n" + md_table(v.get("counts_system_condition"), "{:.0f}"))
    parts.append("## 2. Paired effects (percentage points; family-weighted; block bootstrap within family)\n")
    ct = contrast_table(df, draws)
    parts.append(md_table(ct))
    parts.append("Failure to reject a difference is not evidence of equivalence. With three repetition blocks per family the "
                 "bootstrap interval can be unstable.\n")
    parts.append("## 3. Predictive quality\n" + md_table(skill_table(df, references)) + "\n### Raw metrics of verified runs\n"
                 + md_table(raw_metric_table(df)))
    parts.append("## 4. Cost and efficiency\n" + md_table(efficiency_table(df), "{:.1f}"))
    parts.append("## 5. Failure categories and reliability\n" + md_table(v.get("first_failure"), "{:.0f}")
                 + md_table(v.get("reliability")) + "### Terminal states\n" + md_table(v.get("terminal_states"), "{:.0f}"))
    parts.append("## 6. Negative controls (unrecoverable tasks)\n" + md_table(negative_table(df)))
    for s in extra_sections or []:
        parts.append(s)
    files = figures(df, figdir, title.split()[0].lower())
    import os
    parts.append("## Figures\n" + "\n".join(f"![{p.stem}]({os.path.relpath(p, outdir)})" for p in files) + "\n")
    text = "\n".join(parts)
    for k, t in v.items():
        t.to_csv(outdir / f"{title.split()[0].lower()}_{k}.csv")
    ct.to_csv(outdir / f"{title.split()[0].lower()}_contrasts.csv", index=False)
    return text


### Power simulation, limit calibration and protocol freeze

In [ ]:
%%writefile adm_code/adm/power.py
"""Pilot-based power simulation, limit calibration rule, protocol freeze and freeze verification
(Sections 15.6 and 23)."""
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd

from .analysis import block_bootstrap, estimate, recoverable
from .core import (FAMILIES, PILOT_LIMITS, PROTOCOL_LIMITS, TEMPLATES, CONDITIONS, Paths, json_hash, now_iso,
                   read_json, sha256_tree, write_json)

LIMIT_KEYS = {"aggregate_tokens": ("tokens", 1000), "model_calls": ("model_calls", 1), "tool_calls": ("tool_calls", 1),
              "wall_seconds": ("active_s", 30)}


def _logit(p):
    p = min(max(p, 1e-3), 1 - 1e-3)
    return math.log(p / (1 - p))


def simulate_power(p0: float, effect: float, reps: int, sd_family: float, sd_block: float, sims: int = 200,
                   draws: int = 400, seed: int = 7, n_conditions: int | None = None) -> float:
    """Simulate the planned paired analysis: Bernoulli completions with family and family-repetition
    random effects shared by both systems in a block; power = P(CI lower bound > 0)."""
    rng = np.random.default_rng(seed)
    mu0 = _logit(p0)
    mu1 = _logit(min(p0 + effect, 0.999))
    nf, cells = len(FAMILIES), len(TEMPLATES) * (n_conditions or len(CONDITIONS))
    hits = 0
    for s in range(sims):
        rows = []
        u = rng.normal(0, sd_family, nf)
        for f in range(nf):
            for r in range(reps):
                v = rng.normal(0, sd_block)
                w = rng.normal(0, 0.5, cells)
                p_a = 1 / (1 + np.exp(-(mu1 + u[f] + v + w)))
                p_b = 1 / (1 + np.exp(-(mu0 + u[f] + v + w)))
                d = (rng.random(cells) < p_a).astype(float) - (rng.random(cells) < p_b).astype(float)
                rows.append({"family": f, "rep": r, "D": d.mean()})
        D = pd.DataFrame(rows)
        bs = block_bootstrap(D, draws=draws, seed=int(rng.integers(0, 2**31 - 1)))
        hits += bs["ci_low"] > 0
    return hits / sims


def power_analysis(pilot: pd.DataFrame, out_dir: Path, rep_options=(3, 4, 5, 6), effect: float = 0.10,
                   sims: int = 200, n_conditions: int | None = None, out_name: str = "power_simulation.json") -> dict:
    d = recoverable(pilot)
    p0 = float(d.loc[d["system"] == "M0", "V"].mean()) if (d["system"] == "M0").any() else 0.5
    if not np.isfinite(p0):
        p0 = 0.5
    grid = []
    for sd_block in (0.25, 0.5, 1.0):
        for reps in rep_options:
            pw = simulate_power(p0, effect, reps, sd_family=0.5, sd_block=sd_block, sims=sims, seed=1000 + reps,
                                n_conditions=n_conditions)
            grid.append({"reps": reps, "sd_block_logit": sd_block, "power": pw})
    res = {"assumptions": {"baseline_vcr_M0_pilot": p0, "effect_pp": 100 * effect, "sd_family_logit": 0.5,
                           "sd_cell_logit": 0.5, "sims": sims,
                           "note": "Development pilot tasks differ from the evaluation portfolio; the variance model is an assumption."},
           "grid": grid, "time": now_iso()}
    write_json(out_dir / out_name, res)
    return res


def calibrate_limits(pilot: pd.DataFrame) -> dict:
    """Prespecified rule: each frozen limit is max(protocol-text limit, 1.25 x p95 of pilot usage), rounded
    up, and never above the pilot calibration limit. Usage is pooled over agentic runs that ended in
    SUBMITTED or ABSTAINED. Repair and evaluation caps are unchanged. The floor keeps the protocol's
    stated budgets; the headroom keeps limits from binding by construction on well-behaved runs."""
    d = pilot[(pilot["system"] != "F") & pilot["terminal"].isin(["SUBMITTED", "ABSTAINED"])]
    out = {"rule": "max(protocol-text limit, ceil(1.25 x p95 of pilot usage)), capped at the pilot limit; "
                   "agentic SUBMITTED/ABSTAINED runs pooled", "n_runs": int(len(d)), "p95": {}}
    lim = dict(PROTOCOL_LIMITS)
    for k, (col, step) in LIMIT_KEYS.items():
        if len(d) and d[col].notna().any():
            p95 = float(np.nanquantile(d[col].astype(float), 0.95))
            out["p95"][k] = p95
            v = int(math.ceil(1.25 * p95 / step) * step)
            lim[k] = min(max(PROTOCOL_LIMITS[k], v), PILOT_LIMITS[k])
        else:
            lim[k] = PILOT_LIMITS[k]
    lim["repair_cycles"] = PROTOCOL_LIMITS["repair_cycles"]
    lim["pipeline_evaluations"] = PROTOCOL_LIMITS["pipeline_evaluations"]
    out["limits"] = lim
    out["protocol_text_limits"] = dict(PROTOCOL_LIMITS)
    out["pilot_limits"] = dict(PILOT_LIMITS)
    return out


def throughput(pilot: pd.DataFrame, stage_wall_seconds: float | None, concurrency: int) -> dict:
    ag = pilot[pilot["system"] != "F"]
    out = {"agentic_runs": int(len(ag)), "mean_active_s": float(ag["active_s"].mean()) if len(ag) else None,
           "mean_llm_s": float(ag["llm_s"].mean()) if len(ag) else None, "concurrency": concurrency}
    if stage_wall_seconds and len(pilot):
        out["stage_wall_seconds"] = stage_wall_seconds
        out["seconds_per_run_effective"] = stage_wall_seconds / len(pilot)
    return out


def session_estimates(tp: dict, reps: int, session_hours: float = 10.0, gpus: int = 2) -> dict:
    spr = tp.get("seconds_per_run_effective")
    agentic_e1 = 36 * 7 * 6 * reps
    fixed_e1 = 36 * 7 * reps
    e4 = 6 * 7 * 3 * reps
    e5 = 6 * 2 * 7 * reps
    total = agentic_e1 + fixed_e1 + e4 + e5
    out = {"repetitions": reps, "runs": {"E1": agentic_e1 + fixed_e1, "E4": e4, "E5": e5, "total": total}}
    if spr:
        hours = total * spr / 3600
        out["estimated_hours"] = round(hours, 1)
        out["estimated_sessions"] = math.ceil(hours / session_hours)
        out["estimated_gpu_hours"] = round(gpus * hours, 1)
        out["gpus"] = gpus
    return out


def formal_estimates(spr_t1: float | None, spr_t2: float | None, reps: int, n_t1: int = 7, n_t2: int = 3,
                     spr_b: float | None = None, gpus: int = 1) -> dict:
    """Run counts and time for the v2.1 formal design: E1 and E4 over Tier 1 (n_t1) and Tier 2 (n_t2) conditions,
    E5 negative controls. Seconds per run are measured per tier from pilot stages; E4 uses backbone B's rate if given."""
    e1_t1, e1_t2 = 36 * n_t1 * 7 * reps, 36 * n_t2 * 7 * reps
    e4_t1, e4_t2 = 6 * n_t1 * 3 * reps, 6 * n_t2 * 3 * reps
    e5 = 6 * 2 * 7 * reps
    out = {"repetitions": reps, "runs": {"E1": e1_t1 + e1_t2, "E4": e4_t1 + e4_t2, "E5": e5,
                                         "total": e1_t1 + e1_t2 + e4_t1 + e4_t2 + e5}}
    if spr_t1 and spr_t2:
        b1, b2 = (spr_b or spr_t1), (spr_b or spr_t1) * spr_t2 / spr_t1
        sec = (e1_t1 + e5) * spr_t1 + e1_t2 * spr_t2 + e4_t1 * b1 + e4_t2 * b2
        out["estimated_hours"] = round(sec / 3600, 1)
        out["estimated_gpu_hours"] = round(gpus * sec / 3600, 1)
        out["seconds_per_run"] = {"tier1": round(spr_t1, 2), "tier2": round(spr_t2, 2), "backbone_b": spr_b and round(spr_b, 2)}
    return out


def protocol_hash(pkg_dir: Path, study_cfg: dict) -> dict:
    code = sha256_tree(pkg_dir, patterns=("**/*.py", "**/*.md", "**/*.json"))
    return {"code_hash": code, "config_hash": json_hash(study_cfg), "protocol_hash": json_hash({"code": code, "cfg": study_cfg})}


def frozen_config_subset(cfg: dict) -> dict:
    """The parts of the study configuration that the freeze locks."""
    keys = ["limits", "generation", "repetitions", "optional_budget_experiment", "concurrency", "fit_slots"]
    if cfg.get("tier2_conditions") is not None:  # main-plan-v2.1: the frozen Tier-2 levels are locked too
        keys.append("tier2_conditions")
    return {k: cfg.get(k) for k in keys}


def write_freeze(paths: Paths, pkg_dir: Path, cfg: dict, proposal: dict, model_lock: dict) -> dict:
    sub = frozen_config_subset(cfg)
    ph = protocol_hash(pkg_dir, sub)
    man = {"created": now_iso(), **ph, "frozen_config": sub, "proposal": proposal, "model_lock": model_lock,
           "datasets_manifest_hash": json_hash(read_json(paths.manifests / "datasets.json")),
           "splits_manifest_hash": json_hash(read_json(paths.manifests / "splits.json")),
           "canonical_manifest_hash": json_hash(read_json(paths.manifests / "canonical.json")),
           "primary": ("H1: MQP - M0 verified completion, backbone A, all recoverable core tasks"
                       + (" (Tier 1 and the frozen Tier-2 conditions, pooled; main-plan-v2.1)" if cfg.get("tier2_conditions") else "")),
           "secondary_holm_family": ["Q main effect", "P main effect", "Q x P interaction", "MQP - SQP",
                                     "MQP - F (Tier 1 conditions only)"],
           "pre_freeze_decisions_20261112": [
               "MQP - F in the Holm family is estimated on Tier 1 only, because F abstains on Tier 2 by construction; "
               "MQP - F over all tiers is exploratory.",
               "Prespecified secondary outcome (not Holm-adjusted): use of the content-verifying load policy "
               "(authoritative_verified) on X4 when it is available but not enforced, per system, from trusted traces; "
               "and its use on Tier 1 as a false-alarm measure.",
               "Prespecified sensitivity analyses for the primary estimate (three repetition blocks per family): "
               "(a) mixed-effects logistic regression of V on system with random intercepts for task and for "
               "family-repetition block; (b) the block bootstrap with the small-sample variance inflation n/(n-1); "
               "(c) per-family estimates reported alongside leave-one-family-out.",
               "Prespecified descriptive analysis: terminal state and tool-call composition of budget-exhausted runs by "
               "system and tier (exploration, materialisation, fits, gate-check calls)."],
           "prespecified_tier_analyses": ([
               "Every primary and Holm-family contrast estimated separately on Tier 1 and on Tier 2",
               "Tier-2 minus Tier-1 difference of each contrast (paired family-repetition blocks, block bootstrap)",
               "Sensitivity: all contrasts on Tier 2 excluding X4, and on all recoverable tasks excluding X4"]
               if cfg.get("tier2_conditions") else []),
           "replication_tasks": ["D1-T5", "D2-T2", "D3-T4", "D4-T6", "D5-T3", "D6-T1"]}
    write_json(paths.manifests / "freeze.json", man)
    return man


def verify_freeze(paths: Paths, pkg_dir: Path, cfg: dict, model_lock: dict, require_model_lock: bool) -> dict:
    p = paths.manifests / "freeze.json"
    if not p.exists():
        raise RuntimeError("formal experiments require manifests/freeze.json; run the freeze stage first")
    man = read_json(p)
    sub = frozen_config_subset(cfg)
    ph = protocol_hash(pkg_dir, sub)
    problems = []
    if ph["code_hash"] != man["code_hash"]:
        problems.append("package code or prompts changed since the freeze")
    if ph["config_hash"] != man["config_hash"]:
        problems.append(f"frozen configuration changed: now {sub}, frozen {man['frozen_config']}")
    for k in ("datasets", "splits", "canonical"):
        if json_hash(read_json(paths.manifests / f"{k}.json")) != man[f"{k}_manifest_hash"]:
            problems.append(f"{k} manifest changed since the freeze")
    if require_model_lock:
        for b, e in (man.get("model_lock", {}).get("backbones") or {}).items():
            cur = (model_lock.get("backbones") or {}).get(b)
            if not cur or cur.get("revision") != e.get("revision") or cur.get("repo") != e.get("repo"):
                problems.append(f"backbone {b} differs from the frozen model lock")
        if not (man.get("model_lock", {}).get("backbones") or {}).get("A", {}).get("revision"):
            problems.append("frozen model lock has no resolved revision for backbone A")
    if problems:
        raise RuntimeError("protocol freeze verification failed: " + "; ".join(problems))
    return man


### Evaluator fixtures and E6 replay/invariance checks

In [ ]:
%%writefile adm_code/adm/fixtures.py
"""Known-answer evaluator fixtures (E0 pre-step) and replay/invariance checks (E6).
Results are kept separate from model-capability scores."""
from __future__ import annotations

import copy
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from .core import Paths, SYSTEM_SPEC, instance_dirs, read_json, sha256_file, write_json, PILOT_LIMITS
from .engine import parse_cell
from .evaluator import evaluate
from .fixed import contract_graph, oracle_graph
from .tasks import ensure_instance
from .tools import Budget, Terminal, ToolSession


class InvalidProbaModel:
    """Wraps a fitted pipeline but returns invalid probabilities (fixture only)."""

    def __init__(self, inner):
        self.inner = inner
        self.named_steps = inner.named_steps
        self.classes_ = inner.classes_

    def predict_proba(self, X):
        return self.inner.predict_proba(X) * 1.5 + 0.1

    def predict(self, X):
        return self.inner.predict(X)


def mutate(graph: dict, **flags) -> dict:
    alias, nodes = {}, []
    for n in copy.deepcopy(graph["nodes"]):
        for r in ("input", "left", "right"):
            if r in n:
                n[r] = alias.get(n[r], n[r])
        op = n["op"]
        if (op == "map_alias" and flags.get("skip_alias")) or (op == "dedupe" and flags.get("skip_dedupe")) or \
           (op == "to_canonical" and flags.get("skip_canonical")):
            alias[n["id"]] = n["input"]
            continue
        if op == "load" and flags.get("policy"):
            n["policy"] = flags["policy"]
        if op == "join":
            if flags.get("join_how"):
                n["how"] = flags["join_how"]
            if flags.get("drop_cardinality"):
                n.pop("cardinality", None)
        if op == "aggregate" and flags.get("sample_std"):
            n["stats"] = ["mean", "sample_std", "count"]
        if op == "project" and n["id"] == "features":
            if flags.get("extra_column"):
                n["columns"] = n["columns"] + [flags["extra_column"]]
            if flags.get("drop_missing"):
                nodes.append({"id": "dropna", "op": "drop_missing_rows", "input": n["input"]})
                n["input"] = "dropna"
        nodes.append(n)
    return {"nodes": nodes, "output": alias.get(graph["output"], graph["output"])}


FIXTURES = [
    # name, (family, template, condition), graph flags, candidate index, tamper, expected (None = verified)
    ("correct_T3_C4", ("D5", "T3", "C4"), {}, 2, None, None),
    ("valid_alt_inner_join_T3_C4", ("D5", "T3", "C4"), {"join_how": "inner"}, 0, None, None),
    ("valid_alt_explicit_version_T3_C4", ("D5", "T3", "C4"), {"policy": "version:v2"}, 4, None, None),
    ("wrong_join_unmapped_alias_T3_C4", ("D5", "T3", "C4"), {"skip_alias": True, "join_how": "inner"}, 2, None, "MISSING_RECORD"),
    ("duplicated_predictions_T3_C4", ("D5", "T3", "C4"), {"skip_dedupe": True, "drop_cardinality": True}, 2, None, "ROW_MULTIPLICITY"),
    ("correct_T6_C6", ("D6", "T6", "C6"), {}, 2, None, None),
    ("wrong_version_T6_C6", ("D6", "T6", "C6"), {"policy": "latest_modified"}, 2, None, "SOURCE_VERSION"),
    ("unit_mismatch_T6_C6", ("D6", "T6", "C6"), {"skip_canonical": True}, 2, None, "UNIT_MISMATCH"),
    ("bad_aggregate_T6_C6", ("D6", "T6", "C6"), {"sample_std": True}, 2, None, "BAD_AGGREGATE"),
    ("forbidden_predictor_T6_C6", ("D6", "T6", "C6"), {"extra_column": "batch_id"}, 2, None, "FORBIDDEN_FEATURE"),
    ("wrong_version_T5_C0", ("D6", "T5", "C0"), {"policy": "latest_snapshot"}, 2, None, "SOURCE_VERSION"),
    ("missing_records_drop_rows_T1_C2", ("D5", "T1", "C2"), {"drop_missing": True}, 2, None, "MISSING_RECORD"),
    ("forbidden_identifier_T1_C0", ("D5", "T1", "C0"), {"extra_column": "machine_id"}, 2, None, "FORBIDDEN_FEATURE"),
    ("changed_artifact_hash_T1_C0", ("D5", "T1", "C0"), {}, 2, "graph", "ARTIFACT_MISMATCH"),
    ("invalid_probabilities_T1_C0", ("D5", "T1", "C0"), {}, 2, "proba", "INVALID_PREDICTION"),
]
# Tier 2 (main-plan-v2.1): the oracle repairs each defect; the declared-specification graph (F) does not
TIER2_FIXTURES = [
    ("oracle_T2_X1b", ("D5", "T2", "X1b"), {"oracle": True}, 2, None, None),
    ("spec_graph_near_duplicates_T1_X1b", ("D5", "T1", "X1b"), {}, 2, None, "ROW_MULTIPLICITY"),
    ("oracle_T6_X2b", ("D6", "T6", "X2b"), {"oracle": True}, 2, None, None),
    ("spec_graph_key_variants_T1_X2b", ("D5", "T1", "X2b"), {}, 2, None, "MISSING_RECORD"),
    ("oracle_T4_X3b", ("D6", "T4", "X3b"), {"oracle": True}, 2, None, None),
    ("spec_graph_undeclared_rep_T1_X3b", ("D5", "T1", "X3b"), {}, 2, None, "UNIT_MISMATCH"),
    ("oracle_T3_X4b", ("D5", "T3", "X4b"), {"oracle": True}, 2, None, None),
    ("spec_graph_content_drift_T1_X4b", ("D6", "T1", "X4b"), {}, 2, None, "SOURCE_VERSION"),
    ("latest_modified_content_drift_T2_X4c", ("D5", "T2", "X4c"), {"oracle": True, "policy": "latest_modified"}, 2, None,
     "SOURCE_VERSION"),
]


def _submit(paths: Paths, run_id: str, inst: tuple, graph_flags: dict, cand_idx: int, suite: str, rep: int) -> tuple:
    fam, tmpl, cond = inst
    ensure_instance(paths, suite, fam, tmpl, cond, rep)
    dirs = instance_dirs(paths, suite, fam, tmpl, cond, rep)
    sub = paths.work / "fixtures" / run_id
    if sub.exists():
        shutil.rmtree(sub)
    lim = dict(PILOT_LIMITS)
    s = ToolSession(run_id, dirs["visible"], dirs["trusted"], sub, SYSTEM_SPEC["S0"], Budget(lim),
                    {"estimator": 12345, "agent": 1})
    flags = dict(graph_flags)
    builder = oracle_graph if flags.pop("oracle", False) else contract_graph
    g = mutate(builder(s.contract, s.catalog), **flags)
    obs = s.call("materialize_features", {"graph": g})
    if not obs.get("materialized"):
        raise RuntimeError(f"fixture {run_id}: materialization failed: {obs.get('error')}")
    cand = s.contract["candidate_models"][cand_idx]
    r = s.call("fit_evaluate", {"handle": obs["handle"], "candidate": cand})
    try:
        s.call("submit", {"handle": obs["handle"], "candidate": cand})
    except Terminal:
        pass
    s.close()
    return dirs, sub, s.budget.usage(), lim


def _tamper(sub: Path, kind: str):
    if kind == "graph":
        g = read_json(sub / "graph.json")
        g["nodes"][0]["policy"] = "latest_modified"
        write_json(sub / "graph.json", g)
    elif kind == "proba":
        with open(sub / "model.pkl", "rb") as f:
            m = pickle.load(f)
        with open(sub / "model.pkl", "wb") as f:
            pickle.dump(InvalidProbaModel(m), f)
        b = read_json(sub / "bundle.json")
        b["files"]["model.pkl"] = sha256_file(sub / "model.pkl")
        write_json(sub / "bundle.json", b)


def run_evaluator_fixtures(paths: Paths, suite: str = "development", rep: int = 2, log=print, fixtures=None,
                           out_name: str = "evaluator_fixtures.json") -> dict:
    results = []
    for name, inst, flags, ci, tamper, expected in (FIXTURES if fixtures is None else fixtures):
        dirs, sub, usage, lim = _submit(paths, f"FIX-{name}", inst, flags, ci, suite, rep)
        if tamper:
            _tamper(sub, tamper)
        ev = evaluate({"run_id": f"FIX-{name}", "terminal_state": "SUBMITTED", "abstention_reason": None,
                       "instance": {k: str(v) for k, v in dirs.items()}, "submission_dir": str(sub), "usage": usage,
                       "limits": lim, "negative": None, "retain_model": True})
        codes = ev["external_failure_codes"]
        ok = (ev["verified_completion"] == 1) if expected is None else (expected in codes)
        results.append({"fixture": name, "expected": expected or "VERIFIED", "codes": codes,
                        "verified": ev["verified_completion"], "passed": bool(ok)})
        log(f"  fixture {name:40s} expected {expected or 'VERIFIED':18s} got {codes or 'VERIFIED'} -> {'PASS' if ok else 'FAIL'}")
    # unit-level invariance: equivalent numeric representations parse identically
    eq = all(parse_cell(a)[0] == parse_cell(b)[0] for a, b in [("value:1.50", "1.5"), ("2.0", 2), (" value:3e2", "300")])
    results.append({"fixture": "equivalent_numeric_representations", "expected": "equal", "codes": [], "verified": None,
                    "passed": bool(eq)})
    out = {"fixtures": results, "all_passed": all(r["passed"] for r in results)}
    write_json(paths.outputs / out_name, out)
    return out


def oracle_solvability(paths: Paths, keys: list, log=print) -> dict:
    """Run the oracle solver on every partition of every instance and compare with the hidden reference
    (ABC II.7-II.8: every task is verified to be solvable before agents run)."""
    from .engine import CatalogView, execute
    failures = []
    for k in keys:
        suite, fam, tmpl, cond, rep = k
        ensure_instance(paths, suite, fam, tmpl, cond, rep)
        d = instance_dirs(paths, suite, fam, tmpl, cond, rep)
        g = oracle_graph(read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json"))
        for part, root in [("train", d["visible"]), ("validation", d["visible"]), ("test", d["private"])]:
            out, _ = execute(g, CatalogView.open(root, part))
            ref = pd.read_parquet(d["private"] / f"reference_{part}.parquet")
            o = out.assign(observation_id=out["observation_id"].astype(str)).sort_values("observation_id").reset_index(drop=True)
            r = ref.sort_values("observation_id").reset_index(drop=True)
            ok = len(o) == len(r) and (o["observation_id"].values == r["observation_id"].values).all()
            if ok:
                for c in r.columns[1:]:
                    if pd.api.types.is_numeric_dtype(r[c]):
                        ok &= bool(np.isclose(o[c].astype(float), r[c].astype(float), rtol=1e-6, atol=1e-8, equal_nan=True).all())
                    else:
                        ok &= bool((o[c].astype(str).values == r[c].astype(str).values).all())
            if not ok:
                failures.append(f"{fam}-{tmpl}/{cond}/r{rep}/{part}")
    log(f"oracle solvability: {len(keys)} instances, {len(failures)} failures {failures[:5]}")
    return {"instances": len(keys), "failures": failures, "all_solvable": not failures}


# --------------------------------------------------------------------------- E6
def _copy_instance(dirs: dict, dest: Path) -> dict:
    if dest.exists():
        shutil.rmtree(dest)
    out = {}
    for k, v in dirs.items():
        shutil.copytree(v, dest / k)
        out[k] = dest / k
    return out


def replay_checks(paths: Paths, record: dict, sub_dir: Path, log=print) -> dict:
    """E6 for one frozen submission with a retained model: replay, row/column-order invariance,
    wrong-version substitution, one missing test record, changed artifact hash."""
    r = record
    ensure_instance(paths, r["suite"], r["dataset_id"], r["template_id"], r["condition_id"], r["repetition_id"])
    dirs = instance_dirs(paths, r["suite"], r["dataset_id"], r["template_id"], r["condition_id"], r["repetition_id"])
    base = paths.work / "e6" / r["run_id"]
    out = {"run_id": r["run_id"], "checks": {}}

    def ev(inst, sub):
        return evaluate({"run_id": r["run_id"], "terminal_state": "SUBMITTED", "abstention_reason": None,
                         "instance": {k: str(v) for k, v in inst.items()}, "submission_dir": str(sub),
                         "usage": {}, "limits": {}, "negative": None, "retain_model": True})

    def fresh_sub(tag):
        d = base / f"sub_{tag}"
        if d.exists():
            shutil.rmtree(d)
        shutil.copytree(sub_dir, d)
        return d

    e0 = ev(dirs, fresh_sub("orig"))
    p0 = pd.read_parquet(base / "sub_orig" / "test_predictions.parquet") if (base / "sub_orig" / "test_predictions.parquet").exists() else None
    out["checks"]["replay_matches_record"] = {"passed": e0["verified_completion"] == r["verified_completion"],
                                              "detail": e0["external_failure_codes"]}
    # reordered rows and columns in every hidden test source
    inst = _copy_instance(dirs, base / "inst_reorder")
    rng = np.random.default_rng(99)
    for f in (inst["private"] / "sources").glob("*.parquet"):
        d = pd.read_parquet(f)
        d = d.iloc[rng.permutation(len(d))][list(reversed(d.columns))]
        d.to_parquet(f, index=False)
    e1 = ev(inst, fresh_sub("reorder"))
    same = False
    if p0 is not None and (base / "sub_reorder" / "test_predictions.parquet").exists():
        p1 = pd.read_parquet(base / "sub_reorder" / "test_predictions.parquet")
        a = p0.sort_values("observation_id").reset_index(drop=True)
        b = p1.sort_values("observation_id").reset_index(drop=True)
        num = [c for c in a.columns if c not in ("observation_id",) and pd.api.types.is_numeric_dtype(a[c])]
        same = a["observation_id"].equals(b["observation_id"]) and all(
            np.allclose(a[c], b[c], rtol=1e-6, atol=1e-8, equal_nan=True) for c in num)
    out["checks"]["row_column_reorder_invariant"] = {"passed": bool(same and e1["external_failure_codes"] == e0["external_failure_codes"]),
                                                     "detail": e1["external_failure_codes"]}
    if e0["verified_completion"] == 1:
        # wrong-version substitution: altered content behind an authorised hash
        inst = _copy_instance(dirs, base / "inst_wrongver")
        tcat = read_json(inst["private"] / "catalog.json")
        tab = next(t for t, v in tcat["tables"].items() if v["grain"] == "observation")
        fpath = inst["private"] / next(f["path"] for f in tcat["tables"][tab]["files"] if f["version"] == "v2")
        d = pd.read_parquet(fpath)
        numc = [c for c in d.columns if pd.api.types.is_numeric_dtype(d[c]) and c != "_shard"]
        if numc:
            d.loc[d.index[0], numc[0]] = float(d[numc[0]].iloc[0]) + 1.2345
            d.to_parquet(fpath, index=False)
        e2 = ev(inst, fresh_sub("wrongver"))
        out["checks"]["wrong_version_detected"] = {"passed": "SOURCE_VERSION" in e2["external_failure_codes"],
                                                   "detail": e2["external_failure_codes"]}
        # one missing test record, removed from the primary observation table
        inst = _copy_instance(dirs, base / "inst_missing")
        primary = read_json(dirs["private"] / "instance_meta.json")["primary_table"]
        fpath = inst["private"] / next(f["path"] for f in tcat["tables"][primary]["files"] if f["version"] == "v2")
        d = pd.read_parquet(fpath)
        d.iloc[1:].to_parquet(fpath, index=False)
        e3 = ev(inst, fresh_sub("missing"))
        out["checks"]["missing_record_detected"] = {
            "passed": e3["verified_completion"] == 0 and bool({"MISSING_RECORD", "INVALID_PREDICTION"} & set(e3["external_failure_codes"])),
            "detail": e3["external_failure_codes"]}
        # changed artifact hash
        s4 = fresh_sub("tamper")
        g = read_json(s4 / "graph.json")
        g["nodes"].append({"id": "zz", "op": "project", "input": g["output"], "columns": ["observation_id"]})
        write_json(s4 / "graph.json", g)
        e4 = ev(dirs, s4)
        out["checks"]["changed_artifact_detected"] = {"passed": "ARTIFACT_MISMATCH" in e4["external_failure_codes"],
                                                      "detail": e4["external_failure_codes"]}
    shutil.rmtree(base, ignore_errors=True)
    out["all_passed"] = all(c["passed"] for c in out["checks"].values())
    return out


### Model serving (vLLM: Kaggle 2x T4 or CRC 1x RTX PRO 6000)

In [ ]:
%%writefile adm_code/adm/serving.py
"""Model serving: isolated vLLM environment, model download/lock, server launch and preflight.

Platform profiles: Kaggle (2 x Tesla T4, 4-bit AWQ in float16) and the UA Little Rock CRC JupyterHub server
(1 x RTX PRO 6000 Blackwell 96 GB, the protocol's BF16 weights). Nothing here is used in local (mock) mode
except command planning, which is unit-tested."""
from __future__ import annotations

import concurrent.futures as cf
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from .core import now_iso, read_json, sha256_file, write_json

BACKBONES = {
    "A": {"base": "Qwen/Qwen3-Coder-30B-A3B-Instruct",
          "candidates": ["cyankiwi/Qwen3-Coder-30B-A3B-Instruct-AWQ-4bit",
                         "QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ",
                         "stelterlab/Qwen3-Coder-30B-A3B-Instruct-AWQ"]},
    "B": {"base": "Qwen/Qwen2.5-Coder-14B-Instruct",
          "candidates": ["Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"]},
}
# main-plan-v2.md Section 9.1: two vendors, both dense ~30B, BF16, Apache 2.0. Both checkpoints are vision-language
# models; only text is used, so the vision encoder is not loaded (--language-model-only).
CRC_BACKBONES = {"A": "Qwen/Qwen3.8-27B", "B": "google/gemma-4-31B-it"}
VLLM_LADDER = [None, "0.10.1.1"]  # None = latest release at first install; the lock pins it afterwards
LAUNCH_VARIANTS = [
    {"name": "default", "args": [], "env": {}},
    {"name": "eager", "args": ["--enforce-eager"], "env": {}},
    {"name": "eager_no_p2p", "args": ["--enforce-eager"], "env": {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"}},
]
PROFILES = {
    "kaggle": {"label": "2 x Tesla T4 (16 GB)", "gpus": 2, "tp": 2, "dtype": "float16", "port": 8000,
               "free_failed_weights": True,  # 20 GB-class disk: drop a failed candidate before the next one
               "vllm_ladder": VLLM_LADDER, "env": {}, "args": [],
               "base": {b: v["base"] for b, v in BACKBONES.items()},
               "candidates": {b: v["candidates"] for b, v in BACKBONES.items()}},
    # Section 9.1: BF16 weights of the named backbones; one at a time on a single 96 GB GPU.
    # Port 8000/8001 belong to the JupyterHub proxy on this server; weights are kept (2.7 TB disk, slow re-download).
    # The server has the CUDA driver but no CUDA toolkit (nvcc), so FlashInfer's JIT-compiled top-k/top-p sampler
    # cannot build; vLLM's PyTorch sampler is used instead. vLLM 0.10.x is not a usable fallback here (it fails with
    # transformers 5.x), so only the current release is tried and the lock pins it.
    "crc": {"label": "1 x NVIDIA RTX PRO 6000 Blackwell (96 GB)", "gpus": 1, "tp": 1, "dtype": "bfloat16", "port": 18000,
            "free_failed_weights": False, "vllm_ladder": [None], "env": {"VLLM_USE_FLASHINFER_SAMPLER": "0"},
            "args": ["--language-model-only"],
            "base": dict(CRC_BACKBONES),
            "candidates": {b: [repo] for b, repo in CRC_BACKBONES.items()}},
}
PORT = int(os.environ.get("ADM_VLLM_PORT", "0")) or None  # explicit override; otherwise the profile's port
LOCK_FILE = "models.lock.json"


def find_uv() -> list:
    """Use an existing uv binary if there is one; otherwise pip-install uv (with --user if the kernel env is read-only)."""
    import sysconfig
    for c in (os.environ.get("ADM_UV"), shutil.which("uv")):
        if c and Path(c).exists():
            return [c]
    user = not os.access(sysconfig.get_paths()["purelib"], os.W_OK)
    r = sh([sys.executable, "-m", "pip", "install", "-q", *(["--user"] if user else []), "uv"], 900)
    if r.returncode != 0:
        raise RuntimeError("could not install uv: " + r.stderr[-1500:])
    return [sys.executable, "-m", "uv"]


def sh(cmd, timeout=3600, env=None, log=None) -> subprocess.CompletedProcess:
    if log:
        log(f"$ {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout,
                          env={**os.environ, **(env or {})})


def gpu_info() -> dict:
    try:
        q = sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"], 60)
        rows = [r.split(", ") for r in q.stdout.strip().splitlines() if r.strip()]
        head = sh(["nvidia-smi"], 60).stdout
        cuda = None
        for tok in head.split("|"):
            if "CUDA Version" in tok:
                cuda = tok.split("CUDA Version:")[1].strip().split()[0]
        return {"gpus": [{"name": r[0], "memory": r[1], "driver": r[2], "compute_capability": r[3] if len(r) > 3 else None}
                         for r in rows], "cuda_driver_api": cuda}
    except Exception as e:
        return {"gpus": [], "error": str(e)}


def host_info() -> dict:
    import platform
    du = shutil.disk_usage("/")
    try:
        mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    except (ValueError, OSError, AttributeError):
        mem = None
    return {"python": sys.version.split()[0], "platform": platform.platform(), "cpus": os.cpu_count(),
            "ram_gib": round(mem, 1) if mem else None, "disk_free_gib": round(du.free / 2**30, 1)}


class Server:
    def __init__(self, env_dir: Path, models_dir: Path, configs_dir: Path, logs_dir: Path, log=print,
                 platform: str = "kaggle"):
        self.platform = platform
        self.profile = PROFILES[platform]
        self.env = Path(env_dir)
        self.models = Path(models_dir)
        self.configs = Path(configs_dir)
        self.logs = Path(logs_dir)
        self.logs.mkdir(parents=True, exist_ok=True)
        self.log = log
        self.proc = None
        self.started = None
        self.backbone = None
        self.gpu_seconds = {}
        self.port = PORT or self.profile["port"]

    @property
    def url(self) -> str:
        return f"http://127.0.0.1:{self.port}"

    @property
    def python(self) -> Path:
        return self.env / "bin" / "python"

    @property
    def vllm_bin(self) -> Path:
        return self.env / "bin" / "vllm"

    def lock(self) -> dict:
        p = self.configs / LOCK_FILE
        return read_json(p) if p.exists() else {}

    def save_lock(self, lock: dict):
        write_json(self.configs / LOCK_FILE, lock)

    # ---------------------------------------------------------------- environment
    def installed_version(self) -> str | None:
        if not self.python.exists():
            return None
        r = sh([str(self.python), "-c", "import vllm,sys;sys.stdout.write(vllm.__version__)"], 300)
        return r.stdout.strip() if r.returncode == 0 else None

    def install(self, version: str | None) -> str:
        """Create an isolated environment so the kernel's own Python/PyTorch stack is untouched."""
        cur = self.installed_version()
        if cur and (version is None or cur == version):
            return cur
        if self.env.exists():
            shutil.rmtree(self.env)
        uv = find_uv()
        r = sh(uv + ["venv", str(self.env), "--python", "3.12", "--seed"], 900, log=self.log)
        if r.returncode != 0:
            raise RuntimeError("uv venv failed: " + r.stderr[-1500:])
        spec = f"vllm=={version}" if version else "vllm"
        r = sh(uv + ["pip", "install", "--python", str(self.python), spec, "--torch-backend=auto", "huggingface_hub"],
               3600, log=self.log)
        if r.returncode != 0:
            raise RuntimeError(f"vLLM install ({spec}) failed: " + r.stderr[-2500:])
        v = self.installed_version()
        if not v:
            raise RuntimeError("vLLM installed but cannot be imported")
        freeze = sh(uv + ["pip", "freeze", "--python", str(self.python)], 300).stdout
        (self.configs / "vllm_env.lock.txt").write_text(freeze)
        return v

    # ---------------------------------------------------------------- models
    def download(self, repo: str, revision: str | None) -> dict:
        target = self.models / repo.replace("/", "__")
        code = (
            "import json,sys\nfrom huggingface_hub import snapshot_download, HfApi\n"
            f"repo={repo!r}; rev={revision!r}\n"
            "info=HfApi().model_info(repo, revision=rev)\n"
            f"p=snapshot_download(repo, revision=info.sha, local_dir={str(target)!r}, max_workers=8)\n"
            "print(json.dumps({'sha': info.sha, 'path': p}))\n")
        free = shutil.disk_usage(self.models if self.models.exists() else "/").free / 2**30
        if free < 25:
            self.log(f"warning: only {free:.1f} GiB free for model weights")
        r = sh([str(self.python), "-c", code], 7200, env={"HF_HUB_ENABLE_HF_TRANSFER": "0"}, log=self.log)
        if r.returncode != 0:
            raise RuntimeError(f"download of {repo} failed: " + r.stderr[-2000:])
        info = json.loads(r.stdout.strip().splitlines()[-1])
        files = {}
        for fn in ["config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json",
                   "quantization_config.json", "chat_template.jinja"]:
            if (target / fn).exists():
                files[fn] = sha256_file(target / fn)
        weights = sorted(p.name for p in target.glob("*.safetensors"))
        wh = hashlib.sha256()
        for w in weights:  # weight files: hash of (name, size) plus the hub's commit sha pins content
            wh.update(f"{w}:{(target / w).stat().st_size}".encode())
        cfg = read_json(target / "config.json") if (target / "config.json").exists() else {}
        return {"repo": repo, "revision": info["sha"], "path": str(target), "file_hashes": files,
                "tokenizer_hash": files.get("tokenizer.json"), "weight_files": weights,
                "weight_manifest_hash": wh.hexdigest(),
                "quantization": cfg.get("quantization_config", {}).get("quant_method"),
                "downloaded": now_iso()}

    # ---------------------------------------------------------------- launch
    def command(self, model_path: str, served: str, variant: dict, tp: int | None = None, max_len: int = 16384,
                max_seqs: int = 16, util: float = 0.90) -> list:
        tp = tp or self.profile["tp"]
        return [str(self.vllm_bin), "serve", model_path, "--served-model-name", served,
                "--tensor-parallel-size", str(tp), "--dtype", self.profile["dtype"], "--max-model-len", str(max_len),
                "--gpu-memory-utilization", str(util), "--max-num-seqs", str(max_seqs), "--enable-prefix-caching",
                "--host", "127.0.0.1", "--port", str(self.port), "--seed", "0"] + self.profile["args"] + variant["args"]

    def launch_env(self, variant: dict) -> dict:
        return {"VLLM_WORKER_MULTIPROC_METHOD": "spawn", **self.profile["env"], **variant["env"]}

    def variants(self) -> list:
        # NCCL peer-to-peer settings only matter with more than one GPU
        return [v for v in LAUNCH_VARIANTS if self.profile["gpus"] > 1 or not v["env"]]

    def port_in_use(self) -> bool:
        import socket
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(2)
            return s.connect_ex(("127.0.0.1", self.port)) == 0

    def start(self, model_path: str, served: str, variant: dict, timeout: float = 1800, **kw) -> bool:
        self.stop()
        if self.port_in_use():
            # never talk to (or report readiness of) a process this pipeline did not start
            raise RuntimeError(f"port {self.port} is already in use by another process; set ADM_VLLM_PORT to a free port")
        cmd = self.command(model_path, served, variant, **kw)
        logf = self.logs / f"vllm_{served}_{variant['name']}_{int(time.time())}.log"
        env = {**os.environ, **self.launch_env(variant)}
        for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
            env.pop(v, None)
        self.log("$ " + " ".join(cmd))
        self.proc = subprocess.Popen(cmd, stdout=open(logf, "w"), stderr=subprocess.STDOUT, env=env,
                                     start_new_session=True)
        self.logfile = logf
        t0 = time.time()
        while time.time() - t0 < timeout:
            if self.proc.poll() is not None:
                self.log(f"server exited with code {self.proc.returncode}; log tail:\n" + self.tail())
                self.proc = None
                return False
            try:
                with urllib.request.urlopen(f"{self.url}/v1/models", timeout=5) as r:
                    if r.status == 200:
                        self.started = time.time()
                        self.backbone = served
                        self.log(f"server ready after {time.time() - t0:.0f}s")
                        return True
            except Exception:
                pass
            time.sleep(10)
        self.log("server did not become ready in time; log tail:\n" + self.tail())
        self.stop()
        return False

    def tail(self, n: int = 60) -> str:
        try:
            return "\n".join(Path(self.logfile).read_text(errors="replace").splitlines()[-n:])
        except Exception:
            return ""

    def stop(self):
        if self.proc is not None:
            if self.started:
                self.gpu_seconds[self.backbone] = (self.gpu_seconds.get(self.backbone, 0.0)
                                                   + self.profile["gpus"] * (time.time() - self.started))
            pgid = None
            try:
                pgid = os.getpgid(self.proc.pid)
                os.killpg(pgid, signal.SIGTERM)
                self.proc.wait(60)
            except Exception:
                pass
            if pgid is not None:
                try:
                    os.killpg(pgid, signal.SIGKILL)  # engine-core children can outlive the API server process
                except Exception:
                    pass
            self.proc = None
            self.started = None
            self.backbone = None  # a second stop() (e.g. from a finally block) must not be accounted again
            self.wait_gpu_free()

    def wait_gpu_free(self, util: float = 0.90, timeout: float = 300):
        """Block until every GPU has enough free memory for the next server (vLLM releases memory asynchronously)."""
        t0 = time.time()
        while time.time() - t0 < timeout:
            try:
                q = sh(["nvidia-smi", "--query-gpu=memory.free,memory.total", "--format=csv,noheader,nounits"], 60)
                rows = [[float(x) for x in r.split(",")] for r in q.stdout.strip().splitlines() if r.strip()]
            except Exception:
                rows = []
            if not rows or all(free >= util * total for free, total in rows):
                return
            time.sleep(5)
        self.log(f"warning: GPU memory not released after {timeout:.0f}s")

    # ---------------------------------------------------------------- preflight
    def preflight(self, served: str, generation: dict) -> dict:
        from .llm import LLMError, VLLMClient, action_schema
        from .agents import parse_action
        res = {"served": served, "time": now_iso()}
        mode = None
        msgs = [{"role": "system", "content": "Reply with one JSON object."},
                {"role": "user", "content": 'Choose the action list_sources with empty args. Put the sum 17+25 in the thought.'}]
        for m in ("response_format", "structured_outputs", "guided_json"):
            c = VLLMClient(self.url, served, generation, m, timeout=300)
            try:
                out = c.complete(msgs, "single", 1)
                act, err = parse_action(out["text"], "single")
                if act is not None:
                    mode = m
                    res["schema_test"] = {"mode": m, "text": out["text"][:300], "action": act["action"],
                                          "contains_42": "42" in out["text"], "seconds": round(out["seconds"], 2)}
                    break
                res.setdefault("schema_errors", []).append(f"{m}: {err}: {out['text'][:200]}")
            except LLMError as e:
                res.setdefault("schema_errors", []).append(f"{m}: {str(e)[:300]}")
        res["structured_mode"] = mode
        if mode is None:
            res["ok"] = False
            return res
        # the constrained 'args' object must still carry free-form content
        c = VLLMClient(self.url, served, generation, mode, timeout=300)
        out = c.complete([{"role": "system", "content": "Reply with one JSON object."},
                          {"role": "user", "content": 'Call inspect_schema with args {"table": "observations"}.'}], "single", 2)
        act, _ = parse_action(out["text"], "single")
        res["args_test"] = {"text": out["text"][:300], "passed": bool(act and act["action"] == "inspect_schema"
                                                                        and act["args"].get("table") == "observations")}
        if not res["args_test"]["passed"]:
            res["ok"] = False
            return res
        # throughput: 8 concurrent agent-sized prompts
        filler = ("Catalog table observations has columns observation_id, sensor_1, sensor_2, sensor_3. " * 60)
        prompt = [{"role": "system", "content": "Reply with one JSON object."},
                  {"role": "user", "content": filler + "\nChoose list_sources."}]
        c = VLLMClient(self.url, served, generation, mode, timeout=600)
        t0 = time.time()
        with cf.ThreadPoolExecutor(8) as ex:
            outs = list(ex.map(lambda k: c.complete(prompt, "single", 100 + k), range(8)))
        dt = time.time() - t0
        res["throughput"] = {"concurrent_requests": 8, "wall_seconds": round(dt, 2),
                             "prompt_tokens_each": outs[0]["input_tokens"],
                             "output_tokens_total": sum(o["output_tokens"] for o in outs),
                             "output_tokens_per_second": round(sum(o["output_tokens"] for o in outs) / dt, 2),
                             "valid_json": sum(parse_action(o["text"], "single")[0] is not None for o in outs)}
        res["ok"] = res["throughput"]["valid_json"] >= 6 and res["schema_test"]["action"] == "list_sources"
        return res

    # ---------------------------------------------------------------- ladder
    def ensure(self, backbone: str, generation: dict, frozen: bool = False, max_minutes: float = 150) -> dict:
        """Bring up a backbone. Uses the lock if present; otherwise walks the fixed fallback order and
        records the first configuration that passes preflight. A frozen protocol never falls back."""
        lock = self.lock()
        t0 = time.time()
        entry = lock.get("backbones", {}).get(backbone)
        if entry:
            ladder = [(lock["vllm_version"], entry["repo"], entry["revision"], v) for v in LAUNCH_VARIANTS
                      if v["name"] == entry["launch_variant"]]
        elif frozen:
            raise RuntimeError(f"backbone {backbone} is not in the frozen model lock")
        else:
            versions = [lock["vllm_version"]] if lock.get("vllm_version") else self.profile["vllm_ladder"]
            ladder = [(ver, repo, None, var) for ver in versions for repo in self.profile["candidates"][backbone]
                      for var in self.variants()]
        failures = []
        for ver, repo, rev, var in ladder:
            if (time.time() - t0) / 60 > max_minutes:
                break
            try:
                installed = self.install(ver)
                m = self.download(repo, rev)
                if rev and m["revision"] != rev:
                    raise RuntimeError(f"revision mismatch {m['revision']} != {rev}")
                if not self.start(m["path"], backbone, var):
                    raise RuntimeError(f"server failed to start ({var['name']})")
                pf = self.preflight(backbone, generation)
                write_json(self.logs / f"preflight_{backbone}_{int(time.time())}.json", pf)
                if not pf.get("ok"):
                    raise RuntimeError(f"preflight failed: {json.dumps(pf)[:1500]}")
                lock = self.lock()
                if lock.get("vllm_version") and lock["vllm_version"] != installed and frozen:
                    raise RuntimeError("frozen vLLM version differs from installed version")
                lock["vllm_version"] = installed
                lock.setdefault("backbones", {})[backbone] = {**m, "base_model": self.profile["base"][backbone],
                                                              "launch_variant": var["name"],
                                                              "launch_command": self.command(m["path"], backbone, var),
                                                              "launch_env": self.launch_env(var),
                                                              "structured_mode": pf["structured_mode"],
                                                              "dtype": self.profile["dtype"], "preflight": pf}
                lock["platform"] = {"name": self.platform, "label": self.profile["label"]}
                lock["gpu"] = gpu_info()
                lock["host"] = host_info()
                self.save_lock(lock)
                return lock["backbones"][backbone]
            except Exception as e:
                failures.append({"vllm": ver, "repo": repo, "variant": var["name"], "error": str(e)[:1500]})
                self.log(f"[serving] attempt failed: {failures[-1]}")
                self.stop()
                # free disk from a failed candidate before trying the next repository
                tgt = self.models / repo.replace("/", "__")
                if (self.profile["free_failed_weights"] and not entry and var is self.variants()[-1]
                        and tgt.exists()):
                    shutil.rmtree(tgt, ignore_errors=True)
        write_json(self.logs / f"serving_failures_{backbone}.json", failures)
        raise RuntimeError(f"no working serving configuration for backbone {backbone}. "
                           f"See {self.logs}/serving_failures_{backbone}.json. The protocol forbids silently switching "
                           "to another model; revise and re-freeze the protocol instead.")


### Stage orchestration

In [ ]:
%%writefile adm_code/adm/stages.py
"""Stage orchestration used by the notebook: setup/restore, build, tests, pilot, freeze, formal runs,
analysis and the end-of-session summary."""
from __future__ import annotations

import hashlib
import os
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

from . import analysis as A
from . import datasets as D
from . import fixtures as FX
from . import power as PW
from . import references as REF
from . import runner as R
from . import tasks as T
from .core import (FAMILIES, GENERATION, PILOT_LIMITS, PROTOCOL_LIMITS, TEMPLATES, CONDITIONS, Paths, SessionClock,
                   dir_size_bytes, now_iso, read_json, write_json, append_jsonl, read_jsonl)

MOCK_BANNER = ("> **MOCK MODE.** Agent decisions came from the scripted MockLLM, not a language model. These numbers test the "
               "pipeline only and are not research results.")
PILOT_BANNER = ("> Pilot (E0) on separate development tasks. Not part of confirmatory scores. Used to calibrate limits, "
                "estimate variance and cost, and freeze the protocol.")
REHEARSAL_BANNER = "> **REHEARSAL DESIGN.** A reduced design used to test multi-session execution locally."

_ISOLATION = ("Agents act only through bounded tools; DuckDB runs in memory with external access disabled and configuration "
              "locked; the evaluator runs in a separate process and is the only component given the private evaluation directory.")
PLATFORM_DEVIATIONS = {
    "kaggle": [
        "Backbones are 4-bit AWQ community/official quantizations served in float16 (Tesla T4 has no native BF16 and cannot "
        "hold BF16 30B weights). Section 9.1 requires this to be recorded; the exact repository and revision are in configs/models.lock.json.",
        "Isolation is process-level instead of containers (Kaggle does not provide containers). " + _ISOLATION,
        "The 8 GiB per-run memory cap is measured (peak RSS) rather than enforced.",
        "Monetary cost is not available on Kaggle; GPU-seconds (2 x T4 server wall time) and active seconds are reported instead.",
    ],
    "crc": [
        "This run follows main-plan-v2.md, a revision of main-plan.md v1.0 made before the pilot: backbones changed to "
        "Qwen/Qwen3.8-27B (A) and google/gemma-4-31B-it (B), reasoning mode disabled (Section 9.2), single-GPU hardware "
        "(Section 20.1) and a ceiling check in E0. Exact repositories and revisions are in configs/models.lock.json.",
        "Isolation is process-level instead of containers (the CRC JupyterHub server provides no container runtime). " + _ISOLATION,
        "The 8 GiB per-run memory cap is measured (peak RSS) rather than enforced.",
        "Monetary cost is not available for the CRC server (no per-run tariff); GPU-seconds (1 x RTX PRO 6000 server wall "
        "time) and active seconds are reported instead.",
    ],
}
PLATFORM_DEVIATIONS["local"] = PLATFORM_DEVIATIONS["kaggle"]
COMMON_DEVIATIONS = [
    "Pilot limits are a calibration budget (300k tokens, 48 calls, 40 tool calls, 20 minutes). Frozen limits are set by the "
    "prespecified p90 rule in adm/power.py; the protocol text's 30k-token budget is reported alongside.",
    "The AI4I identifier column is UDI in the archive (the protocol says UID); it is mapped to uid and excluded.",
    "Repair cycles are counted as materialize_features calls issued after a failure verdict (gate failure or execution error).",
    "Workers in the three-agent team cannot submit or abstain; those are coordinator decisions (Section 9.4 responsibilities).",
    "One-hot encoding groups categories beyond 64 levels (max_categories) to protect memory if an identifier is used as a predictor.",
    "Fitted model pickles are deleted after evaluation except for a 2% deterministic sample retained for E6 replay; graphs, "
    "choices, provenance and test predictions are kept for every submission.",
    "Pickled models and hidden data are regenerated deterministically in each session and verified against recorded content digests.",
]
STATIC_DEVIATIONS = PLATFORM_DEVIATIONS["kaggle"] + COMMON_DEVIATIONS  # backwards-compatible name


def deviations_for(platform: str) -> list:
    return PLATFORM_DEVIATIONS.get(platform, PLATFORM_DEVIATIONS["kaggle"]) + COMMON_DEVIATIONS


SESSION_ENV = "ADM_RUN_ROOT"


def _proc_ids() -> list:
    try:
        return [int(p) for p in os.listdir("/proc") if p.isdigit()]
    except OSError:
        return []


def _proc_has_marker(pid: int, marker: bytes) -> bool:
    try:
        with open(f"/proc/{pid}/environ", "rb") as f:
            return marker in (f.read() + b"\0")
    except OSError:
        return False


def _is_python(pid: int) -> bool:
    """A live session owner is a kernel process. (Its own /proc environ predates the marker, so check the command.)"""
    try:
        with open(f"/proc/{pid}/cmdline", "rb") as f:
            return b"python" in f.read()
    except OSError:
        return False


def _ancestors(pid: int) -> set:
    out = set()
    while pid > 1 and pid not in out:
        out.add(pid)
        try:
            with open(f"/proc/{pid}/stat") as f:
                pid = int(f.read().rsplit(")", 1)[1].split()[1])
        except (OSError, ValueError, IndexError):
            break
    return out


def acquire_session(paths: Paths, log=print) -> dict:
    """Make a session safe to start after a previous one was killed (kernel restart, server cull, kill -9).

    Every process this pipeline starts (agent workers, evaluator workers, the vLLM server) inherits the environment
    marker ADM_RUN_ROOT=<state dir>. If the previous session's owner is dead, its leftover processes are terminated
    so the GPU and port are free. If another live session owns this state directory, starting is refused.
    Completed runs are never redone; a run that was in flight is retried once (Section 13.4)."""
    root = str(paths.state.resolve())
    os.environ[SESSION_ENV] = root
    marker = f"{SESSION_ENV}={root}\0".encode()
    lock = paths.state / "session.lock"
    me = os.getpid()
    if lock.exists():
        try:
            other = int(read_json(lock).get("pid", 0))
        except Exception:
            other = 0
        if other and other != me and _is_python(other):
            raise RuntimeError(f"another notebook session (pid {other}) is running on {root}; stop it before starting "
                               "a new one (Kernel -> Shut Down in its tab), then run again")
    mine = _ancestors(me)
    uid = os.getuid()
    stale = []
    for pid in _proc_ids():
        if pid in mine:
            continue
        try:
            if os.stat(f"/proc/{pid}").st_uid != uid:
                continue
        except OSError:
            continue
        if _proc_has_marker(pid, marker):
            stale.append(pid)
    if stale:
        log(f"previous session left {len(stale)} processes running (workers / model server); stopping them")
        for sig in (signal.SIGTERM, signal.SIGKILL):
            for pid in stale:
                try:
                    os.kill(pid, sig)
                except OSError:
                    pass
            t0 = time.time()
            while time.time() - t0 < 30 and any(Path(f"/proc/{p}").exists() for p in stale):
                time.sleep(1)
        t0 = time.time()  # the GPU releases memory asynchronously after the server dies
        while time.time() - t0 < 300:
            q = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], capture_output=True,
                               text=True) if shutil.which("nvidia-smi") else None
            if q is None or not {int(x) for x in q.stdout.split() if x.strip().isdigit()} & set(stale):
                break
            time.sleep(5)
    write_json(lock, {"pid": me, "started": now_iso()})
    return {"stopped_processes": len(stale)}


def release_session(paths: Paths):
    lock = paths.state / "session.lock"
    try:
        if int(read_json(lock).get("pid", 0)) == os.getpid():
            lock.unlink()
    except Exception:
        pass


def detect_mode() -> str:
    forced = os.environ.get("ADM_PLATFORM")
    if forced:
        return forced
    if Path("/kaggle/working").exists() and os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if Path("/srv/exaiops-shared").exists() and "/exaiops/" in os.environ.get("JUPYTERHUB_API_URL", ""):
        return "crc"
    return "local"


def make_paths(mode: str, local_root: Path | None = None, crc_root: Path | None = None) -> tuple[Paths, Path]:
    if mode == "kaggle":
        return Paths(Path("/kaggle/working/adm_state"), Path("/tmp/adm_cache"), Path("/tmp/adm_models")).ensure(), Path("/tmp/adm_vllm_env")
    if mode == "crc":
        # everything the notebook creates lives under one run folder and persists, so later sessions reuse the
        # vLLM environment and model weights
        root = Path(crc_root or "adm_crc_run").resolve()
        return Paths(root / "adm_state", root / "adm_cache", root / "adm_models").ensure(), root / "vllm_env"
    root = Path(local_root or "adm_local_run").resolve()
    return Paths(root / "adm_state", root / "adm_cache", root / "adm_models").ensure(), root / "vllm_env_unused"


# --------------------------------------------------------------------------- state
def state(paths: Paths) -> dict:
    p = paths.state / "STATE.json"
    return read_json(p) if p.exists() else {"created": now_iso(), "sessions": [], "stages": {}}


def save_state(paths: Paths, st: dict):
    st["updated"] = now_iso()
    write_json(paths.state / "STATE.json", st)


def restore_from_inputs(paths: Paths, log=print, roots=("/kaggle/input",)) -> str | None:
    """Copy the most recent previous-session state attached as a Kaggle input."""
    best, best_t = None, ""
    for root in roots:
        r = Path(root)
        if not r.exists():
            continue
        for p in list(r.glob("*/adm_state/STATE.json")) + list(r.glob("*/*/adm_state/STATE.json")) + \
                list(r.glob("*/*/*/adm_state/STATE.json")):
            try:
                t = read_json(p).get("updated", "")
            except Exception:
                continue
            if t > best_t:
                best, best_t = p.parent, t
    cur = state(paths).get("updated", "") if (paths.state / "STATE.json").exists() else ""
    if best and best_t > cur:
        log(f"restoring previous session state from {best} (updated {best_t})")
        shutil.copytree(best, paths.state, dirs_exist_ok=True)
        return str(best)
    log("no newer previous-session state found in inputs" if roots else "")
    return None


def init_config(paths: Paths, mode: str, user: dict) -> dict:
    p = paths.configs / "study.json"
    cfg = read_json(p) if p.exists() else {}
    frozen = (paths.manifests / "freeze.json").exists()
    defaults = {
        "study_id": f"agentic-data-mining-{'crc' if mode == 'crc' else 'kaggle'}-v1", "master_seed": 20261030, "human_subjects": False,
        "human_evaluators": False, "llm_judge": False, "runtime_human_approval": False,
        "limits": dict(PILOT_LIMITS), "generation": dict(GENERATION), "repetitions": [0, 1, 2],
        "optional_budget_experiment": False, "concurrency": 8 if mode in ("kaggle", "crc") else 6, "fit_slots": 3,
        "evaluator_workers": 1, "retain_model_fraction": 0.02,
    }
    for k, v in defaults.items():
        cfg.setdefault(k, v)
    lockable = set(PW.frozen_config_subset(cfg))
    for k, v in user.items():
        if v is None:
            continue
        if frozen and k in lockable and cfg.get(k) != v:
            raise RuntimeError(f"'{k}' is locked by the protocol freeze (frozen {cfg.get(k)}, requested {v})")
        cfg[k] = v
    cfg["mode"] = mode
    cfg["llm"] = cfg.get("llm") or {}
    cfg["llm"]["mode"] = "mock" if mode == "local" or user.get("force_mock") else "vllm"
    write_json(p, cfg)
    return cfg


def write_deviations(paths: Paths, extra: list | None = None):
    study = paths.configs / "study.json"
    platform = (read_json(study).get("mode") if study.exists() else None) or "kaggle"
    lines = ["# Protocol deviations (recorded automatically)", ""]
    lines += [f"{i + 1}. {d}" for i, d in enumerate(deviations_for(platform) + (extra or []))]
    lock = paths.configs / "models.lock.json"
    if lock.exists():
        L = read_json(lock)
        lines.append("")
        hw = (L.get("platform") or {}).get("label")
        lines.append(f"Resolved serving stack: vLLM {L.get('vllm_version')}" + (f" on {hw}." if hw else "."))
        for b, e in (L.get("backbones") or {}).items():
            lines.append(f"- Backbone {b}: {e.get('repo')} @ {e.get('revision')} (base {e.get('base_model')}), "
                         f"dtype {e.get('dtype')}, launch variant '{e.get('launch_variant')}', "
                         f"structured output via {e.get('structured_mode')}.")
    (paths.reports / "deviations.md").write_text("\n".join(lines) + "\n")


# --------------------------------------------------------------------------- build and tests
def stage_build(paths: Paths, log=print) -> dict:
    t0 = time.time()
    man = D.acquire(paths)
    summary = D.build_canonical(paths)
    D.build_splits(paths, "evaluation", FAMILIES, [0, 1, 2, 3, 4, 5])
    D.build_splits(paths, "development", ["D5", "D6"], [0, 1, 2])
    log(f"benchmark sources and splits ready in {time.time() - t0:.0f}s")
    return {"datasets": {k: v["sha256"] for k, v in man.items()}, "canonical": summary}


def stage_tests(pkg_root: Path, log=print) -> int:
    env = {**os.environ, "PYTHONPATH": str(pkg_root)}
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x", "-p", "no:cacheprovider", str(pkg_root / "tests")],
                       capture_output=True, text=True, env=env, cwd=str(pkg_root))
    log(r.stdout[-4000:] + r.stderr[-2000:])
    return r.returncode


# --------------------------------------------------------------------------- LLM
def start_llm(paths: Paths, cfg: dict, server, backbone: str, frozen: bool, log=print):
    if cfg["llm"]["mode"] == "mock":
        return
    entry = server.ensure(backbone, cfg["generation"], frozen=frozen)
    cfg["llm"].update({"base_url": server.url, "served_model": backbone,
                       "served_models": {"A": "A", "B": "B"}, "structured_mode": entry["structured_mode"]})
    lock = server.lock()
    cfg["model_lock"] = {b: {"repo": e["repo"], "revision": e["revision"], "tokenizer_hash": e.get("tokenizer_hash")}
                         for b, e in (lock.get("backbones") or {}).items()}
    write_json(paths.configs / "study.json", cfg)
    write_deviations(paths)


def stop_llm(paths: Paths, server, experiment: str):
    if server is None:
        return
    b = server.backbone
    server.stop()
    if b:
        append_jsonl(paths.outputs / "gpu_accounting.jsonl", {"time": now_iso(), "experiment": experiment, "backbone": b,
                                                              "gpu_seconds_total": server.gpu_seconds.get(b)})


# --------------------------------------------------------------------------- runs
def _prepare(paths: Paths, runs: list, cfg: dict, log=print) -> None:
    keys = {(r["suite"], r["family"], r["template"], r["condition"], r["repetition"]) for r in runs}
    t0 = time.time()
    metas = T.ensure_instances(paths, list(keys), workers=min(4, os.cpu_count() or 2))
    cfg["instance_hashes"] = {**cfg.get("instance_hashes", {}),
                              **{f"{k[0]}/{k[1]}-{k[2]}/{k[3]}/r{k[4]}": m["hashes"] for k, m in metas.items()}}
    cfg["split_hashes"] = read_json(paths.manifests / "splits.json")
    log(f"{len(keys)} task instances ready ({time.time() - t0:.0f}s)")


def _retain(run_id: str, frac: float) -> bool:
    return int(hashlib.sha256(run_id.encode()).hexdigest()[:8], 16) % 10000 < frac * 10000


def run_experiment(paths: Paths, cfg: dict, clock: SessionClock, runs: list, label: str, log=print) -> dict:
    if not runs:
        return {"scheduled": 0}
    rn = R.Runner(paths, cfg, clock, log)
    pend = rn.pending(runs)
    log(f"[{label}] {len(runs)} scheduled, {len(pend)} pending")
    if not pend:
        return {"scheduled": len(runs), "pending_at_start": 0, "completed": 0}
    _prepare(paths, pend, cfg, log)
    retain_ids = {r["run_id"] for r in pend if _retain(r["run_id"], cfg.get("retain_model_fraction", 0.02))}
    t0 = time.time()

    def progress(st, q, a, e):
        log(f"[{label}] done {st['completed']}/{st['pending_at_start']} | queued {q} | running {a} | evaluating {e} | "
            f"elapsed {(time.time() - t0) / 60:.1f} min | session {clock.elapsed() / 3600:.2f} h")

    st = rn.execute(pend, cfg["concurrency"], cfg.get("evaluator_workers", 1), progress, retain_ids)
    st["wall_seconds"] = round(time.time() - t0, 1)
    append_jsonl(paths.outputs / "stage_log.jsonl", {"time": now_iso(), "label": label, **st})
    log(f"[{label}] session result: {st}")
    return st


# --------------------------------------------------------------------------- pilot
def stage_pilot(paths: Paths, cfg: dict, clock: SessionClock, server, log=print) -> dict:
    st = state(paths)
    if st["stages"].get("pilot") == "done":
        log("pilot already complete")
        return st["stages"]
    log("== Evaluator fixtures (known-correct and known-incorrect artifacts) ==")
    fx = FX.run_evaluator_fixtures(paths, log=log)
    if not fx["all_passed"]:
        raise RuntimeError("evaluator fixtures failed; fix the evaluator before running agents")
    cfg["limits"] = dict(PILOT_LIMITS)
    runs = R.experiment_runs("E0", [0, 1])
    start_llm(paths, cfg, server, "A", frozen=False, log=log)
    res = run_experiment(paths, cfg, clock, runs, "E0", log)
    stop_llm(paths, server, "E0")
    if R.Runner(paths, cfg, clock).pending(runs):
        log("pilot incomplete; rerun this stage in the next session")
        save_state(paths, st)
        return st["stages"]
    # backbone-B smoke test (unscored)
    b_runs = R.experiment_runs("E0B", [0])
    start_llm(paths, cfg, server, "B", frozen=False, log=log)
    run_experiment(paths, cfg, clock, b_runs, "E0B", log)
    stop_llm(paths, server, "E0B")
    pilot_report(paths, cfg, log)
    st["stages"]["pilot"] = "done"
    save_state(paths, st)
    return st["stages"]


CEILING_M0_VCR = 0.95  # main-plan-v2.md E0 ceiling check


def ceiling_check(df: pd.DataFrame) -> str:
    """main-plan-v2.md E0: flag a benchmark that is too easy before the freeze (never swap in a weaker model)."""
    if df.empty or "V" not in df:
        return "## 15. Ceiling check\nNo pilot records.\n"
    vcr = df.groupby("system")["V"].mean()
    m0 = float(vcr.get("M0", float("nan")))
    full = sorted(s for s, v in vcr.items() if s != "F" and v >= 1.0)
    flag = (m0 >= CEILING_M0_VCR) or bool(full)
    lines = ["## 15. Ceiling check (main-plan-v2.md, E0)",
             f"M0 pilot VCR: {m0:.3f} (threshold {CEILING_M0_VCR}). Agentic systems at 100%: {', '.join(full) or 'none'}.",
             ("**CEILING WARNING:** revise task difficulty (conditions or tasks) before the freeze and record the change; "
              "do not substitute a weaker backbone." if flag else "No ceiling warning.")]
    return "\n".join(lines) + "\n"


TIER2_LEVELS = {"X1": ["X1a", "X1b", "X1c"], "X2": ["X2a", "X2b", "X2c"], "X3": ["X3a", "X3b", "X3c"]}
TIER2_LEVELS_V2 = {"X2": ["X2a", "X2b", "X2c"], "X3": ["X3a", "X3b", "X3c"], "X4": ["X4a", "X4b", "X4c"]}
CALIBRATIONS = {  # STAGE -> (experiment, repetitions, defect levels, output suffix, state key)
    "calibrate": ("E0X", [0], TIER2_LEVELS, "", "tier2_calibration"),
    "calibrate2": ("E0X2", [1, 2], TIER2_LEVELS_V2, "_v2", "tier2_calibration_v2"),
}
CALIBRATION_SYSTEMS = ["S0", "M0"]  # baseline agents whose success should sit near 50% (main-plan-v2.1 Section 12, E0X)


def tier2_calibration(df: pd.DataFrame, levels_map: dict | None = None) -> dict:
    """Prespecified rule (main-plan-v2.1): for each Tier-2 defect, freeze the level whose pooled S0+M0 verified-completion
    rate is closest to 0.5; ties go to the lower severity. Uses development tasks only."""
    out = {"rule": "argmin |VCR(S0,M0 pooled) - 0.5| per defect; ties -> lower level", "defects": {}}
    for d, levels in (levels_map or TIER2_LEVELS).items():
        rows = []
        for lv in levels:
            sub = df[(df["condition"] == lv) & (df["system"].isin(CALIBRATION_SYSTEMS))] if not df.empty else df
            n = int(len(sub))
            v = float(sub["V"].mean()) if n else None
            by = ({s: float(g["V"].mean()) for s, g in df[df["condition"] == lv].groupby("system")} if not df.empty else {})
            rows.append({"level": lv, "n_baseline_runs": n, "baseline_vcr": v, "vcr_by_system": by})
        scored = [r for r in rows if r["baseline_vcr"] is not None]
        chosen = min(scored, key=lambda r: (abs(r["baseline_vcr"] - 0.5), levels.index(r["level"])))["level"] if scored else None
        out["defects"][d] = {"levels": rows, "chosen": chosen, "complete": all(r["n_baseline_runs"] for r in rows)}
    return out


def stage_tier2_calibration(paths: Paths, cfg: dict, clock: SessionClock, server, log=print,
                            which: str = "calibrate") -> dict:
    """Tier-2 calibration pilot (development tasks, backbone A, pilot limits). Before any agent runs, the Tier-2
    evaluator fixtures must pass and the oracle solver must reproduce the hidden reference on every instance."""
    exp, reps, levels_map, sfx, key = CALIBRATIONS[which]
    st = state(paths)
    if st["stages"].get("pilot") != "done":
        raise RuntimeError("Tier-2 calibration runs after the main pilot (E0)")
    runs = R.experiment_runs(exp, reps)
    if not (paths.outputs / f"tier2_solvability{sfx}.json").exists():
        log("== Tier-2 evaluator fixtures ==")
        fx = FX.run_evaluator_fixtures(paths, log=log, fixtures=FX.TIER2_FIXTURES,
                                       out_name=f"evaluator_fixtures_tier2{sfx}.json")
        if not fx["all_passed"]:
            raise RuntimeError("Tier-2 evaluator fixtures failed; fix before running agents")
        keys = sorted({(r["suite"], r["family"], r["template"], r["condition"], r["repetition"]) for r in runs})
        _prepare(paths, runs, cfg, log)
        sol = FX.oracle_solvability(paths, keys, log)
        if not sol["all_solvable"]:
            raise RuntimeError(f"oracle could not solve {len(sol['failures'])} Tier-2 instances: {sol['failures'][:5]}")
        write_json(paths.outputs / f"tier2_solvability{sfx}.json", sol)
    cfg["limits"] = dict(PILOT_LIMITS)
    if R.Runner(paths, cfg, clock).pending(runs) and clock.may_start_new(120):
        start_llm(paths, cfg, server, "A", frozen=False, log=log)
        run_experiment(paths, cfg, clock, runs, exp, log)
        stop_llm(paths, server, exp)
    df = A.records_df(R.load_records(paths, exp))
    cal = tier2_calibration(df, levels_map)
    cal.update({"experiment": exp, "repetitions": reps})
    write_json(paths.manifests / f"tier2_calibration{sfx}.json", cal)
    pending = len(R.Runner(paths, cfg, clock).pending(runs))
    refs = REF.build_references(paths, "development", ["D5", "D6"], TEMPLATES, [0, 1], log)
    lines = ["## Tier-2 calibration (prespecified rule)", f"Rule: {cal['rule']}. Runs pending: {pending} of {len(runs)}.", ""]
    for d, v in cal["defects"].items():
        lines.append(f"- {d}: chosen level **{v['chosen']}** ({'complete' if v['complete'] else 'INCOMPLETE'}); " + "; ".join(
            f"{r['level']}: S0+M0 VCR {r['baseline_vcr'] if r['baseline_vcr'] is None else round(r['baseline_vcr'], 3)} "
            f"(n={r['n_baseline_runs']})" for r in v["levels"]))
    banner = PILOT_BANNER + f"\n\n> Tier-2 calibration pilot {exp} (main-plan-v2.1), recorded separately from E0."
    title = f"Tier2{sfx} calibration pilot {exp} report"
    text = A.report(df, title, paths.reports, paths.figures, refs, banner, draws=2000,
                    extra_sections=["\n".join(lines) + "\n", ceiling_check(df)]) if not df.empty else "\n".join(lines)
    (paths.reports / f"pilot_tier2{sfx}.md").write_text(text)
    if pending:
        log(f"Tier-2 calibration incomplete ({pending} runs pending); run STAGE = '{which}' again to continue")
    else:
        st["stages"][key] = "done"
    save_state(paths, st)
    log(f"Tier-2 calibration report written: {paths.reports / f'pilot_tier2{sfx}.md'}")
    return cal


def stage_pilot_supplement(paths: Paths, cfg: dict, clock: SessionClock, server, log=print) -> dict:
    """Supplementary pilot E0S: development tasks under C1-C5 (backbone A, pilot limits). Recorded and reported
    separately from E0; it does not change the freeze proposal."""
    st = state(paths)
    if st["stages"].get("pilot") != "done":
        raise RuntimeError("the supplementary pilot runs after the main pilot (E0)")
    cfg["limits"] = dict(PILOT_LIMITS)
    plan = [("E0S", "A"), ("E0SB", "B")]
    for exp, backbone in plan:
        runs = R.experiment_runs(exp, [0, 1])
        if not R.Runner(paths, cfg, clock).pending(runs):
            continue
        if not clock.may_start_new(120):
            log("not enough session time left to start the next supplementary block")
            break
        start_llm(paths, cfg, server, backbone, frozen=False, log=log)
        run_experiment(paths, cfg, clock, runs, exp, log)
        stop_llm(paths, server, exp)
    if any(R.Runner(paths, cfg, clock).pending(R.experiment_runs(e, [0, 1])) for e, _ in plan):
        log("supplementary pilot incomplete; run STAGE = 'supplement' again to continue")
        save_state(paths, st)
        return st["stages"]
    refs = REF.build_references(paths, "development", ["D5", "D6"], TEMPLATES, [0, 1], log)
    banner = PILOT_BANNER + "\n\n> Supplementary pilot, recorded separately from E0 (Section 12)."
    df = A.records_df(R.load_records(paths, "E0S"))
    text = A.report(df, "Supplementary pilot E0S report (backbone A, C1-C5)", paths.reports, paths.figures, refs, banner,
                    draws=2000, extra_sections=[ceiling_check(df)])
    dfb = A.records_df(R.load_records(paths, "E0SB"))
    text += "\n\n" + A.report(dfb, "BackboneB supplementary pilot E0SB report (M0, SQP, MQP; C0-C6)", paths.reports,
                              paths.figures, refs, banner, draws=2000)
    (paths.reports / "pilot_supplement.md").write_text(text)
    st["stages"]["pilot_supplement"] = "done"
    save_state(paths, st)
    log(f"supplementary pilot report written: {paths.reports / 'pilot_supplement.md'}")
    return st["stages"]


def pilot_report(paths: Paths, cfg: dict, log=print) -> dict:
    df = A.records_df(R.load_records(paths, "E0"))
    refs = REF.build_references(paths, "development", ["D5", "D6"], TEMPLATES, [0, 1], log)
    stage_wall = sum(e.get("wall_seconds", 0) for e in read_jsonl(paths.outputs / "stage_log.jsonl") if e["label"] == "E0")
    tp = PW.throughput(df, stage_wall, cfg["concurrency"])
    power = PW.power_analysis(df, paths.outputs)
    cal = PW.calibrate_limits(df)
    gpus = 1 if cfg.get("mode") == "crc" else 2
    est = {r: PW.session_estimates(tp, r, gpus=gpus) for r in (3, 4, 5, 6)}
    feasible = [g for g in power["grid"] if g["sd_block_logit"] == 0.5 and g["power"] >= 0.8]
    rec_reps = min([g["reps"] for g in feasible], default=3)
    proposal = {"limits": cal["limits"], "calibration": cal, "recommended_repetitions": list(range(rec_reps)),
                "throughput": tp, "estimates": est, "power": power, "created": now_iso(),
                "note": "Edit FREEZE_REPETITIONS / FREEZE_LIMIT_OVERRIDES in the config cell to deviate; overrides are recorded."}
    write_json(paths.manifests / "freeze_proposal.json", proposal)
    banner = PILOT_BANNER + ("\n\n" + MOCK_BANNER if cfg["llm"]["mode"] == "mock" else "")
    extra = ["## 7. Usage percentiles (agentic runs ending SUBMITTED or ABSTAINED)\n" + A.md_table(A.usage_percentiles(df), "{:.0f}"),
             "## 8. Budget exhaustion\n" + A.md_table(df[df["system"] != "F"].groupby("system")["terminal"]
                                                      .apply(lambda t: (t == "BUDGET_EXCEEDED").mean()).to_frame("budget_exceeded_rate")),
             "## 9. Throughput and run-time estimates\n" + "```\n" + pd.Series(tp).to_string() + "\n```\n" +
             A.md_table(pd.DataFrame([{"repetitions": r, **v["runs"], **{k: v.get(k) for k in ("estimated_hours", "estimated_sessions", "estimated_gpu_hours")}} for r, v in est.items()])),
             "## 10. Power simulation (10-point VCR improvement, planned paired analysis)\n" + A.md_table(pd.DataFrame(power["grid"])) +
             f"\nAssumptions: {power['assumptions']}\n",
             "## 11. Freeze proposal\n" + f"Frozen limits by the prespecified rule: `{cal['limits']}` (n={cal['n_runs']}).\n\n"
             f"Protocol-text limits: `{PROTOCOL_LIMITS}`.\n\nRecommended repetitions: {list(range(rec_reps))}.\n",
             "## 12. Evaluator fixtures\n" + A.md_table(pd.DataFrame(read_json(paths.outputs / "evaluator_fixtures.json")["fixtures"]))]
    b = A.records_df(R.load_records(paths, "E0B"))
    if not b.empty:
        extra.append("## 13. Backbone-B smoke test (unscored)\n" + A.md_table(b.groupby("system").agg(
            runs=("run_id", "size"), verified=("V", "sum"), terminal=("terminal", lambda t: ", ".join(sorted(set(t)))))))
    exc = int(df["exceptions"].sum()) if not df.empty else 0
    extra.append(f"## 14. Unexpected agent-loop exceptions\n{exc} runs recorded an internal exception (should be 0).\n")
    extra.append(ceiling_check(df))
    text = A.report(df, "Pilot E0 report", paths.reports, paths.figures, refs, banner, draws=2000, extra_sections=extra)
    (paths.reports / "pilot.md").write_text(text)
    log(f"pilot report written: {paths.reports / 'pilot.md'}")
    return proposal


# --------------------------------------------------------------------------- freeze
def stage_freeze(paths: Paths, cfg: dict, pkg_dir: Path, confirm: bool, reps_override=None, limit_overrides=None,
                 enable_e7: bool = False, log=print) -> dict | None:
    st = state(paths)
    if (paths.manifests / "freeze.json").exists():
        log("protocol already frozen")
        return read_json(paths.manifests / "freeze.json")
    if st["stages"].get("pilot") != "done":
        raise RuntimeError("freeze requires a completed pilot")
    if (paths.manifests / "tier2_calibration_v2.json").exists():  # main-plan-v2.1 design
        prop = freeze_proposal_v21(paths, cfg, log)
        cfg["tier2_conditions"] = list(prop["tier2_conditions"])
    else:
        prop = read_json(paths.manifests / "freeze_proposal.json")
    if not confirm:
        log("Freeze proposal (set FREEZE_CONFIRMED = True to accept):")
        log(f"  limits: {prop['limits']}\n  recommended repetitions: {prop['recommended_repetitions']}")
        for r, e in prop["estimates"].items():
            log(f"  reps={r}: {e}")
        return None
    limits = dict(prop["limits"])
    overrides = {}
    for k, v in (limit_overrides or {}).items():
        overrides[k] = {"proposed": limits.get(k), "frozen": v}
        limits[k] = v
    reps = list(reps_override) if reps_override else prop["recommended_repetitions"]
    cfg["limits"] = limits
    cfg["repetitions"] = reps
    cfg["optional_budget_experiment"] = bool(enable_e7)
    write_json(paths.configs / "study.json", cfg)
    # build every formal instance once and record its content digest
    formal = formal_runs(cfg)
    _prepare(paths, formal, cfg, log)
    # ABC II.7-II.8 on the evaluation portfolio: the oracle must solve every formal Tier-2 instance before the freeze
    t2 = set(cfg.get("tier2_conditions") or [])
    t2_keys = sorted({(r["suite"], r["family"], r["template"], r["condition"], r["repetition"])
                      for r in formal if r["condition"] in t2})
    sol = FX.oracle_solvability(paths, t2_keys, log) if t2_keys else {"instances": 0, "failures": [], "all_solvable": True}
    write_json(paths.outputs / "tier2_solvability_formal.json", sol)
    if not sol["all_solvable"]:
        raise RuntimeError(f"oracle could not solve {len(sol['failures'])} formal Tier-2 instances "
                           f"{sol['failures'][:5]}; the protocol is NOT frozen")
    lock = read_json(paths.configs / "models.lock.json") if (paths.configs / "models.lock.json").exists() else {"mock": True}
    man = PW.write_freeze(paths, pkg_dir, cfg, {**prop, "overrides": overrides, "repetitions": reps}, lock)
    lines = ["# Protocol freeze", "", f"Created {man['created']}.", f"Protocol hash `{man['protocol_hash']}`; code hash `{man['code_hash']}`.",
             f"Limits: `{limits}`.", f"Repetitions: {reps}. Optional E7: {enable_e7}.",
             f"Tier-2 conditions (main-plan-v2.1): {cfg.get('tier2_conditions') or 'none'}.",
             f"Overrides of the prespecified rule: {overrides or 'none'}.", f"Formal runs scheduled: {len(formal)}.",
             f"Oracle solvability of formal Tier-2 instances: {sol['instances'] - len(sol['failures'])}/{sol['instances']}.",
             f"LLM mode: {cfg['llm']['mode']}.", "", "Primary analysis: " + man["primary"],
             "Secondary (Holm): " + ", ".join(man["secondary_holm_family"])]
    (paths.reports / "protocol_freeze.md").write_text("\n".join(lines) + "\n")
    st["stages"]["freeze"] = "done"
    save_state(paths, st)
    log("\n".join(lines))
    return man


FORMAL_PILOT_SOURCES = ("E0", "E0S", "E0X2")  # backbone-A pilots run on the conditions of the frozen design


def _stage_rate(paths: Paths, labels: tuple) -> float | None:
    """Measured seconds per run for pilot stages (latest complete stage_log entry per label)."""
    rows = read_jsonl(paths.outputs / "stage_log.jsonl") if (paths.outputs / "stage_log.jsonl").exists() else []
    wall, n = 0.0, 0
    for lab in labels:
        ent = [e for e in rows if e.get("label") == lab and e.get("completed", 0) >= 50]
        if ent:
            wall += ent[-1]["wall_seconds"]
            n += ent[-1]["completed"]
    return wall / n if n else None


def freeze_proposal_v21(paths: Paths, cfg: dict, log=print) -> dict:
    """main-plan-v2.1 freeze proposal. The frozen Tier-2 levels come from the second calibration (E0X2) by the
    prespecified rule. Limits, power and run time are recomputed by the same prespecified rules from the pilot runs
    whose conditions belong to the frozen design (Tier 1 C0-C6 plus the frozen Tier-2 levels; backbone A)."""
    cal2 = read_json(paths.manifests / "tier2_calibration_v2.json")
    if not all(v["complete"] and v["chosen"] for v in cal2["defects"].values()):
        raise RuntimeError("the second Tier-2 calibration (E0X2) is incomplete; run STAGE = 'calibrate2' first")
    chosen = [v["chosen"] for v in cal2["defects"].values()]
    design = list(CONDITIONS) + chosen
    df = A.records_df(sum((R.load_records(paths, e) for e in FORMAL_PILOT_SOURCES), []))
    df = df[df["condition"].isin(design) & (df["backbone"].isin(["A", "none"]))]
    lim = PW.calibrate_limits(df)
    power = PW.power_analysis(df, paths.outputs, n_conditions=len(design), out_name="power_simulation_v21.json")
    feasible = [g for g in power["grid"] if g["sd_block_logit"] == 0.5 and g["power"] >= 0.8]
    rec_reps = max(3, min([g["reps"] for g in feasible], default=3))
    spr1, spr2, sprb = _stage_rate(paths, ("E0", "E0S")), _stage_rate(paths, ("E0X2",)), _stage_rate(paths, ("E0SB",))
    gpus = 1 if cfg.get("mode") == "crc" else 2
    est = {r: PW.formal_estimates(spr1, spr2, r, len(CONDITIONS), len(chosen), sprb, gpus) for r in (3, 4, 5, 6)}
    old = paths.manifests / "freeze_proposal.json"
    keep = paths.manifests / "freeze_proposal_E0only.json"
    if old.exists() and not keep.exists():
        shutil.copy2(old, keep)
    prop = {"version": "v2.1", "tier2_conditions": chosen, "design_conditions": design,
            "pilot_sources": list(FORMAL_PILOT_SOURCES), "n_pilot_runs": int(len(df)),
            "limits": lim["limits"], "calibration": lim, "recommended_repetitions": list(range(rec_reps)),
            "estimates": est, "power": power, "tier2_calibration": cal2, "created": now_iso(),
            "note": "Edit FREEZE_REPETITIONS / FREEZE_LIMIT_OVERRIDES in the config cell to deviate; overrides are recorded."}
    write_json(old, prop)
    lines = ["# Freeze proposal (main-plan-v2.1)", "",
             f"Frozen Tier-2 conditions (prespecified rule on E0X2): {', '.join(chosen)}.",
             f"Design conditions: {', '.join(design)}.",
             f"Pilot runs used for limits and power: {len(df)} (sources {', '.join(FORMAL_PILOT_SOURCES)}, design conditions only).",
             f"Limits ({lim['rule']}): `{lim['limits']}`; p95 `{ {k: round(v) for k, v in lim['p95'].items()} }`.",
             f"Recommended repetitions: {list(range(rec_reps))}.", "", "| repetitions | runs | estimated GPU hours |", "|---|---|---|"]
    lines += [f"| {r} | {e['runs']['total']} | {e.get('estimated_gpu_hours')} |" for r, e in est.items()]
    (paths.reports / "freeze_proposal_v21.md").write_text("\n".join(lines) + "\n")
    log("\n".join(lines))
    return prop


def formal_experiment_runs(cfg: dict, exp: str) -> list:
    """Formal run list with the frozen Tier-2 conditions (main-plan-v2.1) in E1 and E4."""
    return R.experiment_runs(exp, cfg["repetitions"], cfg.get("rehearsal"), cfg.get("tier2_conditions"))


def formal_runs(cfg: dict) -> list:
    reps = cfg["repetitions"]
    reh = cfg.get("rehearsal")
    runs = formal_experiment_runs(cfg, "E1") + formal_experiment_runs(cfg, "E5")
    if cfg.get("optional_budget_experiment"):
        runs += formal_experiment_runs(cfg, "E7")
    runs += formal_experiment_runs(cfg, "E4")
    return runs


# --------------------------------------------------------------------------- formal
def stage_formal(paths: Paths, cfg: dict, clock: SessionClock, server, pkg_dir: Path, log=print) -> dict:
    st = state(paths)
    lock = read_json(paths.configs / "models.lock.json") if (paths.configs / "models.lock.json").exists() else {}
    PW.verify_freeze(paths, pkg_dir, cfg, lock, require_model_lock=cfg["llm"]["mode"] != "mock")
    reps, reh = cfg["repetitions"], cfg.get("rehearsal")
    plan = [("E1", "A"), ("E5", "A")] + ([("E7", "A")] if cfg.get("optional_budget_experiment") else []) + [("E4", "B")]
    for exp, backbone in plan:
        runs = formal_experiment_runs(cfg, exp)
        rn = R.Runner(paths, cfg, clock)
        if not rn.pending(runs):
            continue
        if not clock.may_start_new(120):
            log("not enough session time left to start the next experiment")
            break
        needs_llm = any(r["system"] != "F" for r in rn.pending(runs))
        if needs_llm:
            start_llm(paths, cfg, server, backbone, frozen=True, log=log)
        run_experiment(paths, cfg, clock, runs, exp, log)
        if needs_llm:
            stop_llm(paths, server, exp)
        if rn.pending(runs):
            log(f"{exp} not finished in this session; it resumes next session")
            save_state(paths, st)
            return {"complete": False}
    if any(R.Runner(paths, cfg, clock).pending(formal_experiment_runs(cfg, e)) for e, _ in plan):
        save_state(paths, st)
        return {"complete": False}
    # references, replay and final analysis
    fams = (reh or {}).get("families", FAMILIES)
    tmpls = (reh or {}).get("templates", TEMPLATES)
    refs = REF.build_references(paths, "evaluation", fams, tmpls, reps, log)
    e6 = stage_e6(paths, log)
    final_report(paths, cfg, refs, e6, log)
    st["stages"]["formal"] = "done"
    save_state(paths, st)
    return {"complete": True}


def stage_e6(paths: Paths, log=print) -> dict:
    out_path = paths.outputs / "e6_replay.json"
    if out_path.exists():
        return read_json(out_path)
    results = []
    for rec in R.load_records(paths, "E1") + R.load_records(paths, "E4"):
        sub = paths.submissions / rec["experiment_id"] / rec["run_id"]
        if (sub / "model.pkl").exists() and rec["terminal_state"] == "SUBMITTED":
            results.append(FX.replay_checks(paths, rec, sub, log))
    out = {"submissions_checked": len(results), "all_passed": all(r["all_passed"] for r in results) if results else None,
           "results": results}
    write_json(out_path, out)
    log(f"E6 replay/invariance: {len(results)} retained submissions checked; all passed = {out['all_passed']}")
    return out


def final_report(paths: Paths, cfg: dict, refs: dict, e6: dict, log=print):
    mock = cfg["llm"]["mode"] == "mock"
    banner = (MOCK_BANNER + "\n\n" if mock else "") + (REHEARSAL_BANNER if cfg.get("rehearsal") else "")
    e1 = A.records_df(R.load_records(paths, "E1"))
    e4 = A.records_df(R.load_records(paths, "E4"))
    e5 = A.records_df(R.load_records(paths, "E5"))
    e7 = A.records_df(R.load_records(paths, "E7"))
    extra = []
    if not e4.empty and not e1.empty:
        sub = e1[e1["system"].isin(["M0", "SQP", "MQP"]) & (e1["family"] + "-" + e1["template"]).isin(
            ["D1-T5", "D2-T2", "D3-T4", "D4-T6", "D5-T3", "D6-T1"])]
        both = pd.concat([sub.assign(bb="A"), e4.assign(bb="B")])
        t = both.groupby(["bb", "system"])["V"].agg(["size", "mean"]).unstack(0)
        t.columns = [f"{'runs' if a == 'size' else 'VCR'}_backbone_{b}" for a, b in t.columns]
        dB = A.block_differences(e4, {"MQP": 1, "M0": -1})
        extra.append("## 7. Second-backbone replication (prespecified subset; limited model robustness, not generalization)\n"
                     + A.md_table(t) + (f"\nBackbone B MQP - M0: {100 * A.estimate(dB):.1f} pp\n" if dB is not None else ""))
    if not e5.empty:
        extra.append("## 8. E5 refusal on unrecoverable tasks\n" + A.md_table(A.negative_table(e5)))
    if not e7.empty:
        extra.append("## 9. E7 optional budget sensitivity (exploratory)\n" + A.md_table(
            e7.pivot_table(index="system", columns="budget", values="V", aggfunc="mean")))
    fx = read_json(paths.outputs / "evaluator_fixtures.json") if (paths.outputs / "evaluator_fixtures.json").exists() else {}
    extra.append("## 10. Replay, invariance and evaluator-fixture audit\n"
                 f"Evaluator fixtures all passed: {fx.get('all_passed')}. E6 retained submissions checked: "
                 f"{e6.get('submissions_checked')}; all passed: {e6.get('all_passed')}.\n")
    acct = run_accounting(paths, cfg)
    extra.append("## 11. Run accounting\n" + A.md_table(pd.DataFrame(acct).T, "{:.0f}"))
    gpu = read_jsonl(paths.outputs / "gpu_accounting.jsonl")
    if gpu:
        extra.append("## 12. GPU accounting (server sessions)\n" + A.md_table(pd.DataFrame(gpu)))
    tiers = A.tier_sections(e1, draws=10000)
    if tiers:
        extra.append(tiers)
    if not e4.empty and cfg.get("tier2_conditions"):
        extra.append("## Backbone B by tier\n" + A.md_table(e4.assign(tier=e4["condition"].map(A.tier_of))
                                                         .pivot_table(index="system", columns="tier", values="V", aggfunc="mean")))
    text = A.report(e1, "Final E1 report (backbone A)", paths.reports, paths.figures, refs, banner,
                    draws=10000, extra_sections=extra)
    (paths.reports / "final_report.md").write_text(text)
    log(f"final report written: {paths.reports / 'final_report.md'}")


def run_accounting(paths: Paths, cfg: dict) -> dict:
    out = {}
    for exp in ["E0", "E0B", "E1", "E4", "E5", "E7"]:
        if exp in ("E0",):
            runs = R.experiment_runs("E0", [0, 1])
        elif exp == "E0B":
            runs = R.experiment_runs("E0B", [0])
        else:
            if exp == "E7" and not cfg.get("optional_budget_experiment"):
                continue
            runs = formal_experiment_runs(cfg, exp)
        recs = {r["run_id"] for r in R.load_records(paths, exp)}
        att = list((paths.attempts / exp).glob("*.json")) if (paths.attempts / exp).exists() else []
        out[exp] = {"scheduled": len(runs), "recorded": len(recs & {r["run_id"] for r in runs}),
                    "pending": len({r["run_id"] for r in runs} - recs), "infrastructure_attempts": len(att)}
    return out


def session_summary(paths: Paths, cfg: dict, clock: SessionClock, log=print) -> dict:
    acct = run_accounting(paths, cfg)
    st = state(paths)
    size = dir_size_bytes(paths.state) / 2**20
    nxt = "run the pilot"
    if st["stages"].get("pilot") == "done":
        nxt = "review reports/pilot.md, then set FREEZE_CONFIRMED = True and run again"
    if st["stages"].get("freeze") == "done":
        nxt = ("attach this output as input and run again to continue the formal experiments" if cfg.get("mode") == "kaggle"
               else "run again to continue the formal experiments (completed runs are kept and skipped)")
    if st["stages"].get("formal") == "done":
        nxt = "study complete: see reports/final_report.md"
    st["sessions"].append({"ended": now_iso(), "hours": round(clock.elapsed() / 3600, 2), "accounting": acct})
    save_state(paths, st)
    release_session(paths)
    summary = {"stages": st["stages"], "accounting": acct, "state_size_mb": round(size, 1), "next": nxt}
    log(pd.DataFrame(acct).T.to_string())
    note = " (Kaggle output limit 20 GB)" if cfg.get("mode") == "kaggle" else ""
    log(f"stages: {st['stages']} | saved state size: {size:.1f} MB{note}\nNEXT: {nxt}")
    return summary


### Prompts

In [ ]:
%%writefile adm_code/adm/prompts/common.md
You are part of an automated, bounded data-mining system. No human will answer questions or approve actions. You must either submit a correct, contract-compliant workflow or abstain with a structured reason.

## Response format
Reply with exactly one JSON object and nothing else:
{"thought": "<one or two short sentences>", "action": "<action name>", "args": {...}}

## Tools (each call is charged to a shared budget)
- list_sources {} : tables, grains, keys, versions, shard counts, authority records.
- inspect_schema {"table": T} : columns, files (version, snapshot_date, modified), declared representations, authority.
- profile_data {"table": T, "policy": P, "partition": "train"|"validation", "sample_rows": n<=20} : bounded column summaries.
- query_data {"sql": "SELECT ..."} : read-only SQL; tables are named by file_id (e.g. observations__v2__train); at most 50 rows.
- materialize_features {"graph": G} : run a declarative transform graph on the train and validation partitions; returns a handle such as "H1".
- fit_evaluate {"handle": H, "candidate": C} : fit one permitted candidate; returns inner-CV and validation scores. At most 6 distinct evaluations per run.
- quality_check {"handle": H} / provenance_check {"handle": H} : return gate verdicts when that gate exists in this configuration.
- submit {"handle": H, "candidate": C, "findings": [{"type": ..., "detail": ...}]} : freeze the workflow. Only one successful submission.
- abstain {"reason": R, "detail": "..."} : stop. R is one of REQUIRED_SOURCE_MISSING, AMBIGUOUS_AUTHORITY, CONTRACT_UNSATISFIABLE, BUDGET_INSUFFICIENT.

## Transform graph G
{"nodes": [ ...nodes in execution order... ], "output": "<node id>"}. Each node has "id", "op" and arguments:
- {"op": "load", "table": T, "policy": "authoritative" | "authoritative_verified" | "latest_modified" | "latest_snapshot" | "version:<v>"} - per shard, chooses a source candidate. "authoritative" selects the file whose catalog metadata declares the version and hash listed in the catalog authority record and fails if it is missing or ambiguous. "authoritative_verified" does the same but also recomputes each candidate's actual content hash and keeps only candidates whose content matches the authority record.
- {"op": "parse_numeric", "input": N, "columns": "all" | [..]} - parses numbers, plain decimal strings and 'value:<decimal>' strings; null stays missing.
- {"op": "to_canonical", "input": N} - converts declared shard-level stored representations (stored = a*canonical + b) to canonical units. Apply to a single loaded table after parse_numeric and before joins.
- {"op": "map_alias", "input": N, "key": "observation_id"} - replaces permitted key aliases with canonical keys using the alias table.
- {"op": "normalize_key", "input": N, "key": "observation_id"} - rewrites key values to the canonical format (upper case, <PREFIX>-<six digits>, no surrounding spaces); values that do not match the pattern, such as permitted aliases, are left unchanged.
- {"op": "infer_representation", "input": N} - per shard and numeric column, if most values lie outside the catalog value_ranges, converts them from the documented catalog representation that brings them inside the range; undeclared shards only. Apply to a single loaded table after parse_numeric and to_canonical.
- {"op": "dedupe", "input": N, "mode": "exact" | "key", "key": K} - "exact" drops fully identical rows; "key" keeps the first row per key.
- {"op": "join", "left": N1, "right": N2, "on": [keys], "how": "inner" | "left", "cardinality": "one_to_one" | "many_to_one" | "one_to_many"} - cardinality is optional and enforced if given.
- {"op": "aggregate", "input": N, "key": K, "series_column": S, "value_column": V, "stats": ["mean", "pop_std", "sample_std", "count", "min", "max", "median"]} - one row per key with columns <series>_<stat label> (pop_std and sample_std are both labelled std).
- {"op": "drop_missing_rows", "input": N, "columns": [..]} - removes rows with missing values.
- {"op": "project", "input": N, "columns": [..]} - keeps only the listed columns.
The output must contain observation_id plus exactly the required features. The same frozen graph is later replayed on hidden test shards, so it must work from catalog metadata, not from specific rows.

## Rules
- Follow the task contract exactly: required feature names, excluded predictors, authoritative source versions, keys, units, aggregation rules, missing-value policy.
- Labels are only available inside fit_evaluate. Do not invent results. Use the observations you receive.
- Keep thoughts short. Budget is limited; do not repeat identical calls.


In [ ]:
%%writefile adm_code/adm/prompts/coordinator.md
## Your role: coordinator of a three-agent team
You build and revise the workflow plan, assign work, track the shared budget and decide submission or abstention.
Team members (same model, sequential execution):
- data_manager: discovers sources, resolves joins, units, versions, aliases and duplicates, and materializes a feature handle.
- miner: evaluates permitted candidates on a handle and recommends the best validation candidate.
Allowed actions: delegate {"role": "data_manager"|"miner", "instruction": "..."}, any tool, submit, abstain.
A delegation returns the member's report. Every message of every member is charged to the same budget, so delegate with precise instructions.


In [ ]:
%%writefile adm_code/adm/prompts/data_manager.md
## Your role: data-management agent
You discover sources, inspect schemas, resolve joins, units, versions, aliases and duplicates, and materialize a valid feature table that satisfies the contract.
Allowed actions: list_sources, inspect_schema, profile_data, query_data, materialize_features, quality_check, provenance_check, report.
Finish with report {"summary": "...", "handle": "H<n>" or null, "problem": "<reason if the contract cannot be satisfied>"}.


In [ ]:
%%writefile adm_code/adm/prompts/miner.md
## Your role: mining agent
You select permitted estimators, run bounded evaluations on the handle you are given and choose the final model.
Allowed actions: fit_evaluate, quality_check, provenance_check, report.
Finish with report {"summary": "...", "handle": "H<n>", "candidate": "<best validation candidate>"}.


In [ ]:
%%writefile adm_code/adm/prompts/single.md
## Your role: single agent
You plan the workflow, manage the data, select the model and decide whether to submit or abstain.
Allowed actions: list_sources, inspect_schema, profile_data, query_data, materialize_features, fit_evaluate, quality_check, provenance_check, submit, abstain.
Suggested order: inspect the catalog, build one complete graph, check the materialized summary against the contract, evaluate a few candidates, submit the best validation candidate on a correct handle.


### Schemas

In [ ]:
%%writefile adm_code/adm/schemas/action.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "ADM agent action (role-specific enums are generated by adm.llm.action_schema)",
  "type": "object",
  "required": ["thought", "action", "args"],
  "additionalProperties": false,
  "properties": {
    "thought": {"type": "string"},
    "action": {"type": "string"},
    "args": {"type": "object"}
  }
}


In [ ]:
%%writefile adm_code/adm/schemas/result.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "ADM run record (Section 18.3)",
  "type": "object",
  "required": ["run_id", "protocol_hash", "code_hash", "experiment_id", "dataset_id", "task_id", "condition_id",
               "repetition_id", "system_id", "backbone_id", "model_revision", "tokenizer_hash", "data_hashes",
               "split_hash", "corruption_seed", "estimator_seed", "agent_seed", "start_time", "terminal_state",
               "abstention_reason", "verified_completion", "external_failure_codes", "selected_candidate",
               "test_metrics", "input_tokens", "output_tokens", "model_calls", "tool_calls", "gate_calls",
               "repair_cycles", "fit_count", "active_seconds", "queue_seconds", "peak_worker_memory_mb",
               "server_gpu_seconds", "monetary_cost", "replay_status", "infrastructure_retry_count"],
  "properties": {
    "terminal_state": {"enum": ["SUBMITTED", "ABSTAINED", "BUDGET_EXCEEDED", "EXECUTION_ERROR", "INFRASTRUCTURE_ERROR"]},
    "verified_completion": {"enum": [0, 1, null]},
    "external_failure_codes": {"type": ["array", "null"], "items": {"type": "string"}}
  }
}


In [ ]:
%%writefile adm_code/adm/schemas/task.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "ADM task contract",
  "type": "object",
  "required": ["task_id", "instance_id", "template", "objective", "task_type", "target", "prediction_key",
               "selection_metric", "required_features", "numeric_features", "categorical_features",
               "excluded_predictors", "source_versions", "keys", "duplicates", "joins", "aggregates", "units",
               "numeric_format", "value_ranges", "missing_numeric_policy", "visible_partition_sizes",
               "final_refit_partitions", "candidate_models", "abstention_reasons", "output"],
  "properties": {
    "task_type": {"enum": ["binary", "multiclass", "regression"]},
    "selection_metric": {"enum": ["average_precision", "macro_f1", "rmse"]},
    "required_features": {"type": "array", "items": {"type": "string"}, "minItems": 1},
    "excluded_predictors": {"type": "array", "items": {"type": "string"}},
    "candidate_models": {"type": "array", "items": {"type": "string"}, "minItems": 6, "maxItems": 6},
    "final_refit_partitions": {"const": ["train", "validation"]},
    "joins": {"type": "array", "items": {"type": "object", "required": ["left", "right", "keys", "required_cardinality"]}},
    "aggregates": {"type": "array", "items": {"type": "object", "required": ["child_table", "key", "series", "stats", "output_columns"]}},
    "value_ranges": {"type": "object", "additionalProperties": {"type": "array", "minItems": 2, "maxItems": 2}}
  }
}


### Unit, property and isolation tests

In [ ]:
%%writefile adm_code/tests/conftest.py
import os
import sys
from pathlib import Path

import pytest

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from adm.core import Paths  # noqa: E402
from adm import datasets as D  # noqa: E402


@pytest.fixture(scope="session")
def paths(tmp_path_factory):
    root = tmp_path_factory.mktemp("adm")
    p = Paths(root / "state", root / "cache", root / "models").ensure()
    D.build_canonical(p, include_public=False)
    D.build_splits(p, "development", ["D5", "D6"], [0, 1, 2])
    return p


In [ ]:
%%writefile adm_code/tests/test_analysis_serving.py
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

from adm import analysis as A
from adm import power as PW
from adm.serving import BACKBONES, LAUNCH_VARIANTS, Server


def _synthetic(effect=0.2, reps=3, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for f in ["D1", "D2", "D3", "D4", "D5", "D6"]:
        for r in range(reps):
            for t in ["T1", "T2", "T3", "T4", "T5", "T6"]:
                for c in ["C0", "C1", "C2", "C3", "C4", "C5", "C6"]:
                    base = 0.5
                    for s, p in [("M0", base), ("MQP", base + effect), ("MQ", base + effect / 2), ("MP", base + effect / 2),
                                 ("SQP", base + effect), ("F", base), ("S0", base)]:
                        rows.append({"run_id": f"{f}{r}{t}{c}{s}", "family": f, "rep": r, "template": t, "condition": c,
                                     "system": s, "V": int(rng.random() < p), "terminal": "SUBMITTED"})
    return pd.DataFrame(rows)


def test_contrast_recovers_known_effect():
    df = _synthetic(0.2, reps=6)
    D = A.block_differences(df, {"MQP": 1, "M0": -1})
    est = A.estimate(D)
    assert abs(est - 0.2) < 0.06
    bs = A.block_bootstrap(D, draws=500)
    assert bs["ci_low"] < est < bs["ci_high"]
    assert A.sign_flip_test(D, draws=500) < 0.05
    null = A.block_differences(_synthetic(0.0, reps=3, seed=3), {"MQP": 1, "M0": -1})
    assert abs(A.estimate(null)) < 0.08


def test_bootstrap_resamples_whole_blocks():
    D = pd.DataFrame({"family": ["a", "a", "b", "b"], "rep": [0, 1, 0, 1], "D": [1.0, 1.0, -1.0, -1.0]})
    bs = A.block_bootstrap(D, draws=200)
    assert bs["ci_low"] == bs["ci_high"] == 0.0  # family means are constant under within-family resampling


def test_holm():
    adj = A.holm({"a": 0.01, "b": 0.04, "c": 0.03})
    assert adj["a"] == pytest.approx(0.03) and adj["c"] == pytest.approx(0.06) and adj["b"] == pytest.approx(0.06)


def test_limit_calibration_rule():
    df = pd.DataFrame({"system": ["S0"] * 21 + ["F"], "terminal": ["SUBMITTED"] * 22,
                       "tokens": [80000.0] * 21 + [10 ** 9], "model_calls": [10] * 22,
                       "tool_calls": [45] * 22, "active_s": [2000.0] * 22})
    lim = PW.calibrate_limits(df)["limits"]
    assert lim["aggregate_tokens"] == 100000          # 1.25 x 80k, above the 30k protocol floor
    assert lim["model_calls"] == 48                   # protocol floor
    assert lim["tool_calls"] == 40                    # capped at the pilot limit (40)
    assert lim["wall_seconds"] == 1200                # capped at the pilot limit
    assert lim["pipeline_evaluations"] == 6 and lim["repair_cycles"] == 2


def test_serving_command_and_ladder(tmp_path):
    s = Server(tmp_path / "env", tmp_path / "models", tmp_path / "cfg", tmp_path / "logs", log=lambda *a: None)
    cmd = s.command("/m", "A", LAUNCH_VARIANTS[1])
    assert cmd[1] == "serve" and "--tensor-parallel-size" in cmd and cmd[cmd.index("--dtype") + 1] == "float16"
    assert "--enforce-eager" in cmd and cmd[cmd.index("--max-model-len") + 1] == "16384"
    assert BACKBONES["A"]["candidates"][0].startswith("cyankiwi/") and BACKBONES["B"]["candidates"] == ["Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"]


def test_freeze_refuses_modified_config(tmp_path, paths):
    from adm.core import write_json
    pkg = Path(A.__file__).parent
    for k in ("datasets", "splits", "canonical"):
        if not (paths.manifests / f"{k}.json").exists():
            write_json(paths.manifests / f"{k}.json", {"x": 1})
    cfg = {"limits": {"aggregate_tokens": 1}, "generation": {}, "repetitions": [0, 1, 2], "optional_budget_experiment": False,
           "concurrency": 8, "fit_slots": 3}
    PW.write_freeze(paths, pkg, cfg, {}, {})
    PW.verify_freeze(paths, pkg, cfg, {}, require_model_lock=False)
    with pytest.raises(RuntimeError):
        PW.verify_freeze(paths, pkg, {**cfg, "repetitions": [0]}, {}, require_model_lock=False)
    with pytest.raises(RuntimeError):
        PW.verify_freeze(paths, pkg, cfg, {}, require_model_lock=True)
    (paths.manifests / "freeze.json").unlink()


def test_formal_design_with_tier2_and_locking():
    from adm import runner as R
    t2 = ["X2b", "X3a", "X4b"]
    e1 = R.experiment_runs("E1", [0, 1, 2], None, t2)
    assert len(e1) == 36 * 10 * 7 * 3 and len({r["run_id"] for r in e1}) == len(e1)
    assert len(R.experiment_runs("E4", [0, 1, 2], None, t2)) == 6 * 10 * 3 * 3
    assert len(R.experiment_runs("E5", [0, 1, 2], None, t2)) == 252  # negative controls unchanged
    assert len(R.experiment_runs("E1", [0, 1, 2])) == 5292  # Tier 1 only when no Tier-2 list is given
    cfg = {"limits": {}, "generation": {}, "repetitions": [0], "optional_budget_experiment": False, "concurrency": 8,
           "fit_slots": 3, "tier2_conditions": t2}
    assert PW.frozen_config_subset(cfg)["tier2_conditions"] == t2
    est = PW.formal_estimates(15.0, 20.0, 3, 7, 3, 16.0, 1)
    assert est["runs"] == {"E1": 7560, "E4": 540, "E5": 252, "total": 8352} and est["estimated_gpu_hours"] > 0


def _tiered(tier2_gap: float, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for fam in ["D1", "D2", "D3"]:
        for rep in range(3):
            for cond in ["C0", "C1", "X2b", "X4b"]:
                for tmpl in ["T1", "T2"]:
                    for s in ["M0", "MQ", "MP", "MQP", "SQP", "F"]:
                        p = 0.9 if cond.startswith("C") else (0.5 + (tier2_gap if s in ("MP", "MQP", "SQP") else 0))
                        rows.append({"run_id": f"{fam}{rep}{cond}{tmpl}{s}", "family": fam, "rep": rep, "condition": cond,
                                     "template": tmpl, "system": s, "V": float(rng.random() < p)})
    return pd.DataFrame(rows)


def test_tier_sections_and_difference():
    df = _tiered(0.4)
    r = A.tier_difference(df, A.CONTRASTS["P main effect"], draws=500)
    assert r is not None and r["estimate_pp"] > 10 and r["blocks"] == 9
    text = A.tier_sections(df, draws=300)
    assert "Tier-2 minus Tier-1" in text and "excluding X4" in text
    assert A.tier_sections(df[df["condition"].str.startswith("C")], draws=100) == ""


In [ ]:
%%writefile adm_code/tests/test_core_data.py
import numpy as np
import pandas as pd

from adm.core import SessionClock, derive_seed, df_hash, seed_string
from adm import datasets as D


def test_seed_vectors():
    assert seed_string("evaluation", "D5", "generator", "", "SYN-MACHINE-v1") == \
        "adm-kaggle-1.0|20261030|evaluation|D5|generator||SYN-MACHINE-v1"
    assert derive_seed("evaluation", "D5", "generator", "", "SYN-MACHINE-v1") == 16589070799177694325
    assert derive_seed("corruption", "D1", "D1-T2", 0, "C1:cell:train") == 13713059381983676020
    assert derive_seed("agent", "D3", "D3-T4", 2, "C6") == 2652557224513615479


def test_df_hash_order_invariant_and_value_sensitive():
    a = pd.DataFrame({"x": [1.0, 2.0, np.nan], "y": ["a", "b", None]})
    b = a.iloc[[2, 0, 1]][["y", "x"]]
    assert df_hash(a) == df_hash(b)
    c = a.copy()
    c.loc[0, "x"] = 1.0000001
    assert df_hash(a) != df_hash(c)
    assert df_hash(pd.concat([a, a.iloc[[0]]])) != df_hash(a)


def test_session_clock():
    t = [0.0]
    c = SessionClock(start=0.0, stop_new_h=1.0, hard_stop_h=1.5, now_fn=lambda: t[0])
    assert c.may_start_new(100)
    t[0] = 3550
    assert not c.may_start_new(100)
    assert not c.must_stop()
    t[0] = 5401
    assert c.must_stop()


def test_generators_deterministic_and_formula():
    a, _, _ = D.generate_d5(50, 20, "development", "D5X")
    b, _, _ = D.generate_d5(50, 20, "development", "D5X")
    assert df_hash(a) == df_hash(b)
    e, _, _ = D.generate_d5(50, 20, "evaluation", "D5")
    assert not set(a["observation_id"]) & set(e["observation_id"])
    big, _, parts = D.generate_d5(500, 20)
    s = big[[f"sensor_{j}" for j in range(1, 7)]]
    assert abs(s.to_numpy().std() - 1.0) < 0.05  # 0.4^2 + 0.84 = 1
    g, _, _ = D.generate_d6(300, 30)
    assert 45 < g["process_output"].mean() < 55


def test_splits(paths):
    for fam in ["D5", "D6"]:
        df, spec = D.load_canonical(paths, "development", fam)
        s = D.load_split(paths, "development", fam, 0)
        info = D.check_split(df, spec, s)
        assert sum(info["sizes"].values()) == len(df)
        assert (s.groupby("group")["partition"].nunique() == 1).all()
        folds = D.cv_splits(s, spec.split_rule)
        assert len(folds) == 3 and all(len(a) and len(b) for a, b in folds)


In [ ]:
%%writefile adm_code/tests/test_llm_http.py
"""The vLLM client and preflight against a local fake OpenAI-compatible server (no GPU needed)."""
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer as HTTPServer

import pytest

from adm import tasks as T
from adm.agents import execute_run
from adm.core import GENERATION, PILOT_LIMITS, SYSTEM_SPEC, instance_dirs
from adm.llm import MockLLM, VLLMClient
from adm.serving import Server
from adm.tools import Budget, ToolSession

STATE = {"requests": [], "max_chars": 10 ** 9, "mock": None, "reject_response_format": False}


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def do_GET(self):
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b'{"data": []}')

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        STATE["requests"].append(body)
        chars = sum(len(m["content"]) for m in body["messages"])
        if STATE["reject_response_format"] and "response_format" in body:
            return self._send(400, {"error": "response_format not supported"})
        if chars > STATE["max_chars"]:
            return self._send(400, {"error": f"This model's maximum context length is 16384 tokens; prompt has {chars}"})
        user = body["messages"][-1]["content"]
        if "inspect_schema" in user and "Call" in user:
            text = json.dumps({"thought": "ok", "action": "inspect_schema", "args": {"table": "observations"}})
        elif "Choose" in user:
            text = json.dumps({"thought": "17+25=42", "action": "list_sources", "args": {}})
        else:
            role = body["response_format"]["json_schema"]["name"].replace("_action", "") if "response_format" in body else "single"
            text = STATE["mock"].complete(body["messages"], role, 0)["text"]
        self._send(200, {"choices": [{"message": {"content": text}, "finish_reason": "stop"}],
                         "usage": {"prompt_tokens": chars // 3, "completion_tokens": len(text) // 3}})

    def _send(self, code, obj):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b)))
        self.end_headers()
        self.wfile.write(b)


class _Server(HTTPServer):
    request_queue_size = 64
    daemon_threads = True


@pytest.fixture(scope="module")
def fake_server():
    srv = _Server(("127.0.0.1", 0), Handler)
    t = threading.Thread(target=srv.serve_forever, daemon=True)
    t.start()
    yield srv.server_address[1]
    srv.shutdown()


def test_preflight_detects_structured_mode(fake_server, tmp_path):
    s = Server(tmp_path / "env", tmp_path / "m", tmp_path / "c", tmp_path / "l", log=lambda *a: None)
    s.port = fake_server
    STATE["reject_response_format"] = True
    pf = s.preflight("A", GENERATION)
    assert pf["ok"] and pf["structured_mode"] == "structured_outputs" and pf["args_test"]["passed"]
    STATE["reject_response_format"] = False
    pf = s.preflight("A", GENERATION)
    assert pf["ok"] and pf["structured_mode"] == "response_format" and pf["throughput"]["valid_json"] == 8


def test_agent_run_over_http_with_context_retrim(fake_server, paths, tmp_path):
    T.build_instance(paths, "development", "D5", "T3", "C6", 1)
    d = instance_dirs(paths, "development", "D5", "T3", "C6", 1)
    STATE["mock"] = MockLLM(11, invalid_rate=0.0)
    STATE["max_chars"] = 30000  # forces the client to hit a context-length error and re-trim
    STATE["requests"].clear()
    sess = ToolSession("h", d["visible"], d["trusted"], tmp_path / "sub", SYSTEM_SPEC["MQP"], Budget(dict(PILOT_LIMITS)),
                       {"estimator": 3, "agent": 4})
    client = VLLMClient(f"http://127.0.0.1:{fake_server}", "A", GENERATION, "response_format")
    out = execute_run(sess, "MQP", SYSTEM_SPEC["MQP"], client, GENERATION, 5)
    STATE["max_chars"] = 10 ** 9
    assert out["terminal_state"] in ("SUBMITTED", "ABSTAINED"), out
    req = STATE["requests"][0]
    assert req["response_format"]["type"] == "json_schema" and req["seed"] >= 0 and req["max_tokens"] == 1024
    assert req["response_format"]["json_schema"]["schema"]["properties"]["action"]["enum"][0] == "delegate"
    assert all(sum(len(m["content"]) for m in r["messages"]) <= 30000 for r in STATE["requests"][-3:])
    assert not any(e.get("event") == "exception" for e in out["trace"])


In [ ]:
%%writefile adm_code/tests/test_schemas_runner.py
import json
from pathlib import Path

import jsonschema

from adm import runner as R
from adm import tasks as T
from adm.core import GENERATION, PILOT_LIMITS, SessionClock, instance_dirs, read_json

SCHEMAS = Path(R.__file__).parent / "schemas"


def test_contract_matches_schema(paths):
    schema = json.loads((SCHEMAS / "task.schema.json").read_text())
    for tmpl in ["T1", "T3", "T6"]:
        T.build_instance(paths, "development", "D5", tmpl, "C1", 0)
        jsonschema.validate(read_json(instance_dirs(paths, "development", "D5", tmpl, "C1", 0)["visible"] / "contract.json"), schema)


def test_run_counts():
    assert len(R.experiment_runs("E0", [0, 1])) == 336
    assert len(R.experiment_runs("E1", [0, 1, 2])) == 5292
    assert len(R.experiment_runs("E4", [0, 1, 2])) == 378
    assert len(R.experiment_runs("E5", [0, 1, 2])) == 252
    assert len(R.experiment_runs("E7", [0, 1, 2])) == 216
    assert len(R.experiment_runs("E0X", [0])) == 756
    e0x2 = R.experiment_runs("E0X2", [1, 2])
    assert len(e0x2) == 1512 and len({r["run_id"] for r in e0x2}) == 1512
    assert {r["condition"] for r in e0x2} == {"X2a", "X2b", "X2c", "X3a", "X3b", "X3c", "X4a", "X4b", "X4c"}
    assert len(R.experiment_runs("E0S", [0, 1])) == 840 and len(R.experiment_runs("E0SB", [0, 1])) == 504
    e1 = R.experiment_runs("E1", [0])
    assert len({r["run_id"] for r in e1}) == len(e1)
    blocks = {}
    for r in e1:
        blocks.setdefault(r["block"], []).append(r["system"])
    assert all(sorted(v) == sorted(["F", "S0", "M0", "MQ", "MP", "SQP", "MQP"]) for v in blocks.values())


def test_runner_resume_and_record_schema(paths):
    """Interrupt with a tiny session clock, resume, and check there are no duplicates or gaps."""
    schema = json.loads((SCHEMAS / "result.schema.json").read_text())
    runs = [r for r in R.experiment_runs("E0", [0]) if r["template"] == "T1" and r["family"] == "D6"][:7]
    keys = {(r["suite"], r["family"], r["template"], r["condition"], r["repetition"]) for r in runs}
    T.ensure_instances(paths, list(keys))
    cfg = {"limits": PILOT_LIMITS, "generation": GENERATION, "llm": {"mode": "mock", "invalid_rate": 0.0}, "fit_slots": 2}
    clock = SessionClock(stop_new_h=0.0)  # nothing may start
    st = R.Runner(paths, cfg, clock).execute(runs, concurrency=2)
    assert st["stopped_for_time"] and not R.load_records(paths, "E0")
    st = R.Runner(paths, cfg, SessionClock()).execute(runs, concurrency=4)
    recs = R.load_records(paths, "E0")
    assert sorted(r["run_id"] for r in recs) == sorted(r["run_id"] for r in runs)
    for rec in recs:
        jsonschema.validate(rec, schema)
        assert rec["exception_events"] == 0
    st = R.Runner(paths, cfg, SessionClock()).execute(runs, concurrency=4)
    assert st["pending_at_start"] == 0


In [ ]:
%%writefile adm_code/tests/test_tasks_engine.py
import gzip
import json
import math

import numpy as np
import pandas as pd
import pytest

from adm import engine as E
from adm import tasks as T
from adm.core import instance_dirs, read_json
from adm.fixed import contract_graph


def _matches(out, ref):
    o = out.assign(observation_id=out["observation_id"].astype(str)).sort_values("observation_id").reset_index(drop=True)
    r = ref.sort_values("observation_id").reset_index(drop=True)
    if len(o) != len(r) or not (o["observation_id"].values == r["observation_id"].values).all():
        return False
    for c in r.columns[1:]:
        if pd.api.types.is_numeric_dtype(r[c]):
            if not np.isclose(o[c].astype(float), r[c].astype(float), rtol=1e-6, atol=1e-8, equal_nan=True).all():
                return False
        elif not (o[c].astype(str).values == r[c].astype(str).values).all():
            return False
    return True


@pytest.mark.parametrize("fam,tmpl,cond", [("D5", "T1", "C2"), ("D5", "T2", "C4"), ("D6", "T3", "C3"),
                                           ("D6", "T4", "C6"), ("D5", "T5", "C5"), ("D6", "T6", "C4"), ("D5", "T6", "C1")])
def test_fixed_graph_reproduces_hidden_reference(paths, fam, tmpl, cond):
    T.build_instance(paths, "development", fam, tmpl, cond, 0)
    d = instance_dirs(paths, "development", fam, tmpl, cond, 0)
    c, cat = read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json")
    g = contract_graph(c, cat)
    for part, root in [("train", d["visible"]), ("validation", d["visible"]), ("test", d["private"])]:
        out, _ = E.execute(g, E.CatalogView.open(root, part))
        assert _matches(out, pd.read_parquet(d["private"] / f"reference_{part}.parquet")), (part, cond)


@pytest.mark.parametrize("fam,tmpl,cond", [("D5", "T1", "X1b"), ("D6", "T2", "X1c"), ("D5", "T3", "X2a"),
                                           ("D6", "T6", "X2c"), ("D5", "T4", "X3b"), ("D6", "T5", "X3c"),
                                           ("D5", "T6", "X3a"), ("D6", "T1", "X4a"), ("D5", "T3", "X4b"),
                                           ("D6", "T5", "X4c"), ("D5", "T6", "X4b")])
def test_tier2_oracle_solves_and_spec_graph_does_not(paths, fam, tmpl, cond):
    """Tier 2: every instance is solvable by the oracle; the declared-specification graph (F) fails on it."""
    from adm.fixed import oracle_graph
    T.build_instance(paths, "development", fam, tmpl, cond, 0)
    d = instance_dirs(paths, "development", fam, tmpl, cond, 0)
    c, cat = read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json")
    spec_ok = True
    for part, root in [("train", d["visible"]), ("validation", d["visible"]), ("test", d["private"])]:
        ref = pd.read_parquet(d["private"] / f"reference_{part}.parquet")
        out, _ = E.execute(oracle_graph(c, cat), E.CatalogView.open(root, part))
        assert _matches(out, ref), (part, cond)
        try:
            spec_out, _ = E.execute(contract_graph(c, cat), E.CatalogView.open(root, part))
            spec_ok &= _matches(spec_out, ref)
        except E.GraphError:
            spec_ok = False
    assert not spec_ok, f"{cond}: the declared-specification graph should not solve a Tier-2 instance"
    assert "_ingest_batch" in c["excluded_predictors"] and "value_ranges" in cat


def test_tier1_instances_unchanged_by_tier2_code(paths):
    """Tier-1 contracts and catalogs carry no Tier-2 fields (their content digests must stay reproducible)."""
    T.build_instance(paths, "development", "D5", "T2", "C3", 1)
    d = instance_dirs(paths, "development", "D5", "T2", "C3", 1)
    c, cat = read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json")
    assert "_ingest_batch" not in c["excluded_predictors"] and "value_ranges" not in cat
    assert all("_ingest_batch" not in t["columns"] for t in cat["tables"].values())


def test_content_drift_is_a_provenance_defect(paths):
    """X4: declared metadata matches the authority record, so 'authoritative' loads drifted content; only the provenance
    gate's content-hash check flags it; the quality gate's value-range check does not; heuristics do not repair it."""
    from adm import gates as G
    T.build_instance(paths, "development", "D5", "T1", "X4c", 1)
    d = instance_dirs(paths, "development", "D5", "T1", "X4c", 1)
    c, cat = read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json")
    man = read_json(d["visible"] / "visible_partition_manifest.json")
    view = lambda p: E.CatalogView.open(d["visible"], p)
    g = contract_graph(c, cat)
    frames, traces = {}, {}
    for p in ("train", "validation"):
        frames[p], traces[p] = E.execute(g, view(p))
    handle = {"id": "H1", "graph": g, "frames": frames, "traces": traces}
    prov = G.provenance_gate(handle, c, cat, [])
    assert any(f["check"].startswith("train:content_hash") for f in prov["failures"])
    qual = G.quality_gate(handle, c, cat, man)
    assert not any("value_ranges" in f["check"] for f in qual["failures"])
    for pol in ("latest_modified", "latest_snapshot"):
        g2 = {"nodes": [dict(n, policy=pol) if n["op"] == "load" else n for n in g["nodes"]], "output": g["output"]}
        out, tr = E.execute(g2, view("train"))
        assert any(not v["file_id"].endswith("__replica") for l in tr["loads"] for v in l["shards"].values())


def test_tier2_instances_carry_replicas_and_tier1_do_not(paths):
    for cond, expect in (("X2b", True), ("C3", False)):
        T.build_instance(paths, "development", "D6", "T2", cond, 2)
        cat = read_json(instance_dirs(paths, "development", "D6", "T2", cond, 2)["visible"] / "catalog.json")
        has = any(f["file_id"].endswith("__replica") for t in cat["tables"].values() for f in t["files"])
        assert has == expect, cond


def test_invalid_policy_is_a_tool_error_not_a_crash(paths):
    T.build_instance(paths, "development", "D5", "T1", "C0", 0)
    view = E.CatalogView.open(instance_dirs(paths, "development", "D5", "T1", "C0", 0)["visible"], "train")
    for bad in ("authority", "version", "version:", None, 3):
        with pytest.raises(E.GraphError):
            E.resolve(view, "observations", bad)


def test_normalize_key_value():
    assert E.normalize_key_value(" D5X-000060 ") == "D5X-000060"
    assert E.normalize_key_value("d5x-000060") == "D5X-000060"
    assert E.normalize_key_value("D5X-60") == "D5X-000060"
    assert E.normalize_key_value("d5x_60") == "d5x_60"  # permitted alias format is left for map_alias


def test_corruption_counts_and_ledger(paths):
    meta = T.build_instance(paths, "development", "D5", "T2", "C4", 1)
    d = instance_dirs(paths, "development", "D5", "T2", "C4", 1)
    ledger = json.load(gzip.open(d["private"] / "ledger.json.gz", "rt"))
    for part, info in meta["realized_severity"].items():
        assert info["affected"] == int(math.floor(0.20 * info["denominator"] + 0.5))
        n = sum(1 for e in ledger if e["partition"] == part)
        assert n == info["affected"]
    meta = T.build_instance(paths, "development", "D6", "T1", "C2", 1)
    for part, info in meta["realized_severity"].items():
        assert info["missing"] - info["encoded"] in (0, 1)


def test_visible_dirs_hold_no_test_data(paths):
    T.build_instance(paths, "development", "D5", "T3", "C0", 2)
    d = instance_dirs(paths, "development", "D5", "T3", "C0", 2)
    lab = pd.read_parquet(d["trusted"] / "labels.parquet")
    assert set(lab["partition"]) == {"train", "validation"}
    cat = read_json(d["visible"] / "catalog.json")
    assert all(f["partition"] != "test" for t in cat["tables"].values() for f in t["files"])
    names = {p.name for p in d["visible"].rglob("*")}
    assert not any("test" in n for n in names)


@pytest.mark.parametrize("cond,reason", [("N1", "REQUIRED_SOURCE_MISSING"), ("N2", "AMBIGUOUS_AUTHORITY")])
def test_negative_controls(paths, cond, reason):
    T.build_instance(paths, "development", "D6", "T2", cond, 0)
    d = instance_dirs(paths, "development", "D6", "T2", cond, 0)
    c, cat = read_json(d["visible"] / "contract.json"), read_json(d["visible"] / "catalog.json")
    with pytest.raises(E.ResolutionError) as ei:
        E.execute(contract_graph(c, cat), E.CatalogView.open(d["visible"], "train"))
    assert ei.value.reason == reason


def test_instance_digest_reproducible(paths):
    a = T.build_instance(paths, "development", "D6", "T5", "C6", 1)["hashes"]
    b = T.build_instance(paths, "development", "D6", "T5", "C6", 1)["hashes"]
    assert a == b


def test_graph_validation_errors():
    with pytest.raises(E.GraphError):
        E.validate_graph({"nodes": [{"id": "a", "op": "rm_rf", "table": "x"}], "output": "a"})
    with pytest.raises(E.GraphError):
        E.validate_graph({"nodes": [{"id": "a", "op": "project", "input": "zz", "columns": ["x"]}], "output": "a"})
    assert E.parse_cell("value:1.50")[0] == E.parse_cell("1.5")[0] == 1.5


In [ ]:
%%writefile adm_code/tests/test_tools_agents.py
import pytest

from adm import tasks as T
from adm.agents import Conversation, execute_run, parse_action
from adm.core import PILOT_LIMITS, GENERATION, SYSTEM_SPEC, instance_dirs
from adm.fixed import contract_graph
from adm.llm import MockLLM
from adm.tools import Budget, BudgetExceeded, ToolSession


def _session(paths, tmp_path, system="S0", inst=("D5", "T2", "C4", 0), limits=None):
    T.build_instance(paths, "development", inst[0], inst[1], inst[2], inst[3])
    d = instance_dirs(paths, "development", *inst)
    return ToolSession("t", d["visible"], d["trusted"], tmp_path / system, SYSTEM_SPEC[system],
                       Budget(limits or dict(PILOT_LIMITS)), {"estimator": 1, "agent": 2}), d


def test_sql_isolation(paths, tmp_path):
    s, d = _session(paths, tmp_path)
    priv = d["private"] / "labels_test.parquet"
    assert s.call("query_data", {"sql": "SELECT count(*) FROM block_a__v2__train"})["ok"]
    for sql in [f"SELECT * FROM read_parquet('{priv}')", f"COPY block_a__v2__train TO '/tmp/x.csv'",
                "INSTALL httpfs", "SELECT 1; SELECT 2", "SET enable_external_access=true",
                f"SELECT * FROM '{priv}'"]:
        r = s.call("query_data", {"sql": sql})
        assert not r["ok"], sql
    s.close()


def test_evaluation_cap_cache_and_budget(paths, tmp_path):
    s, _ = _session(paths, tmp_path, inst=("D5", "T1", "C0", 0))
    h = s.call("materialize_features", {"graph": contract_graph(s.contract, s.catalog)})["handle"]
    for c in s.contract["candidate_models"]:
        assert s.call("fit_evaluate", {"handle": h, "candidate": c})["ok"]
    assert s.call("fit_evaluate", {"handle": h, "candidate": "CL1"}).get("cached")
    s.handles["H2"] = s.handles[h]
    r = s.call("fit_evaluate", {"handle": "H2", "candidate": "CL1"})
    assert not r["ok"] and "limit" in r["error"]
    assert s.budget.evaluations == 6
    s.budget.limits["tool_calls"] = s.budget.tool_calls
    with pytest.raises(BudgetExceeded):
        s.call("list_sources", {})
    s.close()


def test_disabled_gates_do_not_leak(paths, tmp_path):
    s, _ = _session(paths, tmp_path, system="M0")
    obs = s.call("materialize_features", {"graph": contract_graph(s.contract, s.catalog)})
    assert "gate_verdicts" not in obs
    assert not s.call("quality_check", {"handle": obs["handle"]})["ok"]
    s.close()
    s2, _ = _session(paths, tmp_path, system="MQP")
    obs = s2.call("materialize_features", {"graph": contract_graph(s2.contract, s2.catalog)})
    assert {g["gate"] for g in obs["gate_verdicts"]} == {"quality", "provenance"}
    assert all(g["passed"] for g in obs["gate_verdicts"])
    s2.close()


def test_gates_detect_defects(paths, tmp_path):
    s, _ = _session(paths, tmp_path, system="MQP", inst=("D6", "T4", "C6", 0))
    g = contract_graph(s.contract, s.catalog)
    g["nodes"] = [dict(n, policy="latest_modified") if n["op"] == "load" else n for n in g["nodes"]]
    alias = {}
    nodes = []
    for n in g["nodes"]:
        n = {k: alias.get(v, v) if k == "input" else v for k, v in n.items()}
        if n["op"] == "to_canonical":
            alias[n["id"]] = n["input"]
            continue
        nodes.append(n)
    obs = s.call("materialize_features", {"graph": {"nodes": nodes, "output": g["output"]}})
    fails = {f["check"].split(":")[-1] if ":" in f["check"] else f["check"] for v in obs["gate_verdicts"] for f in v["failures"]}
    assert "canonical_units" in fails or "value_ranges" in fails
    assert any(c.startswith("authority") or "authority" in c for v in obs["gate_verdicts"] for c in [f["check"] for f in v["failures"]])
    s.close()


def test_conversation_trimming():
    c = Conversation("sys " * 100, "task " * 100)
    for k in range(50):
        c.add_assistant('{"a": %d}' % k, f"step {k}")
        c.add_user("obs " * 800)
    msgs = c.render(4000)
    total = sum(len(m["content"]) for m in msgs) / 3.2
    assert total < 4500
    assert msgs[0]["role"] == "system" and msgs[1]["role"] == "user" and msgs[-1]["role"] == "user"
    assert "Earlier steps" in msgs[1]["content"]
    roles = [m["role"] for m in msgs[1:]]
    assert all(roles[i] != roles[i + 1] for i in range(len(roles) - 1))


def test_parse_action():
    a, e = parse_action('noise {"thought": "x", "action": "list_sources", "args": {}} tail', "single")
    assert a["action"] == "list_sources" and e is None
    assert parse_action('{"thought": "x", "action": "delegate", "args": {}}', "single")[0] is None
    assert parse_action("not json", "miner")[0] is None


@pytest.mark.parametrize("system", ["F", "S0", "SQP", "M0", "MQP"])
def test_systems_reach_terminal_state(paths, tmp_path, system):
    s, _ = _session(paths, tmp_path, system=system, inst=("D6", "T6", "C6", 0))
    out = execute_run(s, system, SYSTEM_SPEC[system], MockLLM(5, invalid_rate=0.0), GENERATION, 7)
    assert out["terminal_state"] in ("SUBMITTED", "ABSTAINED", "BUDGET_EXCEEDED")
    assert not any(e.get("event") == "exception" for e in out["trace"]), out["trace"][-1]


## 4. Initialise paths, restore previous session state, load configuration

In [ ]:
sys.path.insert(0, str((WORK / CODE).resolve()))
import importlib, adm
from adm import stages as S, runner as R
from adm.core import SessionClock, read_json
from adm.serving import Server, gpu_info, host_info

MODE = PLATFORM
paths, VLLM_ENV = S.make_paths(MODE, Path(LOCAL_ROOT), Path(CRC_ROOT))
restored = S.restore_from_inputs(paths) if ON_KAGGLE else None
user = {"concurrency": CONCURRENCY, "force_mock": FORCE_MOCK or None}
if MODE == "local":
    user.update({"rehearsal": LOCAL_REHEARSAL})
user["package_versions"] = {"python": sys.version.split()[0], **PACKAGE_VERSIONS}
print("session:", S.acquire_session(paths))  # stops leftovers of a killed previous session; refuses if one is live
cfg = S.init_config(paths, MODE, user)
clock = SessionClock(start=NOTEBOOK_START, stop_new_h=SESSION_STOP_NEW_HOURS, hard_stop_h=SESSION_HARD_STOP_HOURS)
server = (Server(VLLM_ENV, paths.models, paths.configs, paths.outputs / "serving_logs",
                 platform="crc" if MODE == "crc" else "kaggle") if cfg["llm"]["mode"] == "vllm" else None)
S.write_deviations(paths)
st = S.state(paths)
print("state:", paths.state, "| stages done:", st["stages"], "| LLM:", cfg["llm"]["mode"])
print("host:", host_info())
print("gpu:", gpu_info() if MODE in ("kaggle", "crc") else "not used in local mode")
print("session limits (h): stop new", SESSION_STOP_NEW_HOURS, "| hard stop", SESSION_HARD_STOP_HOURS)


## 5. Build the benchmark sources (download, checksum, import assertions, splits)

In [ ]:
build_info = S.stage_build(paths)
for k, v in build_info["canonical"].items():
    print(f"{k:22s} rows={v['rows']:6d} groups={v['groups']:6d} features={v['features']:3d} {v.get('class_counts', '')}")


## 6. Unit, property and isolation tests

In [ ]:
if RUN_TESTS:
    code = S.stage_tests(WORK / CODE)
    assert code == 0, "unit tests failed: fix before running experiments"


## 7. Decide what this session runs

In [ ]:
st = S.state(paths)
frozen = (paths.manifests / "freeze.json").exists()
if STAGE == "auto":
    if st["stages"].get("pilot") != "done":
        RUN = ["pilot"]
    elif not frozen:
        RUN = ["freeze", "formal"] if FREEZE_CONFIRMED else ["freeze"]
    elif st["stages"].get("formal") != "done":
        RUN = ["formal"]
    else:
        RUN = ["report"]
else:
    RUN = [STAGE]
print("stages to run this session:", RUN)


## 8. Pilot E0: evaluator fixtures, model preflight, 336 pilot runs, backbone-B smoke test, power simulation and freeze proposal

In [ ]:
if "pilot" in RUN:
    try:
        S.stage_pilot(paths, cfg, clock, server)
    finally:
        S.stop_llm(paths, server, "E0")
if "supplement" in RUN:  # STAGE = "supplement": supplementary pilot E0S (C1-C5), recorded separately
    try:
        S.stage_pilot_supplement(paths, cfg, clock, server)
    finally:
        S.stop_llm(paths, server, "E0S")
for _cal in ("calibrate", "calibrate2"):  # STAGE = "calibrate2": second Tier-2 calibration E0X2 (X2, X3, X4)
    if _cal in RUN:
        try:
            S.stage_tier2_calibration(paths, cfg, clock, server, which=_cal)
        finally:
            S.stop_llm(paths, server, S.CALIBRATIONS[_cal][0])


## 9. Protocol freeze (requires `FREEZE_CONFIRMED = True`)

In [ ]:
if "freeze" in RUN:
    reps = FREEZE_REPETITIONS or (LOCAL_REPETITIONS if MODE == "local" else None)
    man = S.stage_freeze(paths, cfg, WORK / CODE, FREEZE_CONFIRMED, reps, FREEZE_LIMIT_OVERRIDES, ENABLE_E7)
    if man is None:
        RUN = [r for r in RUN if r != "formal"]


## 10. Formal experiments: E1 -> E5 -> (E7) -> E4, then references, E6 and the final analysis

In [ ]:
if "formal" in RUN:
    try:
        res = S.stage_formal(paths, cfg, clock, server, WORK / CODE)
        print("formal stage:", res)
    finally:
        S.stop_llm(paths, server, "formal")
if "report" in RUN:
    from adm import references as REF
    refs = REF.build_references(paths, "evaluation", (cfg.get("rehearsal") or {}).get("families", S.FAMILIES),
                                (cfg.get("rehearsal") or {}).get("templates", S.TEMPLATES), cfg["repetitions"])
    S.final_report(paths, cfg, refs, S.stage_e6(paths))


## 11. Session summary and reports

In [ ]:
summary = S.session_summary(paths, cfg, clock)
from IPython.display import Markdown, display
for name in (["final_report.md"] + (["pilot_tier2_v2.md"] if "calibrate2" in RUN else [])
             + (["pilot_tier2.md"] if "calibrate" in RUN else [])
             + (["pilot_supplement.md"] if "supplement" in RUN else []) + ["pilot.md", "protocol_freeze.md", "deviations.md"]):
    p = paths.reports / name
    if p.exists():
        display(Markdown(f"---\n### {name}\n\n" + p.read_text()[:60000]))
        break
